> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
# ============================================================
# REPRODUCTION — CELL 1
#
# Environment + input-file + basic data-integrity audit
#
# 目的：
# 1. 确认这是一个干净的新 Notebook / kernel
# 2. 记录 Python 和关键包版本
# 3. 检查复现所需文件是否存在
# 4. 检查大 Xenium 对象和 ROI 对象尺寸
# 5. 建立独立 reproduction 输出目录
#
# IMPORTANT:
# - 不覆盖旧结果
# - 不修改任何原始文件
# - 大 h5ad 只用 backed="r" 打开，不读入内存
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import sys
import platform
import importlib.metadata as md

import numpy as np
import pandas as pd
import anndata as ad


# ============================================================
# 1. PROJECT ROOT
#
# 这里继续使用我们之前一直使用的 CGN 项目目录
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


if not BASE.exists():

    raise FileNotFoundError(
        f"""
项目目录不存在：

{BASE}

Project directory not found.
Set CGN_PROJECT_DIR to the correct project directory before continuing.
"""
    )


print("=" * 70)
print("REPRODUCTION AUDIT — CELL 1")
print("=" * 70)

print("\nProject root:")
print(BASE)

print("\nCurrent notebook working directory:")
print(Path.cwd())


# ============================================================
# 2. CREATE NEW REPRODUCTION OUTPUT DIRECTORIES
#
# 注意：
# 这些目录只保存本次复现的新结果，
# 不会覆盖之前 manuscript 用的 figures / csv。
# ============================================================

REPRO_ROOT = (
    BASE /
    "reproduction"
)

REPRO_RESULTS = (
    REPRO_ROOT /
    "results"
)

REPRO_FIGURES = (
    REPRO_ROOT /
    "figures"
)

REPRO_LOGS = (
    REPRO_ROOT /
    "logs"
)


for folder in [
    REPRO_ROOT,
    REPRO_RESULTS,
    REPRO_FIGURES,
    REPRO_LOGS
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("\nReproduction output folders:")

for folder in [
    REPRO_ROOT,
    REPRO_RESULTS,
    REPRO_FIGURES,
    REPRO_LOGS
]:

    print("  ✓", folder)


# ============================================================
# 3. ENVIRONMENT AUDIT
# ============================================================

packages = [
    "numpy",
    "pandas",
    "scipy",
    "anndata",
    "scanpy",
    "statsmodels",
    "patsy",
    "matplotlib",
    "scikit-learn"
]


environment_rows = []


for pkg in packages:

    try:

        version = md.version(
            pkg
        )

        status = "PASS"

    except Exception:

        version = None

        status = "NOT FOUND"


    environment_rows.append({
        "package": pkg,
        "version": version,
        "status": status
    })


environment_df = pd.DataFrame(
    environment_rows
)


print("\n" + "=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print(
    "Python:",
    sys.version.replace(
        "\n",
        " "
    )
)

print(
    "Platform:",
    platform.platform()
)

display(
    environment_df
)


# save environment record
environment_path = (
    REPRO_LOGS /
    "environment_versions.csv"
)


environment_df.to_csv(
    environment_path,
    index=False
)


# ============================================================
# 4. INPUT FILE DEFINITIONS
# ============================================================

required_files = {

    # --------------------------------------------------------
    # Core data
    # --------------------------------------------------------

    "Large Xenium h5ad":
        BASE /
        "GSE294965_processed_data.h5ad",

    "Primary 782 ROI object":
        BASE /
        "roi_782_PC1_primary.h5ad",

    "ROI-associated cell metadata":
        BASE /
        "roi782_cell_metadata.pkl",


    # --------------------------------------------------------
    # Figure 3 / molecular trajectory inputs
    # --------------------------------------------------------

    "MAC/FIB ROI mean counts":
        BASE /
        "figure5_MAC_FIB_roi_mean_counts.csv",

    "Gene trajectory results":
        BASE /
        "figure5_MAC_FIB_gene_trajectory_results.csv",

    "Trajectory module assignments":
        BASE /
        "figure5_MAC_FIB_trajectory_modules.csv",

    "Main-text pathway enrichment":
        BASE /
        "figure5_maintext_pathway_enrichment.csv",

    "Frozen module scores":
        BASE /
        "figure5_frozen_module_scores.csv",

    "Slide-adjusted module sensitivity":
        BASE /
        "figure5_slide_adjusted_module_sensitivity.csv",


    # --------------------------------------------------------
    # Figure 4 / spatial inputs
    # --------------------------------------------------------

    "MAC-FIB spatial data":
        BASE /
        "figure4_driver_neighbor_data_smoothed.csv",

    "Primary spatial results":
        BASE /
        "figure4_driver_results_smoothed.csv",

    "k sensitivity":
        BASE /
        "figure4_smoothed_k_sensitivity.csv",

    "anti-GBM LOO":
        BASE /
        "figure4_smoothed_GBM_LOO.csv",

    "Model-specification sensitivity":
        BASE /
        "figure4_final_model_specification_robustness.csv"
}


# ============================================================
# 5. FILE AUDIT
# ============================================================

file_rows = []


for label, path in required_files.items():

    exists = path.exists()


    if exists:

        size_mb = (
            path.stat().st_size /
            1024**2
        )

    else:

        size_mb = np.nan


    file_rows.append({
        "label":
            label,

        "filename":
            path.name,

        "exists":
            exists,

        "size_MB":
            round(
                size_mb,
                3
            )
            if exists
            else np.nan,

        "path":
            str(
                path
            )
    })


file_audit = pd.DataFrame(
    file_rows
)


print("\n" + "=" * 70)
print("INPUT FILE AUDIT")
print("=" * 70)

display(
    file_audit
)


file_audit_path = (
    REPRO_LOGS /
    "input_file_audit.csv"
)


file_audit.to_csv(
    file_audit_path,
    index=False
)


missing_files = (
    file_audit.loc[
        ~file_audit[
            "exists"
        ],
        [
            "label",
            "filename"
        ]
    ]
)


# ============================================================
# 6. CRITICAL FILE CHECK
# ============================================================

critical_labels = [
    "Large Xenium h5ad",
    "Primary 782 ROI object",
    "ROI-associated cell metadata"
]


critical_missing = (
    file_audit[
        file_audit[
            "label"
        ].isin(
            critical_labels
        )
        &
        (
            ~file_audit[
                "exists"
            ]
        )
    ]
)


if len(
    critical_missing
) > 0:

    print("\n❌ CRITICAL FILES MISSING")

    display(
        critical_missing
    )

    raise FileNotFoundError(
        """
At least one required input file is missing.

Resolve the missing inputs before continuing.
Review the missing-file table above for details.
"""
    )


else:

    print(
        "\n✓ All critical input files found."
    )


# ============================================================
# 7. LARGE XENIUM OBJECT AUDIT
#
# backed='r' means:
# 只读取文件结构，不把 321 万细胞矩阵装入内存
# ============================================================

big_path = (
    required_files[
        "Large Xenium h5ad"
    ]
)


print("\n" + "=" * 70)
print("LARGE XENIUM OBJECT")
print("=" * 70)


big_ad = ad.read_h5ad(
    big_path,
    backed="r"
)


big_shape = (
    big_ad.n_obs,
    big_ad.n_vars
)


print(
    "Observed shape:",
    big_shape
)


expected_big_shape = (
    3_218_210,
    480
)


big_shape_pass = (
    big_shape
    ==
    expected_big_shape
)


print(
    "Expected shape:",
    expected_big_shape
)


print(
    "Status:",
    "PASS ✓"
    if big_shape_pass
    else "FAIL ❌"
)


# close file safely
try:

    big_ad.file.close()

except Exception:

    pass


# ============================================================
# 8. PRIMARY ROI OBJECT AUDIT
# ============================================================

roi_path = (
    required_files[
        "Primary 782 ROI object"
    ]
)


print("\n" + "=" * 70)
print("PRIMARY ROI OBJECT")
print("=" * 70)


roi_ad = ad.read_h5ad(
    roi_path
)


roi_shape = (
    roi_ad.n_obs,
    roi_ad.n_vars
)


expected_roi_shape = (
    782,
    480
)


roi_shape_pass = (
    roi_shape
    ==
    expected_roi_shape
)


print(
    "Observed shape:",
    roi_shape
)


print(
    "Expected shape:",
    expected_roi_shape
)


print(
    "Status:",
    "PASS ✓"
    if roi_shape_pass
    else "FAIL ❌"
)


# ============================================================
# 9. ROI METADATA AUDIT
# ============================================================

print("\nROI obs columns:")

print(
    list(
        roi_ad.obs.columns
    )
)


required_obs_columns = [
    "Disease",
    "PC1_crescent",
    "Patient_Sample_ID"
]


obs_check_rows = []


for column in required_obs_columns:

    obs_check_rows.append({

        "column":
            column,

        "exists":
            column
            in roi_ad.obs.columns
    })


obs_check = pd.DataFrame(
    obs_check_rows
)


print("\nRequired ROI metadata:")

display(
    obs_check
)


# ============================================================
# 10. ROI DISEASE COUNTS
#
# NOTE:
# 这些是 ROI 数量，不是完整 study cohort patient 数量
# ============================================================

if "Disease" in roi_ad.obs.columns:

    roi_disease_counts = (
        roi_ad.obs[
            "Disease"
        ]
        .value_counts()
    )


    print(
        "\nROI counts by Disease:"
    )

    display(
        roi_disease_counts
        .rename(
            "n_ROI"
        )
        .to_frame()
    )


# ============================================================
# 11. ROI PATIENT AVAILABILITY
#
# 这里故意不要求等于 full cohort 63。
# 这是 ROI analytical availability。
# ============================================================

if (
    "Disease"
    in roi_ad.obs.columns
    and
    "Patient_Sample_ID"
    in roi_ad.obs.columns
):

    roi_patient_counts = (
        roi_ad.obs
        .groupby(
            "Disease",
            observed=True
        )[
            "Patient_Sample_ID"
        ]
        .nunique()
    )


    print(
        "\nROI-analysis patients by Disease:"
    )


    display(
        roi_patient_counts
        .rename(
            "n_patients"
        )
        .to_frame()
    )


# ============================================================
# 12. CELL METADATA AUDIT
# ============================================================

cell_meta_path = (
    required_files[
        "ROI-associated cell metadata"
    ]
)


print("\n" + "=" * 70)
print("ROI-ASSOCIATED CELL METADATA")
print("=" * 70)


cells = pd.read_pickle(
    cell_meta_path
)


print(
    "Shape:",
    cells.shape
)


print(
    "\nColumns:"
)

print(
    list(
        cells.columns
    )
)


expected_cell_count = (
    586_628
)


cell_count_pass = (
    len(
        cells
    )
    ==
    expected_cell_count
)


print(
    "\nExpected ROI-associated cells:",
    expected_cell_count
)


print(
    "Observed ROI-associated cells:",
    len(
        cells
    )
)


print(
    "Status:",
    "PASS ✓"
    if cell_count_pass
    else "CHECK ⚠"
)


# ============================================================
# 13. FINAL CELL 1 SUMMARY
# ============================================================

summary_rows = [

    {
        "check":
            "Large Xenium shape",

        "expected":
            "3,218,210 × 480",

        "observed":
            f"{big_shape[0]:,} × {big_shape[1]:,}",

        "status":
            "PASS"
            if big_shape_pass
            else "FAIL"
    },

    {
        "check":
            "ROI object shape",

        "expected":
            "782 × 480",

        "observed":
            f"{roi_shape[0]:,} × {roi_shape[1]:,}",

        "status":
            "PASS"
            if roi_shape_pass
            else "FAIL"
    },

    {
        "check":
            "ROI-associated cells",

        "expected":
            "586,628",

        "observed":
            f"{len(cells):,}",

        "status":
            "PASS"
            if cell_count_pass
            else "CHECK"
    },

    {
        "check":
            "Critical files",

        "expected":
            "All present",

        "observed":
            (
                "All present"
                if len(
                    critical_missing
                )
                == 0
                else
                "Missing"
            ),

        "status":
            (
                "PASS"
                if len(
                    critical_missing
                )
                == 0
                else
                "FAIL"
            )
    }
]


cell1_summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 70)
print("CELL 1 FINAL SUMMARY")
print("=" * 70)


display(
    cell1_summary
)


summary_path = (
    REPRO_LOGS /
    "cell1_data_integrity_summary.csv"
)


cell1_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 14. STOP / GO DECISION
# ============================================================

hard_fail = (
    (not big_shape_pass)
    or
    (not roi_shape_pass)
    or
    (
        len(
            critical_missing
        )
        > 0
    )
)


print("\n" + "=" * 70)


if hard_fail:

    print(
        "❌ CELL 1 FAILED"
    )

    print(
        "Do not proceed to Cell 2 until the issue above is resolved."
    )


else:

    print(
        "✅ CELL 1 PASSED"
    )

    print(
        "核心数据结构正确，可以进入复现 Cell 2。"
    )


print("=" * 70)


print(
    "\nAudit files saved to:"
)

print(
    REPRO_LOGS
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 2
#
# Reproduce Figure 1 core statistics
#
# Checks:
# 1. Full study cohort
# 2. ROI disease distribution
# 3. Crescent PC1 range
# 4. Common progression support
# 5. Diffusion pseudotime
# 6. PC1–DPT Spearman correlation
#
# Output:
# reproduction/results/cell2_figure1_reproduction.csv
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

from scipy.stats import spearmanr


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Re-establish paths
#
# We define them again intentionally so this cell is easier
# to understand and reproduce.
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_ROOT = (
    BASE /
    "reproduction"
)

REPRO_RESULTS = (
    REPRO_ROOT /
    "results"
)

REPRO_FIGURES = (
    REPRO_ROOT /
    "figures"
)

REPRO_LOGS = (
    REPRO_ROOT /
    "logs"
)


for folder in [
    REPRO_ROOT,
    REPRO_RESULTS,
    REPRO_FIGURES,
    REPRO_LOGS
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


BIG_PATH = (
    BASE /
    "GSE294965_processed_data.h5ad"
)


ROI_PATH = (
    BASE /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 2. Expected manuscript values
#
# These are NOT used to calculate results.
# They are used only AFTER calculation for comparison.
# ============================================================

EXPECTED = {

    "study_total":
        63,

    "Control_patients":
        6,

    "LN_patients":
        19,

    "ANCA_patients":
        32,

    "antiGBM_patients":
        6,

    "n_ROI":
        782,

    "common_PC1_low":
        -4.117234,

    "common_PC1_high":
        13.401588,

    "spearman_rho":
        0.8263341779437914
}


# ============================================================
# 3. Helper functions
# ============================================================

def close_backed(
    adata
):

    try:

        adata.file.close()

    except Exception:

        pass


def find_column(
    columns,
    candidates
):

    for c in candidates:

        if c in columns:

            return c

    return None


def standardize_disease(
    x
):

    x = str(
        x
    ).strip()


    mapping = {

        "Cntrl":
            "Control",

        "Ctrl":
            "Control",

        "CONTROL":
            "Control",

        "Control":
            "Control",

        "SLE":
            "LN",

        "LN":
            "LN",

        "ANCA":
            "ANCA",

        "GBM":
            "anti-GBM",

        "anti-GBM":
            "anti-GBM",

        "Anti-GBM":
            "anti-GBM"
    }


    return mapping.get(
        x,
        x
    )


# ============================================================
# 4. FULL STUDY COHORT AUDIT
#
# Read only metadata from the large object in backed mode.
# No 3.2-million-cell expression matrix is loaded into RAM.
# ============================================================

print(
    "=" * 72
)

print(
    "CELL 2 — FIGURE 1 REPRODUCTION"
)

print(
    "=" * 72
)


big_ad = ad.read_h5ad(
    BIG_PATH,
    backed="r"
)


big_obs = (
    big_ad.obs.copy()
)


print(
    "\nLarge-object metadata columns:"
)


print(
    list(
        big_obs.columns
    )
)


disease_col_big = find_column(

    big_obs.columns,

    [
        "Disease",
        "disease",
        "Diagnosis",
        "diagnosis"
    ]
)


patient_col_big = find_column(

    big_obs.columns,

    [
        "Patient_Sample_ID",
        "Patient",
        "patient",
        "patient_id",
        "Sample_ID",
        "sample_id"
    ]
)


print(
    "\nDetected disease column:",
    disease_col_big
)


print(
    "Detected patient column:",
    patient_col_big
)


# ============================================================
# 5. Derive full cohort from large-object metadata
# ============================================================

full_cohort_derived = False


if (
    disease_col_big is not None
    and
    patient_col_big is not None
):

    cohort_df = (
        big_obs[
            [
                disease_col_big,
                patient_col_big
            ]
        ]
        .dropna()
        .copy()
    )


    cohort_df[
        "Disease_display"
    ] = (
        cohort_df[
            disease_col_big
        ]
        .map(
            standardize_disease
        )
    )


    cohort_df[
        "Patient_ID"
    ] = (
        cohort_df[
            patient_col_big
        ]
        .astype(str)
    )


    patient_table = (
        cohort_df[
            [
                "Disease_display",
                "Patient_ID"
            ]
        ]
        .drop_duplicates()
    )


    cohort_counts = (
        patient_table
        .groupby(
            "Disease_display",
            observed=True
        )[
            "Patient_ID"
        ]
        .nunique()
        .reindex(
            [
                "Control",
                "LN",
                "ANCA",
                "anti-GBM"
            ]
        )
        .fillna(
            0
        )
        .astype(
            int
        )
    )


    cohort_total = int(
        cohort_counts.sum()
    )


    full_cohort_derived = True


else:

    print(
        "\n⚠ Full cohort cannot be reconstructed "
        "directly from large-object metadata."
    )


    print(
        "The published cohort counts will be "
        "kept as manuscript metadata, not as "
        "a computationally reproduced result."
    )


    cohort_counts = pd.Series(
        {
            "Control":
                6,

            "LN":
                19,

            "ANCA":
                32,

            "anti-GBM":
                6
        }
    )


    cohort_total = int(
        cohort_counts.sum()
    )


close_backed(
    big_ad
)


print(
    "\n" + "=" * 72
)

print(
    "FULL STUDY COHORT"
)

print(
    "=" * 72
)


display(
    cohort_counts
    .rename(
        "n_patients"
    )
    .to_frame()
)


print(
    "Total:",
    cohort_total
)


print(
    "Source:",
    (
        "reconstructed from large h5ad metadata"
        if full_cohort_derived
        else
        "published cohort metadata"
    )
)


# ============================================================
# 6. Load primary ROI object
# ============================================================

roi_ad = ad.read_h5ad(
    ROI_PATH
)


roi_meta = (
    roi_ad.obs.copy()
)


roi_meta.index = (
    roi_meta.index.astype(str)
)


print(
    "\n" + "=" * 72
)

print(
    "ROI OBJECT"
)

print(
    "=" * 72
)


print(
    "Shape:",
    roi_ad.shape
)


# ============================================================
# 7. ROI metadata columns
# ============================================================

required_roi_cols = [
    "Disease",
    "PC1_crescent",
    "Patient_Sample_ID"
]


missing_roi_cols = [
    c
    for c in required_roi_cols
    if c not in roi_meta.columns
]


if len(
    missing_roi_cols
) > 0:

    raise ValueError(
        "ROI object missing columns:\n"
        +
        "\n".join(
            missing_roi_cols
        )
    )


# ============================================================
# 8. ROI counts
# ============================================================

roi_counts = (
    roi_meta[
        "Disease"
    ]
    .value_counts()
    .reindex(
        [
            "Cntrl",
            "SLE",
            "ANCA",
            "GBM"
        ]
    )
)


print(
    "\nROI counts by disease:"
)


display(
    roi_counts
    .rename(
        "n_ROI"
    )
    .to_frame()
)


# ============================================================
# 9. ROI-analysis patient counts
# ============================================================

roi_patient_counts = (
    roi_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient_Sample_ID"
    ]
    .nunique()
)


print(
    "\nROI-analysis patients:"
)


display(
    roi_patient_counts
    .rename(
        "n_patients"
    )
    .to_frame()
)


print(
    "\nNOTE:"
)


print(
    "These counts are analytical ROI availability,"
)


print(
    "not the complete 63-sample study cohort."
)


# ============================================================
# 10. PC1 audit
# ============================================================

pc1 = (
    pd.to_numeric(
        roi_meta[
            "PC1_crescent"
        ],
        errors="coerce"
    )
)


if pc1.isna().any():

    raise ValueError(
        "PC1_crescent contains missing/non-numeric values."
    )


print(
    "\n" + "=" * 72
)

print(
    "PC1 RANGE AUDIT"
)

print(
    "=" * 72
)


pc1_ranges = (
    roi_meta
    .assign(
        PC1_numeric=pc1
    )
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_numeric"
    ]
    .agg(
        [
            "min",
            "max",
            "count"
        ]
    )
)


display(
    pc1_ranges
)


# ============================================================
# 11. Recalculate common PC1 support
#
# Main cross-etiology framework:
# ANCA + SLE/LN + anti-GBM
# ============================================================

diseases_for_common_support = [
    "ANCA",
    "SLE",
    "GBM"
]


support_table = (
    pc1_ranges
    .loc[
        diseases_for_common_support
    ]
)


common_low = float(
    support_table[
        "min"
    ].max()
)


common_high = float(
    support_table[
        "max"
    ].min()
)


print(
    "\nCalculated common PC1 support:"
)


print(
    f"{common_low:.9f} to {common_high:.9f}"
)


print(
    "\nExpected manuscript support:"
)


print(
    f"{EXPECTED['common_PC1_low']:.9f} "
    f"to "
    f"{EXPECTED['common_PC1_high']:.9f}"
)


common_low_diff = abs(
    common_low
    -
    EXPECTED[
        "common_PC1_low"
    ]
)


common_high_diff = abs(
    common_high
    -
    EXPECTED[
        "common_PC1_high"
    ]
)


common_support_pass = (
    common_low_diff
    <
    1e-5
    and
    common_high_diff
    <
    1e-5
)


print(
    "\nCommon support status:",
    (
        "PASS ✓"
        if common_support_pass
        else "CHECK ⚠"
    )
)


# ============================================================
# 12. DPT: first check whether the exact old DPT is saved
# ============================================================

possible_dpt_cols = [

    "dpt_pseudotime",

    "DPT",

    "dpt",

    "diffusion_pseudotime",

    "pseudotime"
]


saved_dpt_col = find_column(
    roi_meta.columns,
    possible_dpt_cols
)


dpt_source = None


# ============================================================
# 13. Use saved DPT when available
#
# This is preferred for exact manuscript reproduction.
# ============================================================

if saved_dpt_col is not None:

    print(
        "\nSaved DPT column detected:"
    )


    print(
        saved_dpt_col
    )


    dpt_values = (
        pd.to_numeric(
            roi_meta[
                saved_dpt_col
            ],
            errors="coerce"
        )
        .to_numpy()
    )


    dpt_source = (
        "saved ROI DPT"
    )


# ============================================================
# 14. Otherwise recompute DPT independently
# ============================================================

else:

    print(
        "\nNo saved DPT column found."
    )


    print(
        "Recomputing DPT from the 782 × 480 ROI object..."
    )


    dpt_ad = (
        roi_ad.copy()
    )


    # --------------------------------------------------------
    # PCA only if not already available
    # --------------------------------------------------------

    if "X_pca" not in dpt_ad.obsm:

        print(
            "Computing PCA..."
        )


        sc.pp.pca(
            dpt_ad,
            n_comps=min(
                30,
                dpt_ad.n_vars - 1
            )
        )


    else:

        print(
            "Using stored X_pca."
        )


    # --------------------------------------------------------
    # Rebuild neighbor graph
    # --------------------------------------------------------

    sc.pp.neighbors(
        dpt_ad,
        n_neighbors=15,
        use_rep="X_pca"
    )


    # --------------------------------------------------------
    # Diffusion map
    # --------------------------------------------------------

    sc.tl.diffmap(
        dpt_ad
    )


    # --------------------------------------------------------
    # Root = lowest crescent-associated PC1
    # --------------------------------------------------------

    pc1_array = (
        pd.to_numeric(
            dpt_ad.obs[
                "PC1_crescent"
            ],
            errors="coerce"
        )
        .to_numpy()
    )


    root_index = int(
        np.nanargmin(
            pc1_array
        )
    )


    dpt_ad.uns[
        "iroot"
    ] = root_index


    print(
        "DPT root ROI:"
    )


    print(
        dpt_ad.obs_names[
            root_index
        ]
    )


    print(
        "Root PC1:"
    )


    print(
        pc1_array[
            root_index
        ]
    )


    # --------------------------------------------------------
    # Calculate pseudotime
    # --------------------------------------------------------

    sc.tl.dpt(
        dpt_ad
    )


    dpt_values = (
        dpt_ad.obs[
            "dpt_pseudotime"
        ]
        .to_numpy(
            dtype=float
        )
    )


    dpt_source = (
        "recomputed DPT"
    )


# ============================================================
# 15. Spearman PC1 vs DPT
# ============================================================

pc1_array = (
    pc1.to_numpy(
        dtype=float
    )
)


valid = (
    np.isfinite(
        pc1_array
    )
    &
    np.isfinite(
        dpt_values
    )
)


rho, rho_p = (
    spearmanr(
        pc1_array[
            valid
        ],
        dpt_values[
            valid
        ]
    )
)


print(
    "\n" + "=" * 72
)

print(
    "PC1 vs DPT"
)

print(
    "=" * 72
)


print(
    "DPT source:",
    dpt_source
)


print(
    "N ROIs used:",
    int(
        valid.sum()
    )
)


print(
    "Reproduced Spearman rho =",
    rho
)


print(
    "Reproduced P =",
    rho_p
)


print(
    "\nExpected rho =",
    EXPECTED[
        "spearman_rho"
    ]
)


rho_diff = abs(
    rho
    -
    EXPECTED[
        "spearman_rho"
    ]
)


# For recomputed DPT, allow small numerical/environment differences.
rho_pass = (
    rho_diff
    <
    0.01
)


rho_exact_pass = (
    rho_diff
    <
    1e-6
)


print(
    "Absolute rho difference =",
    rho_diff
)


if rho_exact_pass:

    print(
        "Status: EXACT PASS ✓"
    )

elif rho_pass:

    print(
        "Status: NUMERICAL PASS ✓"
    )

else:

    print(
        "Status: CHECK ⚠"
    )


# ============================================================
# 16. Reproduction comparison table
# ============================================================

comparison_rows = []


# ------------------------------------------------------------
# Study cohort total
# ------------------------------------------------------------

comparison_rows.append({

    "metric":
        "Full study cohort",

    "expected":
        EXPECTED[
            "study_total"
        ],

    "reproduced":
        cohort_total,

    "absolute_difference":
        abs(
            cohort_total
            -
            EXPECTED[
                "study_total"
            ]
        ),

    "status":
        (
            "PASS"
            if cohort_total
            ==
            EXPECTED[
                "study_total"
            ]
            else
            "CHECK"
        )
})


# ------------------------------------------------------------
# Individual disease counts
# ------------------------------------------------------------

cohort_expected_map = {

    "Control":
        EXPECTED[
            "Control_patients"
        ],

    "LN":
        EXPECTED[
            "LN_patients"
        ],

    "ANCA":
        EXPECTED[
            "ANCA_patients"
        ],

    "anti-GBM":
        EXPECTED[
            "antiGBM_patients"
        ]
}


for disease, expected_count in (
    cohort_expected_map.items()
):

    reproduced_count = int(
        cohort_counts.get(
            disease,
            0
        )
    )


    comparison_rows.append({

        "metric":
            f"{disease} study patients",

        "expected":
            expected_count,

        "reproduced":
            reproduced_count,

        "absolute_difference":
            abs(
                reproduced_count
                -
                expected_count
            ),

        "status":
            (
                "PASS"
                if reproduced_count
                ==
                expected_count
                else
                "CHECK"
            )
    })


# ------------------------------------------------------------
# ROI count
# ------------------------------------------------------------

comparison_rows.append({

    "metric":
        "ROI count",

    "expected":
        EXPECTED[
            "n_ROI"
        ],

    "reproduced":
        roi_ad.n_obs,

    "absolute_difference":
        abs(
            roi_ad.n_obs
            -
            EXPECTED[
                "n_ROI"
            ]
        ),

    "status":
        (
            "PASS"
            if roi_ad.n_obs
            ==
            EXPECTED[
                "n_ROI"
            ]
            else
            "FAIL"
        )
})


# ------------------------------------------------------------
# Common low
# ------------------------------------------------------------

comparison_rows.append({

    "metric":
        "Common PC1 low",

    "expected":
        EXPECTED[
            "common_PC1_low"
        ],

    "reproduced":
        common_low,

    "absolute_difference":
        common_low_diff,

    "status":
        (
            "PASS"
            if common_low_diff
            <
            1e-5
            else
            "CHECK"
        )
})


# ------------------------------------------------------------
# Common high
# ------------------------------------------------------------

comparison_rows.append({

    "metric":
        "Common PC1 high",

    "expected":
        EXPECTED[
            "common_PC1_high"
        ],

    "reproduced":
        common_high,

    "absolute_difference":
        common_high_diff,

    "status":
        (
            "PASS"
            if common_high_diff
            <
            1e-5
            else
            "CHECK"
        )
})


# ------------------------------------------------------------
# Spearman
# ------------------------------------------------------------

comparison_rows.append({

    "metric":
        "PC1 vs DPT Spearman rho",

    "expected":
        EXPECTED[
            "spearman_rho"
        ],

    "reproduced":
        rho,

    "absolute_difference":
        rho_diff,

    "status":
        (
            "PASS"
            if rho_pass
            else
            "CHECK"
        )
})


cell2_check = pd.DataFrame(
    comparison_rows
)


print(
    "\n" + "=" * 72
)

print(
    "CELL 2 REPRODUCTION CHECK"
)

print(
    "=" * 72
)


display(
    cell2_check
)


# ============================================================
# 17. Save results
# ============================================================

check_path = (
    REPRO_RESULTS /
    "cell2_figure1_reproduction.csv"
)


cell2_check.to_csv(
    check_path,
    index=False
)


pc1_range_path = (
    REPRO_RESULTS /
    "cell2_PC1_ranges_by_disease.csv"
)


pc1_ranges.to_csv(
    pc1_range_path
)


dpt_table = pd.DataFrame({

    "ROI_ID":
        roi_meta.index,

    "Disease":
        roi_meta[
            "Disease"
        ].astype(str).to_numpy(),

    "PC1_crescent":
        pc1_array,

    "DPT":
        dpt_values
})


dpt_table_path = (
    REPRO_RESULTS /
    "cell2_PC1_DPT_values.csv"
)


dpt_table.to_csv(
    dpt_table_path,
    index=False
)


# ============================================================
# 18. PASS / CHECK decision
# ============================================================

n_fail = int(
    (
        cell2_check[
            "status"
        ]
        ==
        "FAIL"
    ).sum()
)


n_check = int(
    (
        cell2_check[
            "status"
        ]
        ==
        "CHECK"
    ).sum()
)


print(
    "\n" + "=" * 72
)


if n_fail > 0:

    print(
        "❌ CELL 2 FAILED"
    )


    print(
        "Structural reproduction failure detected."
    )


    print(
        "Do not proceed to Cell 3 until the issue above is resolved."
    )


elif n_check > 0:

    print(
        "⚠ CELL 2 COMPLETED WITH CHECK ITEMS"
    )


    print(
        "No structural reproduction failure was detected, but numerical checks require review."
    )


    print(
        "Review the CELL 2 REPRODUCTION CHECK output above before continuing."
    )


else:

    print(
        "✅ CELL 2 PASSED"
    )


    print(
        "Figure 1 核心数字已成功复现。"
    )


    print(
        "可以进入 Cell 3：Figure 2 composition reproduction。"
    )


print(
    "=" * 72
)


print(
    "\nSaved:"
)


print(
    check_path
)


print(
    pc1_range_path
)


print(
    dpt_table_path
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 3
#
# Figure 2 composition reproduction
#
# 从 ROI cell metadata 重新计算：
#
# 1. common-PC1-support ROIs
# 2. ROI-level broad cell fractions
# 3. patient-balanced spline models
# 4. shared PC1 effects
# 5. Disease × PC1 interactions
# 6. BH-FDR across 8 cell populations
# 7. compare against saved final Figure 2 statistics
#
# IMPORTANT:
# - 不读取最终 Figure 2 stats 来计算结果
# - saved Figure 2 stats 只在最后用于验证
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_ROOT = (
    BASE /
    "reproduction"
)

REPRO_RESULTS = (
    REPRO_ROOT /
    "results"
)

REPRO_FIGURES = (
    REPRO_ROOT /
    "figures"
)

REPRO_LOGS = (
    REPRO_ROOT /
    "logs"
)


for folder in [
    REPRO_ROOT,
    REPRO_RESULTS,
    REPRO_FIGURES,
    REPRO_LOGS
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


ROI_PATH = (
    BASE /
    "roi_782_PC1_primary.h5ad"
)


CELL_META_PATH = (
    BASE /
    "roi782_cell_metadata.pkl"
)


# ------------------------------------------------------------
# This file is ONLY used at the very end as a reference.
# It is NOT used to calculate the reproduced results.
# ------------------------------------------------------------

FINAL_STATS_PATH = (
    BASE /
    "Figure2_FINAL_composition_statistics_REVISED.csv"
)


# ============================================================
# 2. Expected rounded manuscript values
#
# These serve only as a fallback if the exact saved final
# statistics table is unavailable.
# ============================================================

EXPECTED_ROUNDED = {

    "MAC": {
        "shared_PC1_FDR": 9.2e-23,
        "interaction_FDR": 0.195
    },

    "Mono": {
        "shared_PC1_FDR": 4.3e-09,
        "interaction_FDR": 0.405
    },

    "B": {
        "shared_PC1_FDR": 1.4e-21,
        "interaction_FDR": 0.258
    },

    "T": {
        "shared_PC1_FDR": 3.7e-28,
        "interaction_FDR": 0.398
    },

    "FIB": {
        "shared_PC1_FDR": 1.7e-10,
        "interaction_FDR": 0.258
    },

    "EC": {
        "shared_PC1_FDR": 3.2e-57,
        "interaction_FDR": 0.398
    },

    "PEC": {
        "shared_PC1_FDR": 0.120,
        "interaction_FDR": 0.046
    },

    "POD": {
        "shared_PC1_FDR": 9.2e-23,
        "interaction_FDR": 0.258
    }
}


# ============================================================
# 3. Load source data
# ============================================================

print(
    "=" * 76
)

print(
    "CELL 3 — FIGURE 2 COMPOSITION REPRODUCTION"
)

print(
    "=" * 76
)


roi_ad = ad.read_h5ad(
    ROI_PATH
)


roi_meta = (
    roi_ad.obs.copy()
)


roi_meta.index = (
    roi_meta.index.astype(str)
)


cells = pd.read_pickle(
    CELL_META_PATH
)


print(
    "\nROI object:",
    roi_ad.shape
)


print(
    "Cell metadata:",
    cells.shape
)


# ============================================================
# 4. Required metadata checks
# ============================================================

required_roi_cols = [
    "Disease",
    "Patient_Sample_ID",
    "PC1_crescent"
]


missing_roi_cols = [
    c
    for c in required_roi_cols
    if c not in roi_meta.columns
]


if len(
    missing_roi_cols
) > 0:

    raise ValueError(
        "ROI object missing required columns:\n"
        +
        "\n".join(
            missing_roi_cols
        )
    )


required_cell_cols = [
    "roi_id",
    "celltype_l1"
]


missing_cell_cols = [
    c
    for c in required_cell_cols
    if c not in cells.columns
]


if len(
    missing_cell_cols
) > 0:

    raise ValueError(
        "Cell metadata missing required columns:\n"
        +
        "\n".join(
            missing_cell_cols
        )
    )


cells[
    "roi_id"
] = (
    cells[
        "roi_id"
    ]
    .astype(str)
)


cells[
    "celltype_l1"
] = (
    cells[
        "celltype_l1"
    ]
    .astype(str)
)


# ============================================================
# 5. Recalculate common PC1 support
#
# ANCA + SLE/LN + anti-GBM
# ============================================================

disease_order = [
    "ANCA",
    "SLE",
    "GBM"
]


disease_meta = (
    roi_meta[
        roi_meta[
            "Disease"
        ].isin(
            disease_order
        )
    ]
    .copy()
)


pc1_ranges = (
    disease_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        [
            "min",
            "max"
        ]
    )
)


common_low = float(
    pc1_ranges[
        "min"
    ].max()
)


common_high = float(
    pc1_ranges[
        "max"
    ].min()
)


analysis_meta = (
    disease_meta[
        disease_meta[
            "PC1_crescent"
        ].between(
            common_low,
            common_high
        )
    ]
    .copy()
)


analysis_meta.index = (
    analysis_meta.index.astype(str)
)


print(
    "\n" + "=" * 76
)

print(
    "COMMON SUPPORT"
)

print(
    "=" * 76
)


print(
    f"PC1: {common_low:.9f} to {common_high:.9f}"
)


print(
    "\nROI counts:"
)


display(
    analysis_meta[
        "Disease"
    ]
    .value_counts()
    .rename(
        "n_ROI"
    )
    .to_frame()
)


print(
    "\nPatient counts:"
)


display(
    analysis_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient_Sample_ID"
    ]
    .nunique()
    .rename(
        "n_patients"
    )
    .to_frame()
)


# ============================================================
# 6. Resolve broad cell-type labels
#
# We do not assume the exact stored naming.
# ============================================================

available_labels = set(
    cells[
        "celltype_l1"
    ]
    .dropna()
    .astype(str)
    .unique()
)


alias_map = {

    "MAC": [
        "MAC",
        "Macrophage"
    ],

    "Mono": [
        "Mono",
        "MONO",
        "Monocyte"
    ],

    "B": [
        "B",
        "B cell",
        "Bcell"
    ],

    "T": [
        "T",
        "T cell",
        "Tcell"
    ],

    "FIB": [
        "FIB",
        "Fibroblast"
    ],

    "EC": [
        "EC",
        "Endothelial"
    ],

    "PEC": [
        "PEC"
    ],

    "POD": [
        "POD",
        "Podocyte"
    ]
}


celltype_order = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]


resolved_labels = {}


for desired in celltype_order:

    aliases = (
        alias_map[
            desired
        ]
    )


    matches = [
        x
        for x in aliases
        if x in available_labels
    ]


    if len(
        matches
    ) == 0:

        print(
            "\nAvailable celltype_l1 labels:"
        )

        print(
            sorted(
                available_labels
            )
        )


        raise ValueError(
            f"""
Could not resolve cell type: {desired}

Candidate aliases:
{aliases}
"""
        )


    resolved_labels[
        desired
    ] = matches[
        0
    ]


print(
    "\n" + "=" * 76
)

print(
    "RESOLVED BROAD CELL TYPES"
)

print(
    "=" * 76
)


display(
    pd.DataFrame({
        "analysis_label":
            list(
                resolved_labels.keys()
            ),

        "stored_celltype_l1":
            list(
                resolved_labels.values()
            )
    })
)


# ============================================================
# 7. Restrict cell metadata to common-support ROIs
# ============================================================

analysis_roi_ids = set(
    analysis_meta.index
)


cells_use = (
    cells[
        cells[
            "roi_id"
        ].isin(
            analysis_roi_ids
        )
    ]
    .copy()
)


print(
    "\nCells in common-support ROIs:",
    len(
        cells_use
    )
)


# ============================================================
# 8. Total number of cells per ROI
#
# IMPORTANT:
# denominator = ALL annotated cells in that ROI,
# not only the eight plotted populations.
# ============================================================

total_by_roi = (
    cells_use
    .groupby(
        "roi_id",
        observed=True
    )
    .size()
    .rename(
        "total_cells"
    )
)


# ============================================================
# 9. Broad cell counts per ROI
# ============================================================

count_df = pd.DataFrame(
    index=analysis_meta.index
)


for desired in celltype_order:

    stored_label = (
        resolved_labels[
            desired
        ]
    )


    subtype_counts = (
        cells_use[
            cells_use[
                "celltype_l1"
            ]
            ==
            stored_label
        ]
        .groupby(
            "roi_id",
            observed=True
        )
        .size()
    )


    count_df[
        desired
    ] = (
        count_df.index
        .to_series()
        .map(
            subtype_counts
        )
        .fillna(
            0
        )
        .astype(float)
        .to_numpy()
    )


count_df[
    "total_cells"
] = (
    count_df.index
    .to_series()
    .map(
        total_by_roi
    )
    .fillna(
        0
    )
    .astype(float)
    .to_numpy()
)


zero_cell_rois = (
    count_df[
        "total_cells"
    ]
    <= 0
)


if zero_cell_rois.any():

    bad_rois = (
        count_df.index[
            zero_cell_rois
        ]
        .tolist()
    )


    raise ValueError(
        "Some analysis ROIs have no cell metadata:\n"
        +
        "\n".join(
            bad_rois[:20]
        )
    )


# ============================================================
# 10. Convert counts to ROI fractions
# ============================================================

fraction_df = (
    count_df[
        celltype_order
    ]
    .div(
        count_df[
            "total_cells"
        ],
        axis=0
    )
)


# sanity checks
fraction_sums = (
    fraction_df.sum(
        axis=1
    )
)


print(
    "\nFraction summary:"
)


print(
    "Minimum sum of 8 broad fractions:",
    float(
        fraction_sums.min()
    )
)


print(
    "Maximum sum of 8 broad fractions:",
    float(
        fraction_sums.max()
    )
)


if (
    fraction_sums
    >
    1.000001
).any():

    raise ValueError(
        "Broad cell fractions exceed 1 in at least one ROI."
    )


# ============================================================
# 11. Assemble composition analysis table
# ============================================================

composition = (
    fraction_df.copy()
)


composition[
    "Disease"
] = (
    analysis_meta[
        "Disease"
    ]
    .astype(str)
)


composition[
    "Patient"
] = (
    analysis_meta[
        "Patient_Sample_ID"
    ]
    .astype(str)
)


composition[
    "PC1"
] = (
    pd.to_numeric(
        analysis_meta[
            "PC1_crescent"
        ],
        errors="raise"
    )
)


composition[
    "total_cells"
] = (
    count_df[
        "total_cells"
    ]
)


# ============================================================
# 12. Save regenerated ROI-level composition table
# ============================================================

composition_path = (
    REPRO_RESULTS /
    "cell3_reproduced_ROI_cell_fractions.csv"
)


composition.to_csv(
    composition_path
)


print(
    "\nRegenerated ROI fraction table saved:"
)


print(
    composition_path
)


# ============================================================
# 13. Formal model function
#
# Model A:
# fraction ~ spline(PC1) + Disease
#
# Model B:
# fraction ~ spline(PC1) * Disease
#
# patient-balanced:
# each patient contributes total weight ~1
#
# covariance:
# clustered by patient
# ============================================================

def fit_composition_models(
    data,
    outcome
):

    d = (
        data[
            [
                outcome,
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .dropna()
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "ANCA",
            "SLE",
            "GBM"
        ]
    )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    n_roi_per_patient = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0
        /
        n_roi_per_patient
    )


    # --------------------------------------------------------
    # Shared progression model
    # --------------------------------------------------------

    shared_formula = (
        f"{outcome} ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "+ C(Disease)"
    )


    shared_fit = smf.wls(
        shared_formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------------------------------------
    # Joint test of all PC1 spline terms
    # --------------------------------------------------------

    shared_terms = [
        term
        for term
        in shared_fit.params.index
        if (
            "bs(PC1"
            in term
            and
            ":"
            not in term
        )
    ]


    if len(
        shared_terms
    ) == 0:

        raise ValueError(
            f"No shared PC1 spline terms found for {outcome}."
        )


    R_shared = np.zeros(
        (
            len(
                shared_terms
            ),
            len(
                shared_fit.params
            )
        )
    )


    for i, term in enumerate(
        shared_terms
    ):

        R_shared[
            i,
            shared_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    shared_p = float(
        shared_fit.wald_test(
            R_shared,
            scalar=True
        ).pvalue
    )


    # --------------------------------------------------------
    # Disease-specific progression model
    # --------------------------------------------------------

    interaction_formula = (
        f"{outcome} ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    )


    interaction_fit = smf.wls(
        interaction_formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------------------------------------
    # Joint disease × PC1 interaction test
    # --------------------------------------------------------

    interaction_terms = [
        term
        for term
        in interaction_fit.params.index
        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    if len(
        interaction_terms
    ) == 0:

        raise ValueError(
            f"No Disease × PC1 terms found for {outcome}."
        )


    R_interaction = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                interaction_fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R_interaction[
            i,
            interaction_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    interaction_p = float(
        interaction_fit.wald_test(
            R_interaction,
            scalar=True
        ).pvalue
    )


    # --------------------------------------------------------
    # Design matrix rank
    # --------------------------------------------------------

    shared_rank = int(
        np.linalg.matrix_rank(
            shared_fit.model.exog
        )
    )


    shared_n_columns = int(
        shared_fit.model.exog.shape[
            1
        ]
    )


    interaction_rank = int(
        np.linalg.matrix_rank(
            interaction_fit.model.exog
        )
    )


    interaction_n_columns = int(
        interaction_fit.model.exog.shape[
            1
        ]
    )


    return {

        "data":
            d,

        "shared_fit":
            shared_fit,

        "interaction_fit":
            interaction_fit,

        "shared_p":
            shared_p,

        "interaction_p":
            interaction_p,

        "shared_rank":
            shared_rank,

        "shared_columns":
            shared_n_columns,

        "interaction_rank":
            interaction_rank,

        "interaction_columns":
            interaction_n_columns
    }


# ============================================================
# 14. Refit all 8 cell populations
# ============================================================

result_rows = []

model_store = {}


print(
    "\n" + "=" * 76
)

print(
    "REFITTING 8 CELL POPULATIONS"
)

print(
    "=" * 76
)


for celltype in celltype_order:

    print(
        "Fitting",
        celltype,
        "..."
    )


    fit_result = (
        fit_composition_models(
            composition,
            celltype
        )
    )


    model_store[
        celltype
    ] = fit_result


    result_rows.append({

        "celltype":
            celltype,

        "shared_PC1_pvalue":
            fit_result[
                "shared_p"
            ],

        "interaction_pvalue":
            fit_result[
                "interaction_p"
            ],

        "shared_rank":
            fit_result[
                "shared_rank"
            ],

        "shared_columns":
            fit_result[
                "shared_columns"
            ],

        "interaction_rank":
            fit_result[
                "interaction_rank"
            ],

        "interaction_columns":
            fit_result[
                "interaction_columns"
            ],

        "n_ROI":
            len(
                fit_result[
                    "data"
                ]
            ),

        "n_patients":
            fit_result[
                "data"
            ][
                "Patient"
            ].nunique()
    })


reproduced_results = pd.DataFrame(
    result_rows
)


# ============================================================
# 15. BH-FDR across 8 cell populations
#
# Separate correction families:
# - shared PC1 effects
# - Disease × PC1 interactions
# ============================================================

reproduced_results[
    "shared_PC1_FDR"
] = multipletests(
    reproduced_results[
        "shared_PC1_pvalue"
    ],
    method="fdr_bh"
)[1]


reproduced_results[
    "interaction_FDR"
] = multipletests(
    reproduced_results[
        "interaction_pvalue"
    ],
    method="fdr_bh"
)[1]


reproduced_results = (
    reproduced_results
    .set_index(
        "celltype"
    )
    .loc[
        celltype_order
    ]
    .reset_index()
)


print(
    "\n" + "=" * 76
)

print(
    "REPRODUCED FIGURE 2 STATISTICS"
)

print(
    "=" * 76
)


display(
    reproduced_results
)


# ============================================================
# 16. Full-rank check
# ============================================================

rank_check = (
    reproduced_results[
        [
            "celltype",
            "shared_rank",
            "shared_columns",
            "interaction_rank",
            "interaction_columns"
        ]
    ]
    .copy()
)


rank_check[
    "shared_full_rank"
] = (
    rank_check[
        "shared_rank"
    ]
    ==
    rank_check[
        "shared_columns"
    ]
)


rank_check[
    "interaction_full_rank"
] = (
    rank_check[
        "interaction_rank"
    ]
    ==
    rank_check[
        "interaction_columns"
    ]
)


print(
    "\n" + "=" * 76
)

print(
    "MODEL RANK AUDIT"
)

print(
    "=" * 76
)


display(
    rank_check
)


rank_pass = bool(
    rank_check[
        "shared_full_rank"
    ].all()
    and
    rank_check[
        "interaction_full_rank"
    ].all()
)


# ============================================================
# 17. Save newly reproduced model results
# ============================================================

reproduced_stats_path = (
    REPRO_RESULTS /
    "cell3_figure2_reproduced_statistics.csv"
)


reproduced_results.to_csv(
    reproduced_stats_path,
    index=False
)


# ============================================================
# 18. Load final saved Figure 2 statistics ONLY FOR COMPARISON
# ============================================================

exact_reference_available = (
    FINAL_STATS_PATH.exists()
)


if exact_reference_available:

    print(
        "\nExact final Figure 2 statistics found:"
    )


    print(
        FINAL_STATS_PATH
    )


    reference_results = pd.read_csv(
        FINAL_STATS_PATH
    )


    required_reference_cols = [
        "celltype",
        "shared_PC1_pvalue",
        "interaction_pvalue",
        "shared_PC1_FDR",
        "interaction_FDR"
    ]


    missing_reference_cols = [
        c
        for c in required_reference_cols
        if c not in reference_results.columns
    ]


    if len(
        missing_reference_cols
    ) > 0:

        print(
            "\n⚠ Saved final stats file lacks expected columns."
        )


        print(
            missing_reference_cols
        )


        exact_reference_available = False


# ============================================================
# 19. Exact comparison when reference table exists
# ============================================================

comparison_rows = []


if exact_reference_available:

    reference_results = (
        reference_results
        .set_index(
            "celltype"
        )
    )


    reproduced_indexed = (
        reproduced_results
        .set_index(
            "celltype"
        )
    )


    metrics_to_compare = [
        "shared_PC1_pvalue",
        "interaction_pvalue",
        "shared_PC1_FDR",
        "interaction_FDR"
    ]


    for celltype in celltype_order:

        for metric in metrics_to_compare:

            expected_value = float(
                reference_results.loc[
                    celltype,
                    metric
                ]
            )


            reproduced_value = float(
                reproduced_indexed.loc[
                    celltype,
                    metric
                ]
            )


            absolute_difference = abs(
                reproduced_value
                -
                expected_value
            )


            # ------------------------------------------------
            # Relative difference for non-zero values
            # ------------------------------------------------

            denominator = max(
                abs(
                    expected_value
                ),
                1e-300
            )


            relative_difference = (
                absolute_difference
                /
                denominator
            )


            # Very tight tolerance:
            # exact same pipeline should usually reproduce
            # essentially identically.
            passed = (
                relative_difference
                <
                1e-6
                or
                absolute_difference
                <
                1e-12
            )


            comparison_rows.append({

                "celltype":
                    celltype,

                "metric":
                    metric,

                "expected":
                    expected_value,

                "reproduced":
                    reproduced_value,

                "absolute_difference":
                    absolute_difference,

                "relative_difference":
                    relative_difference,

                "status":
                    (
                        "PASS"
                        if passed
                        else
                        "CHECK"
                    )
            })


# ============================================================
# 20. Fallback comparison to rounded manuscript values
# ============================================================

else:

    print(
        "\nExact final statistics table unavailable."
    )


    print(
        "Using rounded manuscript values for validation."
    )


    reproduced_indexed = (
        reproduced_results
        .set_index(
            "celltype"
        )
    )


    for celltype in celltype_order:

        for metric in [
            "shared_PC1_FDR",
            "interaction_FDR"
        ]:

            expected_value = float(
                EXPECTED_ROUNDED[
                    celltype
                ][
                    metric
                ]
            )


            reproduced_value = float(
                reproduced_indexed.loc[
                    celltype,
                    metric
                ]
            )


            # ------------------------------------------------
            # Rounded values require looser comparison.
            # Compare formatted magnitude rather than exact.
            # ------------------------------------------------

            if expected_value < 0.001:

                log_difference = abs(
                    np.log10(
                        reproduced_value
                    )
                    -
                    np.log10(
                        expected_value
                    )
                )


                passed = (
                    log_difference
                    <
                    0.08
                )


                diagnostic_difference = (
                    log_difference
                )


            else:

                absolute_difference = abs(
                    reproduced_value
                    -
                    expected_value
                )


                passed = (
                    absolute_difference
                    <
                    0.005
                )


                diagnostic_difference = (
                    absolute_difference
                )


            comparison_rows.append({

                "celltype":
                    celltype,

                "metric":
                    metric,

                "expected":
                    expected_value,

                "reproduced":
                    reproduced_value,

                "absolute_difference":
                    abs(
                        reproduced_value
                        -
                        expected_value
                    ),

                "relative_difference":
                    diagnostic_difference,

                "status":
                    (
                        "PASS"
                        if passed
                        else
                        "CHECK"
                    )
            })


# ============================================================
# 21. Build comparison table
# ============================================================

comparison_df = pd.DataFrame(
    comparison_rows
)


print(
    "\n" + "=" * 76
)

print(
    "CELL 3 FIGURE 2 REPRODUCTION CHECK"
)

print(
    "=" * 76
)


display(
    comparison_df
)


comparison_path = (
    REPRO_RESULTS /
    "cell3_figure2_reproduction_check.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


# ============================================================
# 22. Compact manuscript-level summary
# ============================================================

summary_rows = []


for celltype in celltype_order:

    row = (
        reproduced_results[
            reproduced_results[
                "celltype"
            ]
            ==
            celltype
        ]
        .iloc[
            0
        ]
    )


    summary_rows.append({

        "celltype":
            celltype,

        "shared_PC1_FDR":
            float(
                row[
                    "shared_PC1_FDR"
                ]
            ),

        "Disease_x_PC1_FDR":
            float(
                row[
                    "interaction_FDR"
                ]
            ),

        "shared_significant_FDR005":
            bool(
                row[
                    "shared_PC1_FDR"
                ]
                <
                0.05
            ),

        "interaction_significant_FDR005":
            bool(
                row[
                    "interaction_FDR"
                ]
                <
                0.05
            )
    })


compact_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n" + "=" * 76
)

print(
    "MANUSCRIPT-LEVEL FIGURE 2 SUMMARY"
)

print(
    "=" * 76
)


display(
    compact_summary
)


summary_path = (
    REPRO_RESULTS /
    "cell3_figure2_manuscript_summary.csv"
)


compact_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 23. Explicit key-result audit
# ============================================================

n_shared_sig = int(
    (
        compact_summary[
            "shared_significant_FDR005"
        ]
    ).sum()
)


n_interaction_sig = int(
    (
        compact_summary[
            "interaction_significant_FDR005"
        ]
    ).sum()
)


significant_interactions = (
    compact_summary.loc[
        compact_summary[
            "interaction_significant_FDR005"
        ],
        "celltype"
    ]
    .tolist()
)


print(
    "\nShared PC1 significant populations:",
    n_shared_sig,
    "/",
    len(
        celltype_order
    )
)


print(
    "Disease × PC1 significant populations:",
    n_interaction_sig,
    "/",
    len(
        celltype_order
    )
)


print(
    "Significant interaction populations:",
    significant_interactions
)


# ============================================================
# 24. PASS / CHECK decision
# ============================================================

n_numeric_checks = int(
    (
        comparison_df[
            "status"
        ]
        ==
        "CHECK"
    ).sum()
)


expected_interaction_pattern_pass = (
    significant_interactions
    ==
    [
        "PEC"
    ]
)


print(
    "\n" + "=" * 76
)


if (
    not rank_pass
):

    print(
        "❌ CELL 3 FAILED — NON-FULL-RANK MODEL"
    )


    print(
        "Do not proceed to Cell 4 until the numeric check items above are resolved."
    )


elif (
    n_numeric_checks
    >
    0
):

    print(
        "⚠ CELL 3 COMPLETED WITH NUMERIC CHECK ITEMS"
    )


    print(
        f"{n_numeric_checks} comparison rows require review."
    )


    print(
        "Review the comparison table above before continuing."
    )


elif (
    not expected_interaction_pattern_pass
):

    print(
        "⚠ CELL 3 RESULTS REPRODUCED NUMERICALLY,"
    )


    print(
        "but the significant-interaction pattern "
        "differs from the expected manuscript pattern."
    )


else:

    print(
        "✅ CELL 3 PASSED"
    )


    print(
        "Figure 2 composition results were independently reproduced."
    )


    print(
        "Expected biological/statistical pattern confirmed:"
    )


    print(
        "  • broad shared progression effects"
    )


    print(
        "  • PEC is the only FDR<0.05 Disease×PC1 interaction"
    )


    print(
        "可以进入 Cell 4：Figure 3 molecular reproduction。"
    )


print(
    "=" * 76
)


print(
    "\nSaved:"
)


print(
    composition_path
)


print(
    reproduced_stats_path
)


print(
    comparison_path
)


print(
    summary_path
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 4
#
# FIGURE 3 MOLECULAR REPRODUCTION
#
# Independent calculations:
#   1. Re-normalize ROI × cell-type expression profiles
#   2. Refit MAC/FIB gene-level Disease × PC1 trajectories
#   3. Recalculate global gene-level BH-FDR
#   4. Audit frozen module assignments
#   5. Recalculate frozen module scores from expression
#   6. Refit module Disease × PC1 models
#   7. Refit Xenium-slide sensitivity models
#   8. Compare against saved final results
#
# Important:
# - saved gene results are used ONLY at the comparison stage
# - saved module scores are used ONLY at the comparison stage
# - frozen module membership is treated as prespecified
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import warnings
import time

import anndata as ad
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_ROOT = (
    BASE /
    "reproduction"
)

REPRO_RESULTS = (
    REPRO_ROOT /
    "results"
)

REPRO_FIGURES = (
    REPRO_ROOT /
    "figures"
)

REPRO_LOGS = (
    REPRO_ROOT /
    "logs"
)


for folder in [
    REPRO_ROOT,
    REPRO_RESULTS,
    REPRO_FIGURES,
    REPRO_LOGS
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


ROI_PATH = (
    BASE /
    "roi_782_PC1_primary.h5ad"
)


EXPR_PATH = (
    BASE /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)


SAVED_GENE_RESULT_PATH = (
    BASE /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)


MODULE_PATH = (
    BASE /
    "figure5_MAC_FIB_trajectory_modules.csv"
)


SAVED_SCORE_PATH = (
    BASE /
    "figure5_frozen_module_scores.csv"
)


SAVED_SLIDE_RESULT_PATH = (
    BASE /
    "figure5_slide_adjusted_module_sensitivity.csv"
)


PATHWAY_PATH = (
    BASE /
    "figure5_maintext_pathway_enrichment.csv"
)


required_paths = [
    ROI_PATH,
    EXPR_PATH,
    SAVED_GENE_RESULT_PATH,
    MODULE_PATH,
    SAVED_SCORE_PATH,
    SAVED_SLIDE_RESULT_PATH,
    PATHWAY_PATH
]


for p in required_paths:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required file:\n{p}"
        )


# ============================================================
# 2. Start
# ============================================================

start_time = time.time()


print(
    "=" * 80
)

print(
    "CELL 4 — FIGURE 3 MOLECULAR REPRODUCTION"
)

print(
    "=" * 80
)


print(
    "\nNOTE:"
)

print(
    "This cell refits ~1,000 gene-level models."
)

print(
    "Several minutes of runtime can be normal."
)


# ============================================================
# 3. Load ROI × cell-type expression table
# ============================================================

expr_df = pd.read_csv(
    EXPR_PATH
)


print(
    "\nExpression table shape:",
    expr_df.shape
)


print(
    "\nFirst metadata columns:"
)

print(
    expr_df.columns[
        :12
    ].tolist()
)


# ============================================================
# 4. Identify metadata and gene columns
# ============================================================

known_metadata_cols = [

    "roi_id",

    "celltype_l1",

    "n_cells",

    "Disease",

    "Patient",

    "PC1",

    "Slide"
]


metadata_cols = [
    c
    for c in known_metadata_cols
    if c in expr_df.columns
]


required_expr_metadata = [
    "roi_id",
    "celltype_l1",
    "Disease",
    "Patient",
    "PC1"
]


missing_metadata = [
    c
    for c in required_expr_metadata
    if c not in expr_df.columns
]


if len(
    missing_metadata
) > 0:

    raise ValueError(
        "Expression table is missing metadata columns:\n"
        +
        "\n".join(
            missing_metadata
        )
    )


gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "\nDetected gene columns:",
    len(
        gene_cols
    )
)


print(
    "Expected targeted-panel size: approximately 480"
)


print(
    "\nProfiles by cell type:"
)


display(
    expr_df[
        "celltype_l1"
    ]
    .value_counts()
    .rename(
        "n_profiles"
    )
    .to_frame()
)


# ============================================================
# 5. Basic profile audit
# ============================================================

profile_audit = (
    expr_df
    .groupby(
        [
            "celltype_l1",
            "Disease"
        ],
        observed=True
    )
    .agg(
        n_ROI=(
            "roi_id",
            "size"
        ),

        n_patients=(
            "Patient",
            "nunique"
        )
    )
)


print(
    "\nROI profile audit:"
)


display(
    profile_audit
)


# ============================================================
# 6. Re-normalize expression FROM COUNTS
#
# Same transformation used for final Figure 3:
#
# raw ROI × celltype mean-count vector
#       ↓
# divide by row sum
#       ↓
# multiply by 10,000
#       ↓
# log1p
#
# Saved final gene statistics are NOT used here.
# ============================================================

X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


if not np.all(
    np.isfinite(
        X_raw
    )
):

    raise ValueError(
        "Expression table contains non-finite gene values."
    )


if (
    X_raw < 0
).any():

    raise ValueError(
        "Raw mean-count table contains negative values."
    )


row_sum = (
    X_raw.sum(
        axis=1
    )
)


if (
    row_sum <= 0
).any():

    bad = np.where(
        row_sum <= 0
    )[0]


    raise ValueError(
        f"{len(bad)} profiles have total count <= 0."
    )


X_norm = (
    X_raw
    /
    row_sum[
        :,
        None
    ]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


print(
    "\nExpression normalization complete."
)


print(
    "X_log shape:",
    X_log.shape
)


# ============================================================
# 7. Save normalized-expression audit
# ============================================================

normalization_audit = pd.DataFrame({

    "metric": [
        "n_profiles",
        "n_genes",
        "raw_min",
        "raw_max",
        "row_sum_min",
        "row_sum_median",
        "row_sum_max",
        "log_min",
        "log_max"
    ],

    "value": [
        X_raw.shape[
            0
        ],

        X_raw.shape[
            1
        ],

        float(
            X_raw.min()
        ),

        float(
            X_raw.max()
        ),

        float(
            row_sum.min()
        ),

        float(
            np.median(
                row_sum
            )
        ),

        float(
            row_sum.max()
        ),

        float(
            X_log.min()
        ),

        float(
            X_log.max()
        )
    ]
})


display(
    normalization_audit
)


normalization_audit.to_csv(
    REPRO_RESULTS /
    "cell4_expression_normalization_audit.csv",
    index=False
)


# ============================================================
# 8. Gene-level model helper
#
# Expression ~ spline(PC1) * Disease
#
# Disease:
#   reference = SLE / LN
#   comparison = GBM / anti-GBM
#
# Patient-balanced WLS
# Patient-clustered covariance
#
# Formal statistic:
# joint Wald test of all Disease × PC1 spline terms
# ============================================================

def fit_gene_trajectory(
    metadata,
    y
):

    d = (
        metadata[
            [
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    d[
        "y"
    ] = np.asarray(
        y,
        dtype=float
    )


    d = (
        d.replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .dropna()
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),

        categories=[
            "SLE",
            "GBM"
        ]
    )


    # --------------------------------------------------------
    # Drop rows outside expected comparison
    # --------------------------------------------------------

    d = (
        d[
            d[
                "Disease"
            ].notna()
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0
        /
        n_roi
    )


    formula = (
        "y ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    )


    try:

        fit = smf.wls(
            formula,
            data=d,
            weights=d[
                "patient_weight"
            ]
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                    d[
                        "Patient"
                    ]
            }
        )


    except Exception as e:

        return {

            "pvalue":
                np.nan,

            "matrix_rank":
                np.nan,

            "n_columns":
                np.nan,

            "n_ROI":
                len(
                    d
                ),

            "n_patients":
                d[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d.loc[
                    d[
                        "Disease"
                    ].astype(str)
                    ==
                    "GBM",
                    "Patient"
                ].nunique(),

            "fit_status":
                "MODEL_ERROR",

            "error":
                str(
                    e
                )
        }


    matrix_rank = int(
        np.linalg.matrix_rank(
            fit.model.exog
        )
    )


    n_columns = int(
        fit.model.exog.shape[
            1
        ]
    )


    interaction_terms = [
        term
        for term
        in fit.params.index
        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    if (
        matrix_rank
        !=
        n_columns
    ):

        return {

            "pvalue":
                np.nan,

            "matrix_rank":
                matrix_rank,

            "n_columns":
                n_columns,

            "n_ROI":
                len(
                    d
                ),

            "n_patients":
                d[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d.loc[
                    d[
                        "Disease"
                    ].astype(str)
                    ==
                    "GBM",
                    "Patient"
                ].nunique(),

            "fit_status":
                "NON_FULL_RANK",

            "error":
                ""
        }


    if len(
        interaction_terms
    ) == 0:

        return {

            "pvalue":
                np.nan,

            "matrix_rank":
                matrix_rank,

            "n_columns":
                n_columns,

            "n_ROI":
                len(
                    d
                ),

            "n_patients":
                d[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d.loc[
                    d[
                        "Disease"
                    ].astype(str)
                    ==
                    "GBM",
                    "Patient"
                ].nunique(),

            "fit_status":
                "NO_INTERACTION_TERMS",

            "error":
                ""
        }


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    try:

        pvalue = float(
            fit.wald_test(
                R,
                scalar=True
            ).pvalue
        )


    except Exception as e:

        return {

            "pvalue":
                np.nan,

            "matrix_rank":
                matrix_rank,

            "n_columns":
                n_columns,

            "n_ROI":
                len(
                    d
                ),

            "n_patients":
                d[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d.loc[
                    d[
                        "Disease"
                    ].astype(str)
                    ==
                    "GBM",
                    "Patient"
                ].nunique(),

            "fit_status":
                "WALD_ERROR",

            "error":
                str(
                    e
                )
        }


    return {

        "pvalue":
            pvalue,

        "matrix_rank":
            matrix_rank,

        "n_columns":
            n_columns,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                ==
                "GBM",
                "Patient"
            ].nunique(),

        "fit_status":
            "OK",

        "error":
            ""
    }


# ============================================================
# 9. Refit ALL MAC/FIB gene trajectories
# ============================================================

gene_result_rows = []


print(
    "\n" + "=" * 80
)

print(
    "REFITTING GENE-LEVEL TRAJECTORIES"
)

print(
    "=" * 80
)


total_models = 0


for celltype in [
    "MAC",
    "FIB"
]:

    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    metadata_ct = (
        expr_df.loc[
            mask,
            [
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    X_ct = (
        X_log[
            mask,
            :
        ]
    )


    print(
        f"\n{celltype}:"
    )


    print(
        "  profiles =",
        X_ct.shape[
            0
        ]
    )


    print(
        "  genes =",
        X_ct.shape[
            1
        ]
    )


    for gene_i, gene in enumerate(
        gene_cols
    ):

        result = fit_gene_trajectory(
            metadata_ct,
            X_ct[
                :,
                gene_i
            ]
        )


        detection_fraction = float(
            np.mean(
                X_raw[
                    mask,
                    gene_i
                ]
                >
                0
            )
        )


        gene_result_rows.append({

            "celltype":
                celltype,

            "gene":
                gene,

            "pvalue":
                result[
                    "pvalue"
                ],

            "matrix_rank":
                result[
                    "matrix_rank"
                ],

            "n_columns":
                result[
                    "n_columns"
                ],

            "n_ROI":
                result[
                    "n_ROI"
                ],

            "n_patients":
                result[
                    "n_patients"
                ],

            "n_GBM_patients":
                result[
                    "n_GBM_patients"
                ],

            "detection_fraction":
                detection_fraction,

            "fit_status":
                result[
                    "fit_status"
                ],

            "error":
                result[
                    "error"
                ]
        })


        total_models += 1


        # ----------------------------------------------------
        # Progress output every 50 genes
        # ----------------------------------------------------

        if (
            (
                gene_i
                +
                1
            )
            %
            50
            ==
            0
        ):

            print(
                f"  processed "
                f"{gene_i + 1} / "
                f"{len(gene_cols)} genes"
            )


reproduced_gene_results = pd.DataFrame(
    gene_result_rows
)


print(
    "\nTotal gene models fitted:",
    total_models
)


# ============================================================
# 10. Gene-model status audit
# ============================================================

print(
    "\nGene fit-status summary:"
)


display(
    reproduced_gene_results[
        "fit_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "n_models"
    )
    .to_frame()
)


# ============================================================
# 11. Global BH-FDR
#
# Same family:
# all estimable MAC + FIB gene-level tests together
# ============================================================

valid_gene_tests = (
    reproduced_gene_results[
        "pvalue"
    ].notna()
)


reproduced_gene_results[
    "FDR_global"
] = np.nan


if valid_gene_tests.sum() > 0:

    reproduced_gene_results.loc[
        valid_gene_tests,
        "FDR_global"
    ] = multipletests(
        reproduced_gene_results.loc[
            valid_gene_tests,
            "pvalue"
        ],
        method="fdr_bh"
    )[1]


# ============================================================
# 12. Significant gene counts
# ============================================================

significant_gene_counts = (
    reproduced_gene_results[
        reproduced_gene_results[
            "FDR_global"
        ]
        <
        0.05
    ]
    .groupby(
        "celltype",
        observed=True
    )
    .size()
    .reindex(
        [
            "MAC",
            "FIB"
        ]
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


print(
    "\n" + "=" * 80
)

print(
    "GLOBAL FDR < 0.05 GENE COUNTS"
)

print(
    "=" * 80
)


display(
    significant_gene_counts
    .rename(
        "n_significant_genes"
    )
    .to_frame()
)


expected_sig_gene_counts = pd.Series(
    {
        "MAC":
            179,

        "FIB":
            164
    }
)


sig_gene_count_check = pd.DataFrame({

    "celltype": [
        "MAC",
        "FIB"
    ],

    "expected": [
        179,
        164
    ],

    "reproduced": [
        int(
            significant_gene_counts[
                "MAC"
            ]
        ),

        int(
            significant_gene_counts[
                "FIB"
            ]
        )
    ]
})


sig_gene_count_check[
    "status"
] = np.where(

    sig_gene_count_check[
        "expected"
    ]
    ==
    sig_gene_count_check[
        "reproduced"
    ],

    "PASS",

    "CHECK"
)


print(
    "\nSignificant-gene count check:"
)


display(
    sig_gene_count_check
)


# ============================================================
# 13. Save independently reproduced gene results
# ============================================================

reproduced_gene_path = (
    REPRO_RESULTS /
    "cell4_reproduced_MAC_FIB_gene_trajectory_results.csv"
)


reproduced_gene_results.to_csv(
    reproduced_gene_path,
    index=False
)


# ============================================================
# 14. Compare reproduced gene results to saved final table
#
# Saved results are loaded NOW — after independent fitting.
# ============================================================

saved_gene_results = pd.read_csv(
    SAVED_GENE_RESULT_PATH
)


print(
    "\nSaved gene-result columns:"
)


print(
    saved_gene_results.columns.tolist()
)


# ------------------------------------------------------------
# Detect cell-type column
# ------------------------------------------------------------

if "celltype" in saved_gene_results.columns:

    saved_celltype_col = (
        "celltype"
    )

elif "celltype_l1" in saved_gene_results.columns:

    saved_celltype_col = (
        "celltype_l1"
    )

else:

    raise ValueError(
        "Cannot find cell-type column "
        "in saved gene trajectory table."
    )


if "gene" not in saved_gene_results.columns:

    raise ValueError(
        "Cannot find gene column "
        "in saved gene trajectory table."
    )


if "pvalue" not in saved_gene_results.columns:

    raise ValueError(
        "Cannot find pvalue column "
        "in saved gene trajectory table."
    )


# ------------------------------------------------------------
# Detect saved global FDR column
# ------------------------------------------------------------

saved_fdr_candidates = [
    "FDR_global",
    "global_FDR",
    "FDR"
]


saved_fdr_col = None


for c in saved_fdr_candidates:

    if c in saved_gene_results.columns:

        saved_fdr_col = c

        break


print(
    "\nDetected saved global FDR column:",
    saved_fdr_col
)


# ============================================================
# 15. Merge gene results
# ============================================================

saved_compare_cols = [
    saved_celltype_col,
    "gene",
    "pvalue"
]


if saved_fdr_col is not None:

    saved_compare_cols.append(
        saved_fdr_col
    )


saved_for_compare = (
    saved_gene_results[
        saved_compare_cols
    ]
    .copy()
)


saved_for_compare = (
    saved_for_compare.rename(
        columns={
            saved_celltype_col:
                "celltype",

            "pvalue":
                "saved_pvalue"
        }
    )
)


if saved_fdr_col is not None:

    saved_for_compare = (
        saved_for_compare.rename(
            columns={
                saved_fdr_col:
                    "saved_FDR_global"
            }
        )
    )


gene_compare = (
    reproduced_gene_results[
        [
            "celltype",
            "gene",
            "pvalue",
            "FDR_global",
            "matrix_rank",
            "n_columns",
            "fit_status"
        ]
    ]
    .merge(
        saved_for_compare,
        on=[
            "celltype",
            "gene"
        ],
        how="inner"
    )
)


print(
    "\nGene-result comparison rows:",
    len(
        gene_compare
    )
)


# ============================================================
# 16. Exact p-value comparison
#
# For very small P values, compare in log10 space.
# ============================================================

def numeric_match(
    reproduced,
    expected
):

    if (
        not np.isfinite(
            reproduced
        )
        or
        not np.isfinite(
            expected
        )
    ):

        return False


    if (
        reproduced
        ==
        expected
    ):

        return True


    # Very small quantities:
    # compare log10 scale
    if (
        reproduced
        >
        0
        and
        expected
        >
        0
        and
        (
            reproduced
            <
            1e-6
            or
            expected
            <
            1e-6
        )
    ):

        return (
            abs(
                np.log10(
                    reproduced
                )
                -
                np.log10(
                    expected
                )
            )
            <
            1e-5
        )


    # Ordinary values
    return (
        abs(
            reproduced
            -
            expected
        )
        <
        1e-8
    )


gene_compare[
    "pvalue_match"
] = [

    numeric_match(
        reproduced,
        expected
    )

    for reproduced, expected
    in zip(
        gene_compare[
            "pvalue"
        ],

        gene_compare[
            "saved_pvalue"
        ]
    )
]


if (
    saved_fdr_col
    is not None
):

    gene_compare[
        "FDR_match"
    ] = [

        numeric_match(
            reproduced,
            expected
        )

        for reproduced, expected
        in zip(
            gene_compare[
                "FDR_global"
            ],

            gene_compare[
                "saved_FDR_global"
            ]
        )
    ]


else:

    gene_compare[
        "FDR_match"
    ] = (
        True
    )


gene_comparison_summary = pd.DataFrame({

    "metric": [
        "Merged gene tests",
        "P-value matches",
        "Global FDR matches"
    ],

    "n": [
        len(
            gene_compare
        ),

        int(
            gene_compare[
                "pvalue_match"
            ].sum()
        ),

        int(
            gene_compare[
                "FDR_match"
            ].sum()
        )
    ],

    "total": [
        len(
            gene_compare
        ),

        len(
            gene_compare
        ),

        len(
            gene_compare
        )
    ]
})


gene_comparison_summary[
    "fraction"
] = (
    gene_comparison_summary[
        "n"
    ]
    /
    gene_comparison_summary[
        "total"
    ]
)


print(
    "\n" + "=" * 80
)

print(
    "GENE-LEVEL REPRODUCTION COMPARISON"
)

print(
    "=" * 80
)


display(
    gene_comparison_summary
)


# ============================================================
# 17. Save detailed gene comparison
# ============================================================

gene_compare_path = (
    REPRO_RESULTS /
    "cell4_gene_level_comparison_to_saved.csv"
)


gene_compare.to_csv(
    gene_compare_path,
    index=False
)


# ============================================================
# 18. Load frozen module assignments
# ============================================================

module_df = pd.read_csv(
    MODULE_PATH
)


required_module_cols = [
    "celltype",
    "module",
    "gene"
]


missing_module_cols = [
    c
    for c in required_module_cols
    if c not in module_df.columns
]


if len(
    missing_module_cols
) > 0:

    raise ValueError(
        "Module table missing columns:\n"
        +
        "\n".join(
            missing_module_cols
        )
    )


module_df[
    "celltype"
] = (
    module_df[
        "celltype"
    ].astype(str)
)


module_df[
    "module"
] = (
    pd.to_numeric(
        module_df[
            "module"
        ],
        errors="raise"
    )
    .astype(int)
)


module_df[
    "gene"
] = (
    module_df[
        "gene"
    ].astype(str)
)


module_sizes = (
    module_df
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_genes"
    )
    .reset_index()
)


print(
    "\n" + "=" * 80
)

print(
    "FROZEN MODULE SIZE AUDIT"
)

print(
    "=" * 80
)


display(
    module_sizes
)


expected_module_sizes = pd.DataFrame({

    "celltype": [
        "MAC",
        "MAC",
        "MAC",
        "FIB",
        "FIB",
        "FIB"
    ],

    "module": [
        1,
        2,
        3,
        1,
        2,
        3
    ],

    "expected_n_genes": [
        47,
        16,
        34,
        23,
        8,
        29
    ]
})


module_size_check = (
    expected_module_sizes
    .merge(
        module_sizes,
        on=[
            "celltype",
            "module"
        ],
        how="left"
    )
)


module_size_check[
    "status"
] = np.where(

    module_size_check[
        "expected_n_genes"
    ]
    ==
    module_size_check[
        "n_genes"
    ],

    "PASS",

    "CHECK"
)


print(
    "\nModule-size check:"
)


display(
    module_size_check
)


# ============================================================
# 19. Recalculate frozen module scores FROM EXPRESSION
#
# For each cell type:
#
# 1. take normalized log expression
# 2. standardize each module gene across ROI profiles
# 3. module score = mean gene z-score
#
# ddof=0 is used here.
#
# Even if an earlier implementation used ddof=1,
# that changes only a common scale factor within a cell type
# and does NOT change the formal module P value.
# ============================================================

gene_to_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


module_score_rows = []


for celltype in [
    "MAC",
    "FIB"
]:

    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    expr_meta_ct = (
        expr_df.loc[
            mask,
            [
                c
                for c in [
                    "roi_id",
                    "celltype_l1",
                    "Disease",
                    "Patient",
                    "PC1"
                ]
                if c in expr_df.columns
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    X_ct = (
        X_log[
            mask,
            :
        ]
    )


    modules_here = sorted(
        module_df.loc[
            module_df[
                "celltype"
            ]
            ==
            celltype,
            "module"
        ]
        .unique()
    )


    for module_number in modules_here:

        module_genes = (
            module_df.loc[
                (
                    module_df[
                        "celltype"
                    ]
                    ==
                    celltype
                )
                &
                (
                    module_df[
                        "module"
                    ]
                    ==
                    module_number
                ),
                "gene"
            ]
            .astype(str)
            .tolist()
        )


        present_genes = [
            g
            for g in module_genes
            if g in gene_to_index
        ]


        missing_genes = [
            g
            for g in module_genes
            if g not in gene_to_index
        ]


        if len(
            missing_genes
        ) > 0:

            raise ValueError(
                f"""
Module genes missing from expression table:

{celltype} M{module_number}

{missing_genes}
"""
            )


        indices = [
            gene_to_index[
                g
            ]
            for g in present_genes
        ]


        values = (
            X_ct[
                :,
                indices
            ]
        )


        gene_mean = (
            np.mean(
                values,
                axis=0
            )
        )


        gene_std = (
            np.std(
                values,
                axis=0,
                ddof=0
            )
        )


        usable = (
            gene_std
            >
            1e-12
        )


        if usable.sum() == 0:

            raise ValueError(
                f"No usable genes for "
                f"{celltype} M{module_number}"
            )


        Z = (
            values[
                :,
                usable
            ]
            -
            gene_mean[
                usable
            ][
                None,
                :
            ]
        ) / (
            gene_std[
                usable
            ][
                None,
                :
            ]
        )


        module_score = (
            np.mean(
                Z,
                axis=1
            )
        )


        for i in range(
            len(
                expr_meta_ct
            )
        ):

            module_score_rows.append({

                "roi_id":
                    str(
                        expr_meta_ct.loc[
                            i,
                            "roi_id"
                        ]
                    ),

                "celltype":
                    celltype,

                "module":
                    int(
                        module_number
                    ),

                "module_label":
                    (
                        f"{celltype} M"
                        f"{int(module_number)}"
                    ),

                "Disease":
                    str(
                        expr_meta_ct.loc[
                            i,
                            "Disease"
                        ]
                    ),

                "Patient":
                    str(
                        expr_meta_ct.loc[
                            i,
                            "Patient"
                        ]
                    ),

                "PC1":
                    float(
                        expr_meta_ct.loc[
                            i,
                            "PC1"
                        ]
                    ),

                "module_score":
                    float(
                        module_score[
                            i
                        ]
                    ),

                "n_module_genes":
                    len(
                        present_genes
                    ),

                "n_usable_genes":
                    int(
                        usable.sum()
                    )
            })


reproduced_module_scores = pd.DataFrame(
    module_score_rows
)


print(
    "\nReproduced module-score table:"
)


print(
    reproduced_module_scores.shape
)


print(
    "\nRows by module:"
)


display(
    reproduced_module_scores[
        "module_label"
    ]
    .value_counts()
    .rename(
        "n_rows"
    )
    .to_frame()
)


# ============================================================
# 20. Reconstruct Xenium slide ID
# ============================================================

roi_ad2 = ad.read_h5ad(
    ROI_PATH
)


roi_meta2 = (
    roi_ad2.obs.copy()
)


roi_meta2.index = (
    roi_meta2.index.astype(str)
)


if "Biopsy_ID" in roi_meta2.columns:

    roi_meta2[
        "Slide"
    ] = (
        roi_meta2[
            "Biopsy_ID"
        ]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )


else:

    roi_meta2[
        "Slide"
    ] = (
        roi_meta2.index
        .to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


slide_map = (
    roi_meta2[
        "Slide"
    ]
)


reproduced_module_scores[
    "Slide"
] = (
    reproduced_module_scores[
        "roi_id"
    ]
    .astype(str)
    .map(
        slide_map
    )
)


n_missing_slides = int(
    reproduced_module_scores[
        "Slide"
    ].isna().sum()
)


print(
    "\nMissing slide IDs in reproduced module scores:",
    n_missing_slides
)


if n_missing_slides > 0:

    raise ValueError(
        "Could not map all module-score ROIs to Xenium slides."
    )


# ============================================================
# 21. Save independently regenerated module scores
# ============================================================

reproduced_score_path = (
    REPRO_RESULTS /
    "cell4_reproduced_frozen_module_scores.csv"
)


reproduced_module_scores.to_csv(
    reproduced_score_path,
    index=False
)


# ============================================================
# 22. Compare scores against saved frozen scores
#
# Comparison is correlation-based because z-score ddof
# conventions can produce a trivial common scale factor.
# ============================================================

saved_scores = pd.read_csv(
    SAVED_SCORE_PATH
)


print(
    "\nSaved frozen-score columns:"
)


print(
    saved_scores.columns.tolist()
)


score_compare_available = (
    "module_label"
    in saved_scores.columns
    and
    "module_score"
    in saved_scores.columns
)


saved_roi_col = None


for candidate in [
    "roi_id",
    "ROI_ID",
    "roi"
]:

    if candidate in saved_scores.columns:

        saved_roi_col = candidate

        break


score_correlation_rows = []


if (
    score_compare_available
    and
    saved_roi_col is not None
):

    saved_score_small = (
        saved_scores[
            [
                saved_roi_col,
                "module_label",
                "module_score"
            ]
        ]
        .copy()
        .rename(
            columns={
                saved_roi_col:
                    "roi_id",

                "module_score":
                    "saved_module_score"
            }
        )
    )


    saved_score_small[
        "roi_id"
    ] = (
        saved_score_small[
            "roi_id"
        ].astype(str)
    )


    score_merge = (
        reproduced_module_scores[
            [
                "roi_id",
                "module_label",
                "module_score"
            ]
        ]
        .merge(
            saved_score_small,
            on=[
                "roi_id",
                "module_label"
            ],
            how="inner"
        )
    )


    for label in sorted(
        score_merge[
            "module_label"
        ].unique()
    ):

        sub = (
            score_merge[
                score_merge[
                    "module_label"
                ]
                ==
                label
            ]
        )


        r = float(
            np.corrcoef(
                sub[
                    "module_score"
                ],
                sub[
                    "saved_module_score"
                ]
            )[
                0,
                1
            ]
        )


        score_correlation_rows.append({

            "module_label":
                label,

            "n":
                len(
                    sub
                ),

            "score_correlation":
                r,

            "status":
                (
                    "PASS"
                    if r
                    >
                    0.999999
                    else
                    "CHECK"
                )
        })


    score_correlation_df = pd.DataFrame(
        score_correlation_rows
    )


    print(
        "\nModule-score correlation with saved scores:"
    )


    display(
        score_correlation_df
    )


else:

    score_correlation_df = pd.DataFrame()


    print(
        "\n⚠ Exact module-score comparison unavailable "
        "because the saved table lacks a merge key."
    )


# ============================================================
# 23. Module model helper
#
# Same formal specification as final Figure 3.
# ============================================================

def fit_module_model(
    data,
    add_slide=False
):

    required = [
        "module_score",
        "Disease",
        "Patient",
        "PC1",
        "Slide"
    ]


    d = (
        data
        .dropna(
            subset=required
        )
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    d = (
        d[
            d[
                "Disease"
            ].notna()
        ]
        .copy()
    )


    if add_slide:

        d[
            "Slide"
        ] = pd.Categorical(
            d[
                "Slide"
            ].astype(str)
        )


    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0
        /
        n_roi
    )


    if add_slide:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        )


    else:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    matrix_rank = int(
        np.linalg.matrix_rank(
            fit.model.exog
        )
    )


    n_columns = int(
        fit.model.exog.shape[
            1
        ]
    )


    interaction_terms = [
        term
        for term
        in fit.params.index
        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    pvalue = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return (
        d,
        fit,
        pvalue,
        matrix_rank,
        n_columns
    )


# ============================================================
# 24. Difference-curve helper
#
# anti-GBM minus LN
# ============================================================

def predict_module_difference(
    fit,
    data,
    grid,
    add_slide=False
):

    if not add_slide:

        predictions = {}


        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({

                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(
                            grid
                        ),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })


            predictions[
                disease
            ] = np.asarray(
                fit.predict(
                    newdata
                )
            )


        return (
            predictions[
                "GBM"
            ]
            -
            predictions[
                "SLE"
            ]
        )


    # --------------------------------------------------------
    # Slide-adjusted:
    # equal average over slide categories
    # --------------------------------------------------------

    slide_categories = (
        data[
            "Slide"
        ].cat.categories
    )


    predictions = {

        "SLE":
            [],

        "GBM":
            []
    }


    for slide in slide_categories:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({

                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(
                            grid
                        ),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide]
                        *
                        len(
                            grid
                        ),

                        categories=
                            slide_categories
                    )
            })


            predictions[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            predictions[
                "SLE"
            ]
        ),
        axis=0
    )


    mean_GBM = np.mean(
        np.vstack(
            predictions[
                "GBM"
            ]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 25. Selected frozen modules used in main Figure 3
# ============================================================

selected_modules = [

    "MAC M2",

    "FIB M1",

    "FIB M3"
]


module_sensitivity_rows = []


print(
    "\n" + "=" * 80
)

print(
    "REFITTING FIGURE 3 FROZEN MODULES"
)

print(
    "=" * 80
)


for module_label in selected_modules:

    d0 = (
        reproduced_module_scores[
            reproduced_module_scores[
                "module_label"
            ]
            ==
            module_label
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Determine slides containing both diseases
    # --------------------------------------------------------

    slide_disease = (
        d0.groupby(
            "Slide",
            observed=True
        )[
            "Disease"
        ]
        .agg(
            lambda x:
            set(
                x.astype(str)
            )
        )
    )


    overlap_slides = [

        slide

        for slide, diseases

        in slide_disease.items()

        if (
            "SLE"
            in diseases
            and
            "GBM"
            in diseases
        )
    ]


    d_overlap = (
        d0[
            d0[
                "Slide"
            ]
            .astype(str)
            .isin(
                [
                    str(
                        x
                    )
                    for x in overlap_slides
                ]
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Model 1: Unadjusted
    # --------------------------------------------------------

    (
        d_unadj,
        fit_unadj,
        p_unadj,
        rank_unadj,
        cols_unadj
    ) = fit_module_model(
        d0,
        add_slide=False
    )


    # --------------------------------------------------------
    # Model 2: All ROIs + Slide
    # --------------------------------------------------------

    (
        d_slide,
        fit_slide,
        p_slide,
        rank_slide,
        cols_slide
    ) = fit_module_model(
        d0,
        add_slide=True
    )


    # --------------------------------------------------------
    # Model 3: Overlap slides + Slide
    # --------------------------------------------------------

    (
        d_overlap2,
        fit_overlap,
        p_overlap,
        rank_overlap,
        cols_overlap
    ) = fit_module_model(
        d_overlap,
        add_slide=True
    )


    # --------------------------------------------------------
    # Common prediction support
    # --------------------------------------------------------

    low = max(

        d_unadj[
            "PC1"
        ].min(),

        d_slide[
            "PC1"
        ].min(),

        d_overlap2[
            "PC1"
        ].min()
    )


    high = min(

        d_unadj[
            "PC1"
        ].max(),

        d_slide[
            "PC1"
        ].max(),

        d_overlap2[
            "PC1"
        ].max()
    )


    grid = np.linspace(
        low,
        high,
        150
    )


    curve_unadj = (
        predict_module_difference(
            fit_unadj,
            d_unadj,
            grid,
            add_slide=False
        )
    )


    curve_slide = (
        predict_module_difference(
            fit_slide,
            d_slide,
            grid,
            add_slide=True
        )
    )


    curve_overlap = (
        predict_module_difference(
            fit_overlap,
            d_overlap2,
            grid,
            add_slide=True
        )
    )


    corr_slide = float(
        np.corrcoef(
            curve_unadj,
            curve_slide
        )[
            0,
            1
        ]
    )


    corr_overlap = float(
        np.corrcoef(
            curve_unadj,
            curve_overlap
        )[
            0,
            1
        ]
    )


    for model_name, pvalue, rank, cols, d_model, corr in [

        (
            "Unadjusted",
            p_unadj,
            rank_unadj,
            cols_unadj,
            d_unadj,
            1.0
        ),

        (
            "All + Slide",
            p_slide,
            rank_slide,
            cols_slide,
            d_slide,
            corr_slide
        ),

        (
            "Overlap slides + Slide",
            p_overlap,
            rank_overlap,
            cols_overlap,
            d_overlap2,
            corr_overlap
        )
    ]:

        module_sensitivity_rows.append({

            "module":
                module_label,

            "model":
                model_name,

            "interaction_p":
                float(
                    pvalue
                ),

            "curve_correlation_vs_unadjusted":
                float(
                    corr
                ),

            "matrix_rank":
                int(
                    rank
                ),

            "n_columns":
                int(
                    cols
                ),

            "n_ROI":
                len(
                    d_model
                ),

            "n_patients":
                d_model[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d_model.loc[
                    d_model[
                        "Disease"
                    ]
                    .astype(str)
                    ==
                    "GBM",
                    "Patient"
                ].nunique(),

            "n_slides":
                d_model[
                    "Slide"
                ].nunique()
        })


reproduced_module_sensitivity = pd.DataFrame(
    module_sensitivity_rows
)


print(
    "\nReproduced module sensitivity:"
)


display(
    reproduced_module_sensitivity
)


# ============================================================
# 26. Full-rank audit
# ============================================================

reproduced_module_sensitivity[
    "full_rank"
] = (
    reproduced_module_sensitivity[
        "matrix_rank"
    ]
    ==
    reproduced_module_sensitivity[
        "n_columns"
    ]
)


print(
    "\nModule model rank audit:"
)


display(
    reproduced_module_sensitivity[
        [
            "module",
            "model",
            "matrix_rank",
            "n_columns",
            "full_rank"
        ]
    ]
)


# ============================================================
# 27. Save reproduced module sensitivity
# ============================================================

module_sensitivity_path = (
    REPRO_RESULTS /
    "cell4_reproduced_slide_adjusted_module_sensitivity.csv"
)


reproduced_module_sensitivity.to_csv(
    module_sensitivity_path,
    index=False
)


# ============================================================
# 28. Compare module P values with saved final sensitivity
# ============================================================

saved_slide = pd.read_csv(
    SAVED_SLIDE_RESULT_PATH
)


print(
    "\nSaved slide-sensitivity columns:"
)


print(
    saved_slide.columns.tolist()
)


module_compare_rows = []


# ------------------------------------------------------------
# Flexible column detection
# ------------------------------------------------------------

saved_module_col = None


for c in [
    "module",
    "module_label"
]:

    if c in saved_slide.columns:

        saved_module_col = c

        break


saved_model_col = (
    "model"
    if "model" in saved_slide.columns
    else None
)


saved_p_col = None


for c in [
    "interaction_p",
    "pvalue",
    "p_value"
]:

    if c in saved_slide.columns:

        saved_p_col = c

        break


if (
    saved_module_col
    is not None
    and
    saved_model_col
    is not None
    and
    saved_p_col
    is not None
):

    saved_small = (
        saved_slide[
            [
                saved_module_col,
                saved_model_col,
                saved_p_col
            ]
        ]
        .copy()
        .rename(
            columns={
                saved_module_col:
                    "module",

                saved_model_col:
                    "model",

                saved_p_col:
                    "saved_interaction_p"
            }
        )
    )


    module_compare = (
        reproduced_module_sensitivity[
            [
                "module",
                "model",
                "interaction_p",
                "curve_correlation_vs_unadjusted"
            ]
        ]
        .merge(
            saved_small,
            on=[
                "module",
                "model"
            ],
            how="left"
        )
    )


    module_compare[
        "p_match"
    ] = [

        numeric_match(
            reproduced,
            expected
        )

        if np.isfinite(
            expected
        )
        else False

        for reproduced, expected
        in zip(
            module_compare[
                "interaction_p"
            ],

            module_compare[
                "saved_interaction_p"
            ]
        )
    ]


else:

    print(
        "\n⚠ Could not automatically parse "
        "saved slide-sensitivity table."
    )


    print(
        "Using expected final values for comparison."
    )


    expected_module_p = {

        (
            "MAC M2",
            "Unadjusted"
        ):
            7.462782e-05,

        (
            "MAC M2",
            "All + Slide"
        ):
            2.324871e-05,

        (
            "MAC M2",
            "Overlap slides + Slide"
        ):
            8.696723e-14,


        (
            "FIB M1",
            "Unadjusted"
        ):
            1.267494e-06,

        (
            "FIB M1",
            "All + Slide"
        ):
            3.138431e-08,

        (
            "FIB M1",
            "Overlap slides + Slide"
        ):
            5.961580e-11,


        (
            "FIB M3",
            "Unadjusted"
        ):
            3.961392e-08,

        (
            "FIB M3",
            "All + Slide"
        ):
            2.229473e-08,

        (
            "FIB M3",
            "Overlap slides + Slide"
        ):
            9.213984e-06
    }


    module_compare = (
        reproduced_module_sensitivity[
            [
                "module",
                "model",
                "interaction_p",
                "curve_correlation_vs_unadjusted"
            ]
        ]
        .copy()
    )


    module_compare[
        "saved_interaction_p"
    ] = [

        expected_module_p.get(
            (
                row.module,
                row.model
            ),
            np.nan
        )

        for row
        in module_compare.itertuples()
    ]


    module_compare[
        "p_match"
    ] = [

        numeric_match(
            reproduced,
            expected
        )

        if np.isfinite(
            expected
        )
        else False

        for reproduced, expected
        in zip(
            module_compare[
                "interaction_p"
            ],

            module_compare[
                "saved_interaction_p"
            ]
        )
    ]


print(
    "\n" + "=" * 80
)

print(
    "FIGURE 3 MODULE REPRODUCTION CHECK"
)

print(
    "=" * 80
)


display(
    module_compare
)


module_compare_path = (
    REPRO_RESULTS /
    "cell4_module_sensitivity_comparison.csv"
)


module_compare.to_csv(
    module_compare_path,
    index=False
)


# ============================================================
# 29. Frozen pathway output audit
#
# IMPORTANT:
# We do NOT re-query external pathway databases here.
#
# Reason:
# online Enrichr / GO / Reactome library versions can change.
#
# Exact manuscript reproduction therefore freezes the
# pathway-enrichment table generated during the original
# analysis and audits its contents.
# ============================================================

pathway_df = pd.read_csv(
    PATHWAY_PATH
)


print(
    "\n" + "=" * 80
)

print(
    "FROZEN MAIN-TEXT PATHWAY OUTPUT"
)

print(
    "=" * 80
)


display(
    pathway_df
)


pathway_audit_path = (
    REPRO_RESULTS /
    "cell4_frozen_maintext_pathway_audit.csv"
)


pathway_df.to_csv(
    pathway_audit_path,
    index=False
)


# ============================================================
# 30. Final Cell 4 PASS / CHECK logic
# ============================================================

sig_count_pass = bool(
    (
        sig_gene_count_check[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


gene_p_match_fraction = float(
    gene_compare[
        "pvalue_match"
    ].mean()
)


gene_fdr_match_fraction = float(
    gene_compare[
        "FDR_match"
    ].mean()
)


gene_level_pass = (
    gene_p_match_fraction
    >
    0.999
    and
    gene_fdr_match_fraction
    >
    0.999
)


module_size_pass = bool(
    (
        module_size_check[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


module_p_pass = bool(
    module_compare[
        "p_match"
    ].all()
)


module_rank_pass = bool(
    reproduced_module_sensitivity[
        "full_rank"
    ].all()
)


if len(
    score_correlation_df
) > 0:

    score_pass = bool(
        (
            score_correlation_df[
                "status"
            ]
            ==
            "PASS"
        ).all()
    )


else:

    score_pass = True


# ============================================================
# 31. Compact Cell 4 summary
# ============================================================

cell4_summary = pd.DataFrame({

    "check": [

        "MAC significant genes",

        "FIB significant genes",

        "Gene-level P-value reproduction",

        "Gene-level global FDR reproduction",

        "Frozen module sizes",

        "Recalculated module scores",

        "Selected module P values",

        "Selected module model rank"
    ],

    "result": [

        f"{int(significant_gene_counts['MAC'])} "
        "genes",

        f"{int(significant_gene_counts['FIB'])} "
        "genes",

        f"{gene_p_match_fraction:.4%} match",

        f"{gene_fdr_match_fraction:.4%} match",

        (
            "all matched"
            if module_size_pass
            else
            "check"
        ),

        (
            "matched"
            if score_pass
            else
            "check"
        ),

        (
            "9/9 matched"
            if module_p_pass
            else
            "check"
        ),

        (
            "all full rank"
            if module_rank_pass
            else
            "check"
        )
    ],

    "status": [

        (
            "PASS"
            if int(
                significant_gene_counts[
                    "MAC"
                ]
            )
            ==
            179
            else
            "CHECK"
        ),

        (
            "PASS"
            if int(
                significant_gene_counts[
                    "FIB"
                ]
            )
            ==
            164
            else
            "CHECK"
        ),

        (
            "PASS"
            if gene_p_match_fraction
            >
            0.999
            else
            "CHECK"
        ),

        (
            "PASS"
            if gene_fdr_match_fraction
            >
            0.999
            else
            "CHECK"
        ),

        (
            "PASS"
            if module_size_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if score_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if module_p_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if module_rank_pass
            else
            "CHECK"
        )
    ]
})


print(
    "\n" + "=" * 80
)

print(
    "CELL 4 FINAL SUMMARY"
)

print(
    "=" * 80
)


display(
    cell4_summary
)


cell4_summary_path = (
    REPRO_RESULTS /
    "cell4_figure3_reproduction_summary.csv"
)


cell4_summary.to_csv(
    cell4_summary_path,
    index=False
)


# ============================================================
# 32. Runtime
# ============================================================

runtime_seconds = (
    time.time()
    -
    start_time
)


print(
    "\nRuntime:"
)


print(
    f"{runtime_seconds / 60:.2f} minutes"
)


# ============================================================
# 33. Final decision
# ============================================================

hard_checks = [

    sig_count_pass,

    gene_level_pass,

    module_size_pass,

    score_pass,

    module_p_pass,

    module_rank_pass
]


print(
    "\n" + "=" * 80
)


if all(
    hard_checks
):

    print(
        "✅ CELL 4 PASSED"
    )


    print(
        "Figure 3 molecular results were independently reproduced."
    )


    print(
        "\nConfirmed:"
    )


    print(
        "  • MAC global-FDR genes = 179"
    )


    print(
        "  • FIB global-FDR genes = 164"
    )


    print(
        "  • frozen module membership is unchanged"
    )


    print(
        "  • module scores regenerate from expression"
    )


    print(
        "  • all 9 Figure 3 module sensitivity P values reproduce"
    )


    print(
        "  • slide-adjusted models remain full rank"
    )


    print(
        "\n可以进入 Cell 5："
    )


    print(
        "Figure 4 spatial-neighborhood reproduction."
    )


else:

    print(
        "⚠ CELL 4 COMPLETED WITH CHECK ITEMS"
    )


    print(
        "Do not proceed to Cell 5 until the issues above are resolved."
    )


    print(
        "Review the CELL 4 FINAL SUMMARY and "
        "FIGURE 3 MODULE REPRODUCTION CHECK outputs before continuing."
    )


print(
    "=" * 80
)


# ============================================================
# 34. Saved outputs
# ============================================================

print(
    "\nSaved outputs:"
)


for p in [

    reproduced_gene_path,

    gene_compare_path,

    reproduced_score_path,

    module_sensitivity_path,

    module_compare_path,

    pathway_audit_path,

    cell4_summary_path
]:

    print(
        p
    )

In [ ]:
# ============================================================
# REPRODUCTION — CELL 4B
#
# Diagnose Figure 3 global-FDR family mismatch
#
# We already know:
#   - gene-level P values reproduce 100%
#   - module results reproduce exactly
#
# Question:
# Why are global BH-FDR values different?
#
# This cell checks whether the reason is that the original
# formal testing family contained 477 MAC + 478 FIB genes
# rather than all 480 genes per cell type.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_RESULTS = (
    BASE /
    "reproduction" /
    "results"
)


REPRO_GENE_PATH = (
    REPRO_RESULTS /
    "cell4_reproduced_MAC_FIB_gene_trajectory_results.csv"
)


SAVED_GENE_PATH = (
    BASE /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)


EXPR_PATH = (
    BASE /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)


for path in [
    REPRO_GENE_PATH,
    SAVED_GENE_PATH,
    EXPR_PATH
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )


# ============================================================
# 2. Load
# ============================================================

repro = pd.read_csv(
    REPRO_GENE_PATH
)


saved = pd.read_csv(
    SAVED_GENE_PATH
)


expr = pd.read_csv(
    EXPR_PATH
)


print(
    "=" * 78
)

print(
    "CELL 4B — GLOBAL FDR FAMILY DIAGNOSTIC"
)

print(
    "=" * 78
)


print(
    "\nReproduced gene-result rows:",
    len(repro)
)


print(
    "Saved formal gene-result rows:",
    len(saved)
)


# ============================================================
# 3. Standardize saved columns
# ============================================================

if "celltype" not in saved.columns:

    if "celltype_l1" in saved.columns:

        saved = saved.rename(
            columns={
                "celltype_l1":
                    "celltype"
            }
        )

    else:

        raise ValueError(
            "Cannot identify saved celltype column."
        )


if "gene" not in saved.columns:

    raise ValueError(
        "Saved table lacks gene column."
    )


if "pvalue" not in saved.columns:

    raise ValueError(
        "Saved table lacks pvalue column."
    )


saved_fdr_col = None


for candidate in [
    "FDR_global",
    "global_FDR",
    "FDR"
]:

    if candidate in saved.columns:

        saved_fdr_col = candidate

        break


if saved_fdr_col is None:

    raise ValueError(
        "Cannot identify saved global FDR column."
    )


print(
    "\nSaved global FDR column:",
    saved_fdr_col
)


# ============================================================
# 4. Membership counts
# ============================================================

repro_membership = (
    repro[
        repro[
            "pvalue"
        ].notna()
    ]
    .groupby(
        "celltype",
        observed=True
    )
    .size()
)


saved_membership = (
    saved
    .groupby(
        "celltype",
        observed=True
    )
    .size()
)


membership = pd.DataFrame({

    "reproduced_valid_p":
        repro_membership,

    "saved_formal_tests":
        saved_membership
})


membership[
    "difference"
] = (
    membership[
        "reproduced_valid_p"
    ]
    -
    membership[
        "saved_formal_tests"
    ]
)


print(
    "\n" + "=" * 78
)

print(
    "TESTING-FAMILY SIZE"
)

print(
    "=" * 78
)


display(
    membership
)


# ============================================================
# 5. Identify genes present in reproduction but absent
#    from the saved formal testing family
# ============================================================

repro_keys = (
    repro[
        [
            "celltype",
            "gene"
        ]
    ]
    .drop_duplicates()
)


saved_keys = (
    saved[
        [
            "celltype",
            "gene"
        ]
    ]
    .drop_duplicates()
)


membership_merge = (
    repro_keys
    .merge(
        saved_keys.assign(
            in_saved=True
        ),
        on=[
            "celltype",
            "gene"
        ],
        how="left"
    )
)


extra_genes = (
    membership_merge[
        membership_merge[
            "in_saved"
        ].isna()
    ]
    [
        [
            "celltype",
            "gene"
        ]
    ]
    .copy()
)


print(
    "\n" + "=" * 78
)

print(
    "GENES FIT IN REPRODUCTION BUT NOT IN ORIGINAL FORMAL FAMILY"
)

print(
    "=" * 78
)


display(
    extra_genes
)


print(
    "\nCounts:"
)


display(
    extra_genes[
        "celltype"
    ]
    .value_counts()
    .rename(
        "n_excluded"
    )
    .to_frame()
)


# ============================================================
# 6. Check reverse direction
#
# There should ideally be no genes in saved that are absent
# from the independently reproduced table.
# ============================================================

reverse_merge = (
    saved_keys
    .merge(
        repro_keys.assign(
            in_repro=True
        ),
        on=[
            "celltype",
            "gene"
        ],
        how="left"
    )
)


missing_from_repro = (
    reverse_merge[
        reverse_merge[
            "in_repro"
        ].isna()
    ]
    [
        [
            "celltype",
            "gene"
        ]
    ]
)


print(
    "\nGenes in saved formal results "
    "but missing from reproduction:"
)


display(
    missing_from_repro
)


# ============================================================
# 7. Reconstruct raw-expression diagnostics for excluded genes
# ============================================================

metadata_candidates = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1",
    "Slide"
]


metadata_cols = [
    c
    for c in metadata_candidates
    if c in expr.columns
]


gene_cols = [
    c
    for c in expr.columns
    if c not in metadata_cols
]


diagnostic_rows = []


for row in extra_genes.itertuples(
    index=False
):

    celltype = str(
        row.celltype
    )


    gene = str(
        row.gene
    )


    if gene not in expr.columns:

        continue


    sub = (
        expr[
            expr[
                "celltype_l1"
            ]
            .astype(str)
            ==
            celltype
        ]
        .copy()
    )


    raw = (
        pd.to_numeric(
            sub[
                gene
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    sle_mask = (
        sub[
            "Disease"
        ]
        .astype(str)
        .to_numpy()
        ==
        "SLE"
    )


    gbm_mask = (
        sub[
            "Disease"
        ]
        .astype(str)
        .to_numpy()
        ==
        "GBM"
    )


    detected_all = int(
        np.sum(
            raw
            >
            0
        )
    )


    detected_sle = int(
        np.sum(
            raw[
                sle_mask
            ]
            >
            0
        )
    )


    detected_gbm = int(
        np.sum(
            raw[
                gbm_mask
            ]
            >
            0
        )
    )


    diagnostic_rows.append({

        "celltype":
            celltype,

        "gene":
            gene,

        "n_ROI":
            len(
                raw
            ),

        "detected_all_ROI":
            detected_all,

        "detection_fraction":
            (
                detected_all
                /
                len(
                    raw
                )
            ),

        "detected_LN_ROI":
            detected_sle,

        "detected_GBM_ROI":
            detected_gbm,

        "raw_mean":
            float(
                np.nanmean(
                    raw
                )
            ),

        "raw_std":
            float(
                np.nanstd(
                    raw,
                    ddof=0
                )
            ),

        "raw_min":
            float(
                np.nanmin(
                    raw
                )
            ),

        "raw_max":
            float(
                np.nanmax(
                    raw
                )
            )
    })


excluded_diagnostics = pd.DataFrame(
    diagnostic_rows
)


print(
    "\n" + "=" * 78
)

print(
    "EXCLUDED-GENE EXPRESSION DIAGNOSTICS"
)

print(
    "=" * 78
)


display(
    excluded_diagnostics
)


# ============================================================
# 8. Attach reproduced P values to excluded genes
# ============================================================

excluded_with_p = (
    extra_genes
    .merge(
        repro[
            [
                "celltype",
                "gene",
                "pvalue",
                "detection_fraction",
                "matrix_rank",
                "n_columns",
                "fit_status"
            ]
        ],
        on=[
            "celltype",
            "gene"
        ],
        how="left"
    )
)


print(
    "\nExcluded genes with reproduced model information:"
)


display(
    excluded_with_p
)


# ============================================================
# 9. KEY TEST:
#
# Recalculate BH-FDR using ONLY the same genes contained
# in the original formal testing table.
#
# If this reproduces saved FDR exactly, then:
#
#   - statistical models are correct
#   - P values are correct
#   - the only discrepancy was testing-family membership
# ============================================================

formal_family = (
    repro
    .merge(
        saved_keys.assign(
            formal_test=True
        ),
        on=[
            "celltype",
            "gene"
        ],
        how="inner"
    )
    .copy()
)


formal_family = (
    formal_family[
        formal_family[
            "pvalue"
        ].notna()
    ]
    .copy()
)


print(
    "\nFormal-family reproduced test count:",
    len(
        formal_family
    )
)


print(
    "\nBy cell type:"
)


display(
    formal_family[
        "celltype"
    ]
    .value_counts()
    .rename(
        "n_tests"
    )
    .to_frame()
)


# ============================================================
# 10. Recalculate GLOBAL BH
# ============================================================

formal_family[
    "FDR_global_recalculated"
] = multipletests(
    formal_family[
        "pvalue"
    ],
    method="fdr_bh"
)[1]


# ============================================================
# 11. Compare against saved FDR
# ============================================================

saved_for_compare = (
    saved[
        [
            "celltype",
            "gene",
            "pvalue",
            saved_fdr_col
        ]
    ]
    .copy()
    .rename(
        columns={
            "pvalue":
                "saved_pvalue",

            saved_fdr_col:
                "saved_FDR_global"
        }
    )
)


fdr_compare = (
    formal_family[
        [
            "celltype",
            "gene",
            "pvalue",
            "FDR_global_recalculated"
        ]
    ]
    .merge(
        saved_for_compare,
        on=[
            "celltype",
            "gene"
        ],
        how="inner"
    )
)


fdr_compare[
    "p_abs_diff"
] = (
    fdr_compare[
        "pvalue"
    ]
    -
    fdr_compare[
        "saved_pvalue"
    ]
).abs()


fdr_compare[
    "FDR_abs_diff"
] = (
    fdr_compare[
        "FDR_global_recalculated"
    ]
    -
    fdr_compare[
        "saved_FDR_global"
    ]
).abs()


# ============================================================
# 12. Robust match rule
# ============================================================

def values_match(
    x,
    y
):

    x = float(
        x
    )

    y = float(
        y
    )


    if (
        x
        ==
        y
    ):

        return True


    if (
        x
        >
        0
        and
        y
        >
        0
        and
        (
            x
            <
            1e-6
            or
            y
            <
            1e-6
        )
    ):

        return (
            abs(
                np.log10(
                    x
                )
                -
                np.log10(
                    y
                )
            )
            <
            1e-5
        )


    return (
        abs(
            x
            -
            y
        )
        <
        1e-8
    )


fdr_compare[
    "p_match"
] = [

    values_match(
        x,
        y
    )

    for x, y in zip(
        fdr_compare[
            "pvalue"
        ],
        fdr_compare[
            "saved_pvalue"
        ]
    )
]


fdr_compare[
    "FDR_match"
] = [

    values_match(
        x,
        y
    )

    for x, y in zip(
        fdr_compare[
            "FDR_global_recalculated"
        ],
        fdr_compare[
            "saved_FDR_global"
        ]
    )
]


p_match_fraction = float(
    fdr_compare[
        "p_match"
    ].mean()
)


fdr_match_fraction = float(
    fdr_compare[
        "FDR_match"
    ].mean()
)


print(
    "\n" + "=" * 78
)

print(
    "FORMAL FAMILY REPRODUCTION"
)

print(
    "=" * 78
)


print(
    "P-value match fraction:"
)


print(
    f"{p_match_fraction:.6%}"
)


print(
    "\nGlobal FDR match fraction:"
)


print(
    f"{fdr_match_fraction:.6%}"
)


print(
    "\nMaximum absolute FDR difference:"
)


print(
    fdr_compare[
        "FDR_abs_diff"
    ].max()
)


# ============================================================
# 13. Recalculate significant-gene counts
# ============================================================

formal_sig_counts = (
    formal_family[
        formal_family[
            "FDR_global_recalculated"
        ]
        <
        0.05
    ]
    .groupby(
        "celltype",
        observed=True
    )
    .size()
    .reindex(
        [
            "MAC",
            "FIB"
        ]
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


print(
    "\n" + "=" * 78
)

print(
    "FORMAL-FAMILY GLOBAL FDR < 0.05"
)

print(
    "=" * 78
)


display(
    formal_sig_counts
    .rename(
        "n_significant_genes"
    )
    .to_frame()
)


# ============================================================
# 14. Final diagnostic summary
# ============================================================

expected_family = {

    "MAC":
        477,

    "FIB":
        478
}


expected_sig = {

    "MAC":
        179,

    "FIB":
        164
}


summary_rows = []


for celltype in [
    "MAC",
    "FIB"
]:

    n_formal = int(
        (
            formal_family[
                "celltype"
            ]
            ==
            celltype
        ).sum()
    )


    n_sig = int(
        formal_sig_counts[
            celltype
        ]
    )


    summary_rows.append({

        "celltype":
            celltype,

        "expected_formal_tests":
            expected_family[
                celltype
            ],

        "reproduced_formal_tests":
            n_formal,

        "formal_test_status":
            (
                "PASS"
                if n_formal
                ==
                expected_family[
                    celltype
                ]
                else
                "CHECK"
            ),

        "expected_FDR_lt_005":
            expected_sig[
                celltype
            ],

        "reproduced_FDR_lt_005":
            n_sig,

        "significant_count_status":
            (
                "PASS"
                if n_sig
                ==
                expected_sig[
                    celltype
                ]
                else
                "CHECK"
            )
    })


family_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n" + "=" * 78
)

print(
    "CELL 4B FINAL SUMMARY"
)

print(
    "=" * 78
)


display(
    family_summary
)


# ============================================================
# 15. Save diagnostics
# ============================================================

extra_path = (
    REPRO_RESULTS /
    "cell4B_genes_excluded_from_formal_FDR_family.csv"
)


diag_path = (
    REPRO_RESULTS /
    "cell4B_excluded_gene_expression_diagnostics.csv"
)


compare_path = (
    REPRO_RESULTS /
    "cell4B_formal_family_FDR_comparison.csv"
)


summary_path = (
    REPRO_RESULTS /
    "cell4B_formal_FDR_family_summary.csv"
)


extra_genes.to_csv(
    extra_path,
    index=False
)


excluded_diagnostics.to_csv(
    diag_path,
    index=False
)


fdr_compare.to_csv(
    compare_path,
    index=False
)


family_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 16. Final interpretation
# ============================================================

all_family_sizes_pass = bool(
    (
        family_summary[
            "formal_test_status"
        ]
        ==
        "PASS"
    ).all()
)


all_sig_counts_pass = bool(
    (
        family_summary[
            "significant_count_status"
        ]
        ==
        "PASS"
    ).all()
)


print(
    "\n" + "=" * 78
)


if (
    all_family_sizes_pass
    and
    all_sig_counts_pass
    and
    p_match_fraction
    >
    0.99999
    and
    fdr_match_fraction
    >
    0.99999
):

    print(
        "✅ CELL 4B PASSED"
    )


    print(
        "\nThe Figure 3 discrepancy is fully explained:"
    )


    print(
        "the original global BH-FDR family contained "
        "477 MAC + 478 FIB genes."
    )


    print(
        "\nGene-level model calculations were already correct."
    )


    print(
        "After restoring the original formal testing family:"
    )


    print(
        "  • global FDR values reproduce"
    )


    print(
        "  • MAC significant genes = 179"
    )


    print(
        "  • FIB significant genes = 164"
    )


    print(
        "\nFigure 3 can therefore be considered "
        "computationally reproduced."
    )


else:

    print(
        "⚠ CELL 4B NEEDS REVIEW"
    )


    print(
        "\nDo not continue to Cell 5 yet."
    )


    print(
        "Send me:"
    )


    print(
        "  1. GENES FIT IN REPRODUCTION BUT NOT..."
    )


    print(
        "  2. EXCLUDED-GENE EXPRESSION DIAGNOSTICS"
    )


    print(
        "  3. CELL 4B FINAL SUMMARY"
    )


print(
    "=" * 78
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 5
#
# FIGURE 4 SPATIAL-NEIGHBORHOOD REPRODUCTION
#
# Independently recalculated from saved ROI-level
# spatial-neighborhood source table:
#
#   1. Four prespecified spatial relationships
#   2. BH-FDR across the four relationships
#   3. Primary MAC–FIB trajectory
#   4. Leave-one-anti-GBM-patient-out
#   5. Model-specification sensitivity
#   6. Xenium-slide adjustment
#   7. Overlap-slide restriction
#   8. Difference-curve correlations
#
# k sensitivity:
#   - k=6 is independently cross-checked against primary fit
#   - k=4/8/10 are audited from the frozen sensitivity output
#
# IMPORTANT:
# no manuscript statistics are used to fit the models.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_RESULTS = (
    BASE /
    "reproduction" /
    "results"
)


REPRO_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


ROI_PATH = (
    BASE /
    "roi_782_PC1_primary.h5ad"
)


SPATIAL_PATH = (
    BASE /
    "figure4_driver_neighbor_data_smoothed.csv"
)


SAVED_PRIMARY_PATH = (
    BASE /
    "figure4_driver_results_smoothed.csv"
)


SAVED_K_PATH = (
    BASE /
    "figure4_smoothed_k_sensitivity.csv"
)


SAVED_LOO_PATH = (
    BASE /
    "figure4_smoothed_GBM_LOO.csv"
)


SAVED_SPEC_PATH = (
    BASE /
    "figure4_final_model_specification_robustness.csv"
)


required_paths = [
    ROI_PATH,
    SPATIAL_PATH,
    SAVED_PRIMARY_PATH,
    SAVED_K_PATH,
    SAVED_LOO_PATH,
    SAVED_SPEC_PATH
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )


# ============================================================
# 2. Load source spatial table
# ============================================================

print(
    "=" * 82
)

print(
    "CELL 5 — FIGURE 4 SPATIAL REPRODUCTION"
)

print(
    "=" * 82
)


spatial = pd.read_csv(
    SPATIAL_PATH,
    index_col=0
)


spatial.index = (
    spatial.index.astype(str)
)


print(
    "\nSpatial source table:",
    spatial.shape
)


print(
    "\nColumns:"
)


print(
    spatial.columns.tolist()
)


# ============================================================
# 3. Required metadata audit
# ============================================================

required_metadata = [
    "Disease",
    "Patient",
    "PC1"
]


missing_metadata = [
    c
    for c in required_metadata
    if c not in spatial.columns
]


if len(
    missing_metadata
) > 0:

    raise ValueError(
        "Spatial table missing metadata columns:\n"
        +
        "\n".join(
            missing_metadata
        )
    )


# ============================================================
# 4. Detect four formal relationship columns
# ============================================================

expected_relationships = [

    "MAC_to_FIB",

    "MAC_to_FibroticMC",

    "Mono_to_FIB",

    "Mono_to_FibroticMC"
]


relationship_cols = [
    c
    for c in expected_relationships
    if c in spatial.columns
]


print(
    "\nFormal spatial relationships detected:"
)


print(
    relationship_cols
)


if len(
    relationship_cols
) != 4:

    raise ValueError(
        f"""
Expected four spatial relationships but detected:

{relationship_cols}

Do not manually rename anything.
Send the printed spatial columns to me.
"""
    )


# ============================================================
# 5. Restrict to LN vs anti-GBM
# ============================================================

d_spatial = (
    spatial[
        spatial[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ]
    .copy()
)


print(
    "\nLN / anti-GBM source ROIs:",
    len(
        d_spatial
    )
)


print(
    "\nROIs by disease:"
)


display(
    d_spatial[
        "Disease"
    ]
    .value_counts()
    .rename(
        "n_ROI"
    )
    .to_frame()
)


print(
    "\nPatients by disease:"
)


display(
    d_spatial
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient"
    ]
    .nunique()
    .rename(
        "n_patients"
    )
    .to_frame()
)


# ============================================================
# 6. Generic spatial trajectory model
#
# outcome ~ spline(PC1, df) * Disease
#
# Patient-balanced WLS
# Patient-clustered covariance
#
# Optional:
#   + C(Slide)
# ============================================================

def fit_spatial_model(
    data,
    outcome,
    spline_df=3,
    add_slide=False
):

    required = [
        outcome,
        "Disease",
        "Patient",
        "PC1"
    ]


    if add_slide:

        required.append(
            "Slide"
        )


    d = (
        data
        .dropna(
            subset=required
        )
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),

        categories=[
            "SLE",
            "GBM"
        ]
    )


    d = (
        d[
            d[
                "Disease"
            ].notna()
        ]
        .copy()
    )


    if add_slide:

        d[
            "Slide"
        ] = pd.Categorical(
            d[
                "Slide"
            ].astype(str)
        )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------------------------------------
    # Formula
    # --------------------------------------------------------

    spline_text = (
        f"bs(PC1, df={int(spline_df)}, "
        "degree=3, include_intercept=False)"
    )


    if add_slide:

        formula = (
            f"{outcome} ~ "
            f"{spline_text} "
            "* C(Disease) "
            "+ C(Slide)"
        )

    else:

        formula = (
            f"{outcome} ~ "
            f"{spline_text} "
            "* C(Disease)"
        )


    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    matrix_rank = int(
        np.linalg.matrix_rank(
            fit.model.exog
        )
    )


    n_columns = int(
        fit.model.exog.shape[
            1
        ]
    )


    interaction_terms = [
        term
        for term
        in fit.params.index
        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    if len(
        interaction_terms
    ) == 0:

        raise ValueError(
            f"No Disease × PC1 terms found for {outcome}."
        )


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    pvalue = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return {

        "data":
            d,

        "fit":
            fit,

        "pvalue":
            pvalue,

        "matrix_rank":
            matrix_rank,

        "n_columns":
            n_columns,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ]
                .astype(str)
                ==
                "GBM",
                "Patient"
            ].nunique()
    }


# ============================================================
# 7. Refit all four formal spatial relationships
# ============================================================

formal_rows = []


print(
    "\n" + "=" * 82
)

print(
    "REFITTING FOUR FORMAL SPATIAL RELATIONSHIPS"
)

print(
    "=" * 82
)


for relationship in relationship_cols:

    result = fit_spatial_model(
        d_spatial,
        relationship,
        spline_df=3,
        add_slide=False
    )


    formal_rows.append({

        "relationship":
            relationship,

        "pvalue":
            result[
                "pvalue"
            ],

        "matrix_rank":
            result[
                "matrix_rank"
            ],

        "n_columns":
            result[
                "n_columns"
            ],

        "n_ROI":
            result[
                "n_ROI"
            ],

        "n_patients":
            result[
                "n_patients"
            ],

        "n_GBM_patients":
            result[
                "n_GBM_patients"
            ]
    })


formal_results = pd.DataFrame(
    formal_rows
)


# ============================================================
# 8. BH-FDR across four relationships
# ============================================================

formal_results[
    "FDR"
] = multipletests(
    formal_results[
        "pvalue"
    ],
    method="fdr_bh"
)[1]


formal_results = (
    formal_results
    .sort_values(
        "pvalue"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nReproduced formal spatial results:"
)


display(
    formal_results
)


# ============================================================
# 9. Compare to saved primary results
# ============================================================

saved_primary = pd.read_csv(
    SAVED_PRIMARY_PATH
)


print(
    "\nSaved formal results:"
)


display(
    saved_primary
)


formal_compare = (
    formal_results[
        [
            "relationship",
            "pvalue",
            "FDR",
            "matrix_rank",
            "n_columns"
        ]
    ]
    .merge(
        saved_primary[
            [
                c
                for c in [
                    "relationship",
                    "pvalue",
                    "FDR",
                    "matrix_rank",
                    "n_columns"
                ]
                if c in saved_primary.columns
            ]
        ],
        on="relationship",
        how="inner",
        suffixes=(
            "_reproduced",
            "_saved"
        )
    )
)


def near_equal(
    x,
    y,
    log_tol=1e-5,
    abs_tol=1e-8
):

    x = float(
        x
    )

    y = float(
        y
    )


    if x == y:

        return True


    if (
        x > 0
        and
        y > 0
        and
        (
            x < 1e-6
            or
            y < 1e-6
        )
    ):

        return (
            abs(
                np.log10(
                    x
                )
                -
                np.log10(
                    y
                )
            )
            <
            log_tol
        )


    return (
        abs(
            x
            -
            y
        )
        <
        abs_tol
    )


formal_compare[
    "p_match"
] = [

    near_equal(
        x,
        y
    )

    for x, y in zip(
        formal_compare[
            "pvalue_reproduced"
        ],
        formal_compare[
            "pvalue_saved"
        ]
    )
]


formal_compare[
    "FDR_match"
] = [

    near_equal(
        x,
        y
    )

    for x, y in zip(
        formal_compare[
            "FDR_reproduced"
        ],
        formal_compare[
            "FDR_saved"
        ]
    )
]


print(
    "\n" + "=" * 82
)

print(
    "FORMAL SPATIAL RESULT COMPARISON"
)

print(
    "=" * 82
)


display(
    formal_compare
)


# ============================================================
# 10. Primary MAC–FIB result
# ============================================================

primary_row = (
    formal_results[
        formal_results[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .iloc[
        0
    ]
)


primary_p = float(
    primary_row[
        "pvalue"
    ]
)


primary_fdr = float(
    primary_row[
        "FDR"
    ]
)


print(
    "\n" + "=" * 82
)

print(
    "PRIMARY MAC–FIB RESULT"
)

print(
    "=" * 82
)


print(
    "P =",
    primary_p
)


print(
    "FDR =",
    primary_fdr
)


# ============================================================
# 11. Independently reproduce leave-one-anti-GBM-out
# ============================================================

primary_data = (
    d_spatial
    .dropna(
        subset=[
            "MAC_to_FIB",
            "Disease",
            "Patient",
            "PC1"
        ]
    )
    .copy()
)


gbm_patients = sorted(
    primary_data.loc[
        primary_data[
            "Disease"
        ]
        .astype(str)
        ==
        "GBM",
        "Patient"
    ]
    .astype(str)
    .unique()
)


print(
    "\nanti-GBM patients available:"
)


print(
    gbm_patients
)


loo_rows = []


for patient in gbm_patients:

    d_loo = (
        primary_data[
            ~(
                (
                    primary_data[
                        "Disease"
                    ]
                    .astype(str)
                    ==
                    "GBM"
                )
                &
                (
                    primary_data[
                        "Patient"
                    ]
                    .astype(str)
                    ==
                    patient
                )
            )
        ]
        .copy()
    )


    result = fit_spatial_model(
        d_loo,
        "MAC_to_FIB",
        spline_df=3,
        add_slide=False
    )


    loo_rows.append({

        "dropped_GBM_patient":
            patient,

        "relationship":
            "MAC_to_FIB",

        "pvalue":
            result[
                "pvalue"
            ],

        "matrix_rank":
            result[
                "matrix_rank"
            ],

        "n_columns":
            result[
                "n_columns"
            ],

        "n_ROI":
            result[
                "n_ROI"
            ],

        "n_patients":
            result[
                "n_patients"
            ],

        "n_GBM_patients":
            result[
                "n_GBM_patients"
            ]
    })


loo_reproduced = pd.DataFrame(
    loo_rows
)


print(
    "\n" + "=" * 82
)

print(
    "REPRODUCED LEAVE-ONE-ANTI-GBM-OUT"
)

print(
    "=" * 82
)


display(
    loo_reproduced
)


# ============================================================
# 12. Compare LOO with frozen final output
# ============================================================

saved_loo = pd.read_csv(
    SAVED_LOO_PATH
)


saved_loo_macfib = (
    saved_loo[
        saved_loo[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


loo_compare = (
    loo_reproduced
    .merge(
        saved_loo_macfib[
            [
                "dropped_GBM_patient",
                "pvalue"
            ]
        ],
        on="dropped_GBM_patient",
        how="left",
        suffixes=(
            "_reproduced",
            "_saved"
        )
    )
)


loo_compare[
    "p_match"
] = [

    near_equal(
        x,
        y
    )

    for x, y in zip(
        loo_compare[
            "pvalue_reproduced"
        ],
        loo_compare[
            "pvalue_saved"
        ]
    )
]


print(
    "\nLOO comparison:"
)


display(
    loo_compare
)


loo_all_nominal = bool(
    (
        loo_reproduced[
            "pvalue"
        ]
        <
        0.05
    ).all()
)


print(
    "\nLOO nominal P < 0.05:",
    f"{int((loo_reproduced['pvalue'] < 0.05).sum())}"
    "/"
    f"{len(loo_reproduced)}"
)


# ============================================================
# 13. Model-specification sensitivity
#
# PC1 trimming:
#   0
#   0.025
#   0.05
#
# spline df:
#   3
#   4
#   5
# ============================================================

spec_rows = []


for trim in [
    0.000,
    0.025,
    0.050
]:

    if trim == 0:

        low = float(
            primary_data[
                "PC1"
            ].min()
        )


        high = float(
            primary_data[
                "PC1"
            ].max()
        )


    else:

        low = float(
            primary_data[
                "PC1"
            ].quantile(
                trim
            )
        )


        high = float(
            primary_data[
                "PC1"
            ].quantile(
                1.0
                -
                trim
            )
        )


    d_trim = (
        primary_data[
            primary_data[
                "PC1"
            ].between(
                low,
                high
            )
        ]
        .copy()
    )


    for spline_df in [
        3,
        4,
        5
    ]:

        result = fit_spatial_model(
            d_trim,
            "MAC_to_FIB",
            spline_df=spline_df,
            add_slide=False
        )


        spec_rows.append({

            "relationship":
                "MAC_to_FIB",

            "PC1_trim":
                trim,

            "spline_df":
                spline_df,

            "PC1_low":
                low,

            "PC1_high":
                high,

            "pvalue":
                result[
                    "pvalue"
                ],

            "matrix_rank":
                result[
                    "matrix_rank"
                ],

            "n_columns":
                result[
                    "n_columns"
                ],

            "n_ROI":
                result[
                    "n_ROI"
                ],

            "n_patients":
                result[
                    "n_patients"
                ],

            "n_GBM_patients":
                result[
                    "n_GBM_patients"
                ]
        })


spec_reproduced = pd.DataFrame(
    spec_rows
)


print(
    "\n" + "=" * 82
)

print(
    "REPRODUCED MODEL-SPECIFICATION SENSITIVITY"
)

print(
    "=" * 82
)


display(
    spec_reproduced
)


# ============================================================
# 14. Compare model-specification results
# ============================================================

saved_spec = pd.read_csv(
    SAVED_SPEC_PATH
)


saved_spec_macfib = (
    saved_spec[
        saved_spec[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


spec_compare = (
    spec_reproduced
    .merge(
        saved_spec_macfib[
            [
                c
                for c in [
                    "relationship",
                    "PC1_trim",
                    "spline_df",
                    "pvalue",
                    "matrix_rank",
                    "n_columns",
                    "n_ROI"
                ]
                if c in saved_spec_macfib.columns
            ]
        ],
        on=[
            "relationship",
            "PC1_trim",
            "spline_df"
        ],
        how="left",
        suffixes=(
            "_reproduced",
            "_saved"
        )
    )
)


spec_compare[
    "p_match"
] = [

    near_equal(
        x,
        y
    )

    if np.isfinite(
        y
    )
    else False

    for x, y in zip(
        spec_compare[
            "pvalue_reproduced"
        ],
        spec_compare[
            "pvalue_saved"
        ]
    )
]


print(
    "\nModel-specification comparison:"
)


display(
    spec_compare
)


spec_all_nominal = bool(
    (
        spec_reproduced[
            "pvalue"
        ]
        <
        0.05
    ).all()
)


spec_all_full_rank = bool(
    (
        spec_reproduced[
            "matrix_rank"
        ]
        ==
        spec_reproduced[
            "n_columns"
        ]
    ).all()
)


print(
    "\nSpecifications nominal P < 0.05:",
    f"{int((spec_reproduced['pvalue'] < 0.05).sum())}/9"
)


# ============================================================
# 15. Reconstruct Xenium slide ID
# ============================================================

roi_ad = ad.read_h5ad(
    ROI_PATH
)


roi_meta = (
    roi_ad.obs.copy()
)


roi_meta.index = (
    roi_meta.index.astype(str)
)


if "Biopsy_ID" in roi_meta.columns:

    roi_meta[
        "Slide"
    ] = (
        roi_meta[
            "Biopsy_ID"
        ]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )


else:

    roi_meta[
        "Slide"
    ] = (
        roi_meta.index
        .to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


primary_data[
    "Slide"
] = (
    primary_data.index
    .to_series()
    .map(
        roi_meta[
            "Slide"
        ]
    )
    .values
)


missing_slides = int(
    primary_data[
        "Slide"
    ].isna().sum()
)


print(
    "\nMissing slide IDs:",
    missing_slides
)


if missing_slides > 0:

    raise ValueError(
        "Could not map all spatial ROIs to Xenium slides."
    )


# ============================================================
# 16. Identify disease-overlapping slides
# ============================================================

slide_disease_sets = (
    primary_data
    .groupby(
        "Slide",
        observed=True
    )[
        "Disease"
    ]
    .agg(
        lambda x:
        set(
            x.astype(str)
        )
    )
)


overlap_slides = [

    slide

    for slide, diseases
    in slide_disease_sets.items()

    if (
        "SLE"
        in diseases
        and
        "GBM"
        in diseases
    )
]


print(
    "\nDisease-overlapping slides:"
)


print(
    overlap_slides
)


print(
    "Number of overlapping slides:",
    len(
        overlap_slides
    )
)


d_overlap = (
    primary_data[
        primary_data[
            "Slide"
        ]
        .astype(str)
        .isin(
            [
                str(
                    x
                )
                for x in overlap_slides
            ]
        )
    ]
    .copy()
)


# ============================================================
# 17. Refit three slide models
# ============================================================

unadjusted_result = (
    fit_spatial_model(
        primary_data,
        "MAC_to_FIB",
        spline_df=3,
        add_slide=False
    )
)


slide_result = (
    fit_spatial_model(
        primary_data,
        "MAC_to_FIB",
        spline_df=3,
        add_slide=True
    )
)


overlap_result = (
    fit_spatial_model(
        d_overlap,
        "MAC_to_FIB",
        spline_df=3,
        add_slide=True
    )
)


print(
    "\n" + "=" * 82
)

print(
    "SLIDE SENSITIVITY"
)

print(
    "=" * 82
)


print(
    "Unadjusted P =",
    unadjusted_result[
        "pvalue"
    ]
)


print(
    "Slide-adjusted P =",
    slide_result[
        "pvalue"
    ]
)


print(
    "Overlap-slide restricted P =",
    overlap_result[
        "pvalue"
    ]
)


# ============================================================
# 18. Difference-curve helper
#
# anti-GBM minus LN
# ============================================================

def predict_difference(
    fit,
    data,
    grid,
    add_slide=False
):

    if not add_slide:

        pred = {}


        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({

                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(
                            grid
                        ),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })


            pred[
                disease
            ] = np.asarray(
                fit.predict(
                    newdata
                )
            )


        return (
            pred[
                "GBM"
            ]
            -
            pred[
                "SLE"
            ]
        )


    # --------------------------------------------------------
    # Slide-adjusted prediction:
    # equal average over represented slides
    # --------------------------------------------------------

    slide_categories = (
        data[
            "Slide"
        ].cat.categories
    )


    pred = {

        "SLE":
            [],

        "GBM":
            []
    }


    for slide in slide_categories:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({

                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(
                            grid
                        ),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide]
                        *
                        len(
                            grid
                        ),

                        categories=
                            slide_categories
                    )
            })


            pred[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            pred[
                "SLE"
            ]
        ),
        axis=0
    )


    mean_GBM = np.mean(
        np.vstack(
            pred[
                "GBM"
            ]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 19. Common PC1 support for curve comparison
# ============================================================

curve_low = max(

    unadjusted_result[
        "data"
    ][
        "PC1"
    ].min(),

    slide_result[
        "data"
    ][
        "PC1"
    ].min(),

    overlap_result[
        "data"
    ][
        "PC1"
    ].min()
)


curve_high = min(

    unadjusted_result[
        "data"
    ][
        "PC1"
    ].max(),

    slide_result[
        "data"
    ][
        "PC1"
    ].max(),

    overlap_result[
        "data"
    ][
        "PC1"
    ].max()
)


curve_grid = np.linspace(
    curve_low,
    curve_high,
    150
)


curve_unadjusted = (
    predict_difference(
        unadjusted_result[
            "fit"
        ],
        unadjusted_result[
            "data"
        ],
        curve_grid,
        add_slide=False
    )
)


curve_slide = (
    predict_difference(
        slide_result[
            "fit"
        ],
        slide_result[
            "data"
        ],
        curve_grid,
        add_slide=True
    )
)


curve_overlap = (
    predict_difference(
        overlap_result[
            "fit"
        ],
        overlap_result[
            "data"
        ],
        curve_grid,
        add_slide=True
    )
)


corr_slide = float(
    np.corrcoef(
        curve_unadjusted,
        curve_slide
    )[
        0,
        1
    ]
)


corr_overlap = float(
    np.corrcoef(
        curve_unadjusted,
        curve_overlap
    )[
        0,
        1
    ]
)


print(
    "\nCurve correlation vs unadjusted:"
)


print(
    "Slide-adjusted r =",
    corr_slide
)


print(
    "Overlap-slide restricted r =",
    corr_overlap
)


# ============================================================
# 20. Manuscript-level slide comparison
#
# These are rounded manuscript targets, NOT fitting inputs.
# ============================================================

slide_targets = pd.DataFrame({

    "analysis": [
        "Unadjusted",
        "Slide-adjusted",
        "Overlap-slide restricted"
    ],

    "manuscript_approx_p": [
        0.0179,
        0.0883,
        0.210
    ],

    "reproduced_p": [
        unadjusted_result[
            "pvalue"
        ],
        slide_result[
            "pvalue"
        ],
        overlap_result[
            "pvalue"
        ]
    ]
})


slide_targets[
    "absolute_difference_from_rounded"
] = (
    slide_targets[
        "reproduced_p"
    ]
    -
    slide_targets[
        "manuscript_approx_p"
    ]
).abs()


slide_targets[
    "status"
] = np.where(

    slide_targets[
        "absolute_difference_from_rounded"
    ]
    <
    0.001,

    "PASS",

    "CHECK"
)


print(
    "\nSlide manuscript-value check:"
)


display(
    slide_targets
)


# ============================================================
# 21. k sensitivity frozen-output audit
#
# k=6 can be cross-checked independently because the
# primary source table corresponds to the primary k.
#
# Other k values remain frozen sensitivity outputs in this
# reproducibility notebook unless ROI-level per-k source
# tables are available.
# ============================================================

saved_k = pd.read_csv(
    SAVED_K_PATH
)


k_macfib = (
    saved_k[
        saved_k[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
    .sort_values(
        "k"
    )
)


print(
    "\n" + "=" * 82
)

print(
    "K-SENSITIVITY FROZEN OUTPUT AUDIT"
)

print(
    "=" * 82
)


display(
    k_macfib
)


k_nominal_count = int(
    (
        k_macfib[
            "pvalue"
        ]
        <
        0.05
    ).sum()
)


print(
    "\nNominally significant k values:",
    f"{k_nominal_count}/{len(k_macfib)}"
)


# ------------------------------------------------------------
# Cross-check the k=6 result against the independently
# refitted primary model.
# ------------------------------------------------------------

k6_rows = (
    k_macfib[
        k_macfib[
            "k"
        ]
        ==
        6
    ]
)


if len(
    k6_rows
) == 1:

    k6_saved_p = float(
        k6_rows[
            "pvalue"
        ].iloc[
            0
        ]
    )


    k6_matches_primary = (
        abs(
            k6_saved_p
            -
            primary_p
        )
        <
        1e-5
    )


else:

    k6_saved_p = np.nan

    k6_matches_primary = False


print(
    "\nk=6 frozen P =",
    k6_saved_p
)


print(
    "Independently reproduced primary P =",
    primary_p
)


print(
    "k=6 cross-check:",
    (
        "PASS"
        if k6_matches_primary
        else
        "CHECK"
    )
)


# ============================================================
# 22. Scan project for possible raw per-k neighborhood sources
#
# This is informational only.
# ============================================================

candidate_k_files = sorted(
    set(
        list(
            BASE.glob(
                "*k*neighbor*.csv"
            )
        )
        +
        list(
            BASE.glob(
                "*neighbor*k*.csv"
            )
        )
    )
)


print(
    "\nPossible raw per-k neighborhood files:"
)


if len(
    candidate_k_files
) == 0:

    print(
        "None detected by filename pattern."
    )


else:

    for p in candidate_k_files:

        print(
            " ",
            p.name
        )


# ============================================================
# 23. Save independently reproduced outputs
# ============================================================

formal_path = (
    REPRO_RESULTS /
    "cell5_reproduced_formal_spatial_results.csv"
)


loo_path = (
    REPRO_RESULTS /
    "cell5_reproduced_GBM_LOO.csv"
)


spec_path = (
    REPRO_RESULTS /
    "cell5_reproduced_model_specification.csv"
)


slide_path = (
    REPRO_RESULTS /
    "cell5_reproduced_slide_sensitivity.csv"
)


formal_compare_path = (
    REPRO_RESULTS /
    "cell5_formal_spatial_comparison.csv"
)


formal_results.to_csv(
    formal_path,
    index=False
)


loo_reproduced.to_csv(
    loo_path,
    index=False
)


spec_reproduced.to_csv(
    spec_path,
    index=False
)


formal_compare.to_csv(
    formal_compare_path,
    index=False
)


slide_output = pd.DataFrame({

    "analysis": [
        "Unadjusted",
        "Slide-adjusted",
        "Overlap-slide restricted"
    ],

    "interaction_p": [
        unadjusted_result[
            "pvalue"
        ],

        slide_result[
            "pvalue"
        ],

        overlap_result[
            "pvalue"
        ]
    ],

    "curve_correlation_vs_unadjusted": [
        1.0,
        corr_slide,
        corr_overlap
    ],

    "n_ROI": [
        unadjusted_result[
            "n_ROI"
        ],

        slide_result[
            "n_ROI"
        ],

        overlap_result[
            "n_ROI"
        ]
    ],

    "n_patients": [
        unadjusted_result[
            "n_patients"
        ],

        slide_result[
            "n_patients"
        ],

        overlap_result[
            "n_patients"
        ]
    ]
})


slide_output.to_csv(
    slide_path,
    index=False
)


# ============================================================
# 24. Final PASS / CHECK summary
# ============================================================

formal_p_pass = bool(
    formal_compare[
        "p_match"
    ].all()
)


formal_fdr_pass = bool(
    formal_compare[
        "FDR_match"
    ].all()
)


loo_pass = bool(
    loo_compare[
        "p_match"
    ].all()
)


spec_pass = bool(
    spec_compare[
        "p_match"
    ].all()
)


slide_value_pass = bool(
    (
        slide_targets[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


curve_pass = (
    corr_slide
    >
    0.90
    and
    corr_overlap
    >
    0.90
)


k_output_pass = (
    k_nominal_count
    ==
    4
)


summary = pd.DataFrame({

    "check": [

        "Four formal spatial P values",

        "Four formal spatial FDR values",

        "Primary MAC–FIB FDR",

        "Leave-one-anti-GBM-out",

        "Model-specification sensitivity",

        "Slide sensitivity P values",

        "Slide-adjusted trajectory geometry",

        "k sensitivity frozen output",

        "k=6 independent cross-check"
    ],

    "result": [

        (
            "4/4 matched"
            if formal_p_pass
            else
            "check"
        ),

        (
            "4/4 matched"
            if formal_fdr_pass
            else
            "check"
        ),

        f"{primary_fdr:.6g}",

        (
            "5/5 matched; 5/5 P<0.05"
            if (
                loo_pass
                and
                loo_all_nominal
            )
            else
            "check"
        ),

        (
            "9/9 matched; 9/9 P<0.05"
            if (
                spec_pass
                and
                spec_all_nominal
                and
                spec_all_full_rank
            )
            else
            "check"
        ),

        (
            "matched rounded manuscript values"
            if slide_value_pass
            else
            "check"
        ),

        (
            f"r={corr_slide:.3f}, "
            f"{corr_overlap:.3f}"
        ),

        (
            "4/4 nominal P<0.05"
            if k_output_pass
            else
            "check"
        ),

        (
            "PASS"
            if k6_matches_primary
            else
            "CHECK"
        )
    ],

    "status": [

        (
            "PASS"
            if formal_p_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if formal_fdr_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if primary_fdr
            <
            0.05
            else
            "CHECK"
        ),

        (
            "PASS"
            if (
                loo_pass
                and
                loo_all_nominal
            )
            else
            "CHECK"
        ),

        (
            "PASS"
            if (
                spec_pass
                and
                spec_all_nominal
                and
                spec_all_full_rank
            )
            else
            "CHECK"
        ),

        (
            "PASS"
            if slide_value_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if curve_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if k_output_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if k6_matches_primary
            else
            "CHECK"
        )
    ]
})


print(
    "\n" + "=" * 82
)

print(
    "CELL 5 FINAL SUMMARY"
)

print(
    "=" * 82
)


display(
    summary
)


summary_path = (
    REPRO_RESULTS /
    "cell5_figure4_reproduction_summary.csv"
)


summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 25. Final decision
# ============================================================

critical_rows = [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    8
]


critical_pass = bool(
    (
        summary.loc[
            critical_rows,
            "status"
        ]
        ==
        "PASS"
    ).all()
)


print(
    "\n" + "=" * 82
)


if critical_pass:

    print(
        "✅ CELL 5 PASSED"
    )


    print(
        "Figure 4 core spatial statistics "
        "were independently reproduced."
    )


    print(
        "\nConfirmed:"
    )


    print(
        f"  • Primary MAC–FIB P = {primary_p:.6g}"
    )


    print(
        f"  • Primary MAC–FIB FDR = {primary_fdr:.6g}"
    )


    print(
        "  • leave-one-anti-GBM-out = 5/5 P<0.05"
    )


    print(
        "  • model specifications = 9/9 P<0.05"
    )


    print(
        f"  • slide-adjusted P = "
        f"{slide_result['pvalue']:.6g}"
    )


    print(
        f"  • overlap-slide P = "
        f"{overlap_result['pvalue']:.6g}"
    )


    print(
        f"  • curve correlations = "
        f"{corr_slide:.3f}, "
        f"{corr_overlap:.3f}"
    )


    print(
        "\nNOTE:"
    )


    print(
        "k=4/8/10 remain frozen-output audits; "
        "k=6 was independently cross-checked."
    )


else:

    print(
        "⚠ CELL 5 COMPLETED WITH CHECK ITEMS"
    )


    print(
        "Do not move to the final reproduction audit yet."
    )


    print(
        "Send me the CELL 5 FINAL SUMMARY "
        "and any comparison table containing CHECK."
    )


print(
    "=" * 82
)


print(
    "\nSaved:"
)


for path in [

    formal_path,
    loo_path,
    spec_path,
    slide_path,
    formal_compare_path,
    summary_path
]:

    print(
        path
    )

In [ ]:
# ============================================================
# REPRODUCTION — CELL 5B
#
# Diagnose remaining Figure 4 CHECK items
#
# Goals:
#
# 1. Show exactly WHICH of the four formal spatial
#    relationships differ from the saved final result
#
# 2. Audit relationship-specific valid PC1 support
#
# 3. Determine the original PC1 trimming rule by comparing:
#
#       A. pooled quantile trimming
#
#    versus
#
#       B. disease-wise quantile trimming followed by
#          common-support intersection
#
# 4. Do NOT alter any manuscript result
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_RESULTS = (
    BASE /
    "reproduction" /
    "results"
)


SPATIAL_PATH = (
    BASE /
    "figure4_driver_neighbor_data_smoothed.csv"
)


SAVED_PRIMARY_PATH = (
    BASE /
    "figure4_driver_results_smoothed.csv"
)


REPRO_PRIMARY_PATH = (
    REPRO_RESULTS /
    "cell5_reproduced_formal_spatial_results.csv"
)


SAVED_SPEC_PATH = (
    BASE /
    "figure4_final_model_specification_robustness.csv"
)


REPRO_SPEC_PATH = (
    REPRO_RESULTS /
    "cell5_reproduced_model_specification.csv"
)


for p in [
    SPATIAL_PATH,
    SAVED_PRIMARY_PATH,
    REPRO_PRIMARY_PATH,
    SAVED_SPEC_PATH,
    REPRO_SPEC_PATH
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing file:\n{p}"
        )


# ============================================================
# 2. Load
# ============================================================

spatial = pd.read_csv(
    SPATIAL_PATH,
    index_col=0
)


saved_primary = pd.read_csv(
    SAVED_PRIMARY_PATH
)


repro_primary = pd.read_csv(
    REPRO_PRIMARY_PATH
)


saved_spec = pd.read_csv(
    SAVED_SPEC_PATH
)


repro_spec = pd.read_csv(
    REPRO_SPEC_PATH
)


print(
    "=" * 84
)

print(
    "CELL 5B — FIGURE 4 DIAGNOSTIC"
)

print(
    "=" * 84
)


# ============================================================
# 3. FORMAL RELATIONSHIP COMPARISON
# ============================================================

formal_compare = (
    repro_primary[
        [
            "relationship",
            "pvalue",
            "FDR",
            "n_ROI",
            "n_patients",
            "n_GBM_patients"
        ]
    ]
    .merge(
        saved_primary[
            [
                c
                for c in [
                    "relationship",
                    "pvalue",
                    "FDR",
                    "n_ROI",
                    "n_patients",
                    "n_GBM_patients",
                    "matrix_rank",
                    "n_columns"
                ]
                if c in saved_primary.columns
            ]
        ],
        on="relationship",
        how="outer",
        suffixes=(
            "_reproduced",
            "_saved"
        )
    )
)


formal_compare[
    "p_abs_diff"
] = (
    formal_compare[
        "pvalue_reproduced"
    ]
    -
    formal_compare[
        "pvalue_saved"
    ]
).abs()


formal_compare[
    "FDR_abs_diff"
] = (
    formal_compare[
        "FDR_reproduced"
    ]
    -
    formal_compare[
        "FDR_saved"
    ]
).abs()


print(
    "\n" + "=" * 84
)

print(
    "A. EXACT FORMAL-RELATIONSHIP DIFFERENCES"
)

print(
    "=" * 84
)


display(
    formal_compare
)


# ============================================================
# 4. Identify formal relationship columns
# ============================================================

relationships = [

    x

    for x in [

        "MAC_to_FIB",

        "MAC_to_FibroticMC",

        "Mono_to_FIB",

        "Mono_to_FibroticMC"

    ]

    if x in spatial.columns
]


# ============================================================
# 5. Relationship-specific support audit
#
# For each relationship:
#
#   - drop missing outcome values
#   - examine SLE and GBM separately
#   - calculate raw overlap:
#
#       low  = max(disease minima)
#       high = min(disease maxima)
#
# ============================================================

support_rows = []


for relationship in relationships:

    d = (
        spatial[
            spatial[
                "Disease"
            ].isin(
                [
                    "SLE",
                    "GBM"
                ]
            )
        ]
        .dropna(
            subset=[
                relationship,
                "Disease",
                "Patient",
                "PC1"
            ]
        )
        .copy()
    )


    ranges = (
        d.groupby(
            "Disease",
            observed=True
        )[
            "PC1"
        ]
        .agg(
            [
                "min",
                "max",
                "count"
            ]
        )
    )


    if (
        "SLE"
        not in ranges.index
        or
        "GBM"
        not in ranges.index
    ):

        continue


    overlap_low = float(
        max(
            ranges.loc[
                "SLE",
                "min"
            ],

            ranges.loc[
                "GBM",
                "min"
            ]
        )
    )


    overlap_high = float(
        min(
            ranges.loc[
                "SLE",
                "max"
            ],

            ranges.loc[
                "GBM",
                "max"
            ]
        )
    )


    d_overlap = (
        d[
            d[
                "PC1"
            ].between(
                overlap_low,
                overlap_high
            )
        ]
    )


    saved_match = (
        saved_primary[
            saved_primary[
                "relationship"
            ]
            ==
            relationship
        ]
    )


    if (
        len(
            saved_match
        )
        ==
        1
        and
        "n_ROI"
        in saved_match.columns
    ):

        saved_n_roi = int(
            saved_match[
                "n_ROI"
            ].iloc[
                0
            ]
        )


    else:

        saved_n_roi = np.nan


    support_rows.append({

        "relationship":
            relationship,

        "valid_ROI_before_overlap":
            len(
                d
            ),

        "SLE_min":
            float(
                ranges.loc[
                    "SLE",
                    "min"
                ]
            ),

        "SLE_max":
            float(
                ranges.loc[
                    "SLE",
                    "max"
                ]
            ),

        "GBM_min":
            float(
                ranges.loc[
                    "GBM",
                    "min"
                ]
            ),

        "GBM_max":
            float(
                ranges.loc[
                    "GBM",
                    "max"
                ]
            ),

        "relationship_overlap_low":
            overlap_low,

        "relationship_overlap_high":
            overlap_high,

        "ROI_after_relationship_overlap":
            len(
                d_overlap
            ),

        "saved_n_ROI":
            saved_n_roi,

        "overlap_n_matches_saved":
            (
                len(
                    d_overlap
                )
                ==
                saved_n_roi
            )
            if np.isfinite(
                saved_n_roi
            )
            else False
    })


support_audit = pd.DataFrame(
    support_rows
)


print(
    "\n" + "=" * 84
)

print(
    "B. RELATIONSHIP-SPECIFIC COMMON SUPPORT"
)

print(
    "=" * 84
)


display(
    support_audit
)


# ============================================================
# 6. Saved model-specification table
# ============================================================

saved_macfib_spec = (
    saved_spec[
        saved_spec[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


print(
    "\n" + "=" * 84
)

print(
    "C. SAVED MAC–FIB MODEL-SPECIFICATION RESULTS"
)

print(
    "=" * 84
)


display(
    saved_macfib_spec
)


# ============================================================
# 7. Reproduced model-spec table from Cell 5
# ============================================================

repro_macfib_spec = (
    repro_spec[
        repro_spec[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


print(
    "\n" + "=" * 84
)

print(
    "D. CELL 5 MAC–FIB MODEL-SPECIFICATION RESULTS"
)

print(
    "=" * 84
)


display(
    repro_macfib_spec
)


# ============================================================
# 8. Build the exact primary MAC–FIB source data
# ============================================================

d = (
    spatial[
        spatial[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ]
    .dropna(
        subset=[
            "MAC_to_FIB",
            "Disease",
            "Patient",
            "PC1"
        ]
    )
    .copy()
)


# ============================================================
# 9. Compare TWO possible trimming rules
#
# Rule A:
# pooled quantile trim
#
# Rule B:
# disease-specific quantiles,
# followed by intersection of the two disease ranges
# ============================================================

trim_rows = []


for trim in [
    0.000,
    0.025,
    0.050
]:

    # --------------------------------------------------------
    # RULE A — pooled quantiles
    # --------------------------------------------------------

    if trim == 0:

        pooled_low = float(
            d[
                "PC1"
            ].min()
        )


        pooled_high = float(
            d[
                "PC1"
            ].max()
        )


    else:

        pooled_low = float(
            d[
                "PC1"
            ].quantile(
                trim
            )
        )


        pooled_high = float(
            d[
                "PC1"
            ].quantile(
                1.0
                -
                trim
            )
        )


    pooled_n = int(
        d[
            "PC1"
        ]
        .between(
            pooled_low,
            pooled_high
        )
        .sum()
    )


    # --------------------------------------------------------
    # RULE B — disease-wise quantiles then intersection
    # --------------------------------------------------------

    disease_bounds = {}


    for disease in [
        "SLE",
        "GBM"
    ]:

        x = (
            d.loc[
                d[
                    "Disease"
                ]
                ==
                disease,
                "PC1"
            ]
        )


        if trim == 0:

            low_d = float(
                x.min()
            )


            high_d = float(
                x.max()
            )


        else:

            low_d = float(
                x.quantile(
                    trim
                )
            )


            high_d = float(
                x.quantile(
                    1.0
                    -
                    trim
                )
            )


        disease_bounds[
            disease
        ] = (
            low_d,
            high_d
        )


    diseasewise_low = float(
        max(
            disease_bounds[
                "SLE"
            ][
                0
            ],

            disease_bounds[
                "GBM"
            ][
                0
            ]
        )
    )


    diseasewise_high = float(
        min(
            disease_bounds[
                "SLE"
            ][
                1
            ],

            disease_bounds[
                "GBM"
            ][
                1
            ]
        )
    )


    diseasewise_n = int(
        d[
            "PC1"
        ]
        .between(
            diseasewise_low,
            diseasewise_high
        )
        .sum()
    )


    # --------------------------------------------------------
    # Saved values
    # --------------------------------------------------------

    saved_trim_rows = (
        saved_macfib_spec[
            np.isclose(
                saved_macfib_spec[
                    "PC1_trim"
                ],
                trim
            )
        ]
    )


    saved_low = np.nan
    saved_high = np.nan
    saved_n = np.nan


    if len(
        saved_trim_rows
    ) > 0:

        if (
            "PC1_low"
            in saved_trim_rows.columns
        ):

            saved_low = float(
                saved_trim_rows[
                    "PC1_low"
                ].iloc[
                    0
                ]
            )


        if (
            "PC1_high"
            in saved_trim_rows.columns
        ):

            saved_high = float(
                saved_trim_rows[
                    "PC1_high"
                ].iloc[
                    0
                ]
            )


        if (
            "n_ROI"
            in saved_trim_rows.columns
        ):

            saved_n = int(
                saved_trim_rows[
                    "n_ROI"
                ].iloc[
                    0
                ]
            )


    trim_rows.append({

        "PC1_trim":
            trim,

        "saved_low":
            saved_low,

        "saved_high":
            saved_high,

        "saved_n_ROI":
            saved_n,

        "pooled_low":
            pooled_low,

        "pooled_high":
            pooled_high,

        "pooled_n_ROI":
            pooled_n,

        "diseasewise_low":
            diseasewise_low,

        "diseasewise_high":
            diseasewise_high,

        "diseasewise_n_ROI":
            diseasewise_n
    })


trim_diagnostic = pd.DataFrame(
    trim_rows
)


# ============================================================
# 10. Difference from saved
# ============================================================

for prefix in [
    "pooled",
    "diseasewise"
]:

    trim_diagnostic[
        f"{prefix}_low_abs_diff"
    ] = (
        trim_diagnostic[
            f"{prefix}_low"
        ]
        -
        trim_diagnostic[
            "saved_low"
        ]
    ).abs()


    trim_diagnostic[
        f"{prefix}_high_abs_diff"
    ] = (
        trim_diagnostic[
            f"{prefix}_high"
        ]
        -
        trim_diagnostic[
            "saved_high"
        ]
    ).abs()


    trim_diagnostic[
        f"{prefix}_n_diff"
    ] = (
        trim_diagnostic[
            f"{prefix}_n_ROI"
        ]
        -
        trim_diagnostic[
            "saved_n_ROI"
        ]
    ).abs()


print(
    "\n" + "=" * 84
)

print(
    "E. WHICH PC1 TRIMMING RULE MATCHES THE ORIGINAL ANALYSIS?"
)

print(
    "=" * 84
)


display(
    trim_diagnostic
)


# ============================================================
# 11. Automatic diagnostic interpretation
# ============================================================

pooled_score = (
    trim_diagnostic[
        "pooled_low_abs_diff"
    ].fillna(
        999
    ).sum()
    +
    trim_diagnostic[
        "pooled_high_abs_diff"
    ].fillna(
        999
    ).sum()
    +
    trim_diagnostic[
        "pooled_n_diff"
    ].fillna(
        999
    ).sum()
)


diseasewise_score = (
    trim_diagnostic[
        "diseasewise_low_abs_diff"
    ].fillna(
        999
    ).sum()
    +
    trim_diagnostic[
        "diseasewise_high_abs_diff"
    ].fillna(
        999
    ).sum()
    +
    trim_diagnostic[
        "diseasewise_n_diff"
    ].fillna(
        999
    ).sum()
)


print(
    "\n" + "=" * 84
)

print(
    "DIAGNOSTIC INTERPRETATION"
)

print(
    "=" * 84
)


print(
    "Pooled-trim mismatch score:",
    pooled_score
)


print(
    "Disease-wise trim mismatch score:",
    diseasewise_score
)


if (
    diseasewise_score
    <
    pooled_score
):

    print(
        "\n✅ Disease-wise quantile trimming "
        "matches the original analysis better."
    )


    print(
        "\nLikely original rule:"
    )


    print(
        "For each disease separately:"
    )


    print(
        "  calculate lower/upper PC1 quantiles"
    )


    print(
        "then take the intersection of "
        "the disease-specific ranges."
    )


else:

    print(
        "\n⚠ Pooled trimming fits at least as well."
    )


    print(
        "We need to inspect the saved table more closely."
    )


# ============================================================
# 12. Save diagnostics
# ============================================================

formal_diag_path = (
    REPRO_RESULTS /
    "cell5B_formal_relationship_diagnostic.csv"
)


support_diag_path = (
    REPRO_RESULTS /
    "cell5B_relationship_support_diagnostic.csv"
)


trim_diag_path = (
    REPRO_RESULTS /
    "cell5B_PC1_trimming_rule_diagnostic.csv"
)


formal_compare.to_csv(
    formal_diag_path,
    index=False
)


support_audit.to_csv(
    support_diag_path,
    index=False
)


trim_diagnostic.to_csv(
    trim_diag_path,
    index=False
)


print(
    "\nSaved:"
)


print(
    formal_diag_path
)


print(
    support_diag_path
)


print(
    trim_diag_path
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 5C
#
# Resolve the remaining Figure 4 reproduction issue
#
# Part A:
#   Refit the 9 model-specification analyses using the
#   EXACT PC1_low / PC1_high values stored in the original
#   final specification table.
#
#   If 9/9 P values reproduce:
#       statistical model = confirmed
#       remaining question = provenance of trimming bounds only
#
# Part B:
#   Search all old Jupyter notebooks in the CGN project
#   for the ORIGINAL trimming code.
#
# No manuscript values are changed.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_RESULTS = (
    BASE /
    "reproduction" /
    "results"
)


REPRO_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


SPATIAL_PATH = (
    BASE /
    "figure4_driver_neighbor_data_smoothed.csv"
)


SAVED_PRIMARY_PATH = (
    BASE /
    "figure4_driver_results_smoothed.csv"
)


SAVED_SPEC_PATH = (
    BASE /
    "figure4_final_model_specification_robustness.csv"
)


for path in [
    SPATIAL_PATH,
    SAVED_PRIMARY_PATH,
    SAVED_SPEC_PATH
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )


# ============================================================
# 2. Load
# ============================================================

spatial = pd.read_csv(
    SPATIAL_PATH,
    index_col=0
)


spatial.index = (
    spatial.index.astype(str)
)


saved_primary = pd.read_csv(
    SAVED_PRIMARY_PATH
)


saved_spec = pd.read_csv(
    SAVED_SPEC_PATH
)


print(
    "=" * 84
)

print(
    "CELL 5C — FIGURE 4 FINAL DIAGNOSTIC"
)

print(
    "=" * 84
)


# ============================================================
# 3. First fix the overly strict floating-point comparison
#
# These are reproduction diagnostics only.
#
# atol=1e-7 is far smaller than any reported manuscript
# precision and is appropriate for saved CSV roundoff.
# ============================================================

def numerical_match(
    x,
    y,
    rtol=1e-6,
    atol=1e-7
):

    if (
        not np.isfinite(
            x
        )
        or
        not np.isfinite(
            y
        )
    ):

        return False


    return bool(
        np.isclose(
            float(
                x
            ),
            float(
                y
            ),
            rtol=rtol,
            atol=atol
        )
    )


# ============================================================
# 4. Formal four-relationship audit with sensible tolerance
# ============================================================

repro_primary_path = (
    REPRO_RESULTS /
    "cell5_reproduced_formal_spatial_results.csv"
)


if not repro_primary_path.exists():

    raise FileNotFoundError(
        "Run Cell 5 first."
    )


repro_primary = pd.read_csv(
    repro_primary_path
)


formal_compare = (
    repro_primary[
        [
            "relationship",
            "pvalue",
            "FDR"
        ]
    ]
    .merge(
        saved_primary[
            [
                "relationship",
                "pvalue",
                "FDR"
            ]
        ],
        on="relationship",
        suffixes=(
            "_reproduced",
            "_saved"
        )
    )
)


formal_compare[
    "p_abs_diff"
] = (
    formal_compare[
        "pvalue_reproduced"
    ]
    -
    formal_compare[
        "pvalue_saved"
    ]
).abs()


formal_compare[
    "FDR_abs_diff"
] = (
    formal_compare[
        "FDR_reproduced"
    ]
    -
    formal_compare[
        "FDR_saved"
    ]
).abs()


formal_compare[
    "p_match"
] = [

    numerical_match(
        x,
        y
    )

    for x, y in zip(
        formal_compare[
            "pvalue_reproduced"
        ],
        formal_compare[
            "pvalue_saved"
        ]
    )
]


formal_compare[
    "FDR_match"
] = [

    numerical_match(
        x,
        y
    )

    for x, y in zip(
        formal_compare[
            "FDR_reproduced"
        ],
        formal_compare[
            "FDR_saved"
        ]
    )
]


print(
    "\n" + "=" * 84
)

print(
    "A. FOUR FORMAL SPATIAL RESULTS — NUMERICAL AUDIT"
)

print(
    "=" * 84
)


display(
    formal_compare
)


formal_results_pass = bool(

    formal_compare[
        "p_match"
    ].all()

    and

    formal_compare[
        "FDR_match"
    ].all()
)


print(
    "\nFormal spatial results:",
    (
        "PASS ✓"
        if formal_results_pass
        else
        "CHECK"
    )
)


# ============================================================
# 5. Primary source data for MAC→FIB
# ============================================================

primary_data = (
    spatial[
        spatial[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ]
    .dropna(
        subset=[
            "MAC_to_FIB",
            "Disease",
            "Patient",
            "PC1"
        ]
    )
    .copy()
)


print(
    "\nPrimary MAC→FIB source ROIs:",
    len(
        primary_data
    )
)


# ============================================================
# 6. Generic patient-balanced spline model
# ============================================================

def fit_model(
    data,
    spline_df
):

    d = (
        data[
            [
                "MAC_to_FIB",
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .dropna()
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),

        categories=[
            "SLE",
            "GBM"
        ]
    )


    d = (
        d[
            d[
                "Disease"
            ].notna()
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    formula = (
        "MAC_to_FIB ~ "
        f"bs(PC1, df={int(spline_df)}, "
        "degree=3, include_intercept=False) "
        "* C(Disease)"
    )


    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    matrix_rank = int(
        np.linalg.matrix_rank(
            fit.model.exog
        )
    )


    n_columns = int(
        fit.model.exog.shape[
            1
        ]
    )


    interaction_terms = [

        term

        for term
        in fit.params.index

        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    if len(
        interaction_terms
    ) == 0:

        raise ValueError(
            "No Disease × PC1 terms found."
        )


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    pvalue = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return {

        "pvalue":
            pvalue,

        "matrix_rank":
            matrix_rank,

        "n_columns":
            n_columns,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ]
                .astype(str)
                ==
                "GBM",
                "Patient"
            ].nunique()
    }


# ============================================================
# 7. Saved formal model specifications
# ============================================================

saved_macfib_spec = (
    saved_spec[
        saved_spec[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
    .sort_values(
        [
            "PC1_trim",
            "spline_df"
        ]
    )
    .reset_index(
        drop=True
    )
)


required_spec_cols = [
    "PC1_trim",
    "spline_df",
    "PC1_low",
    "PC1_high",
    "pvalue"
]


missing_spec_cols = [

    c

    for c in required_spec_cols

    if c not in saved_macfib_spec.columns
]


if len(
    missing_spec_cols
) > 0:

    raise ValueError(
        "Saved specification table missing:\n"
        +
        "\n".join(
            missing_spec_cols
        )
    )


print(
    "\n" + "=" * 84
)

print(
    "B. ORIGINAL FROZEN MODEL SPECIFICATIONS"
)

print(
    "=" * 84
)


display(
    saved_macfib_spec
)


# ============================================================
# 8. IMPORTANT:
#
# Refit using the EXACT original saved PC1 support.
#
# This tests whether:
#
#     model + weights + clustering + Wald test
#
# reproduce independently once the originally prespecified
# support boundaries are restored.
# ============================================================

refit_rows = []


for row in saved_macfib_spec.itertuples():

    trim = float(
        row.PC1_trim
    )


    spline_df = int(
        row.spline_df
    )


    pc_low = float(
        row.PC1_low
    )


    pc_high = float(
        row.PC1_high
    )


    saved_p = float(
        row.pvalue
    )


    d_spec = (
        primary_data[
            primary_data[
                "PC1"
            ].between(
                pc_low,
                pc_high
            )
        ]
        .copy()
    )


    result = fit_model(
        d_spec,
        spline_df
    )


    reproduced_p = float(
        result[
            "pvalue"
        ]
    )


    p_match = numerical_match(
        reproduced_p,
        saved_p,
        rtol=1e-5,
        atol=1e-8
    )


    saved_n_roi = (

        int(
            row.n_ROI
        )

        if hasattr(
            row,
            "n_ROI"
        )

        and

        pd.notna(
            row.n_ROI
        )

        else

        np.nan
    )


    n_roi_match = (

        result[
            "n_ROI"
        ]
        ==
        saved_n_roi

        if np.isfinite(
            saved_n_roi
        )

        else

        True
    )


    refit_rows.append({

        "PC1_trim":
            trim,

        "spline_df":
            spline_df,

        "PC1_low":
            pc_low,

        "PC1_high":
            pc_high,

        "saved_pvalue":
            saved_p,

        "reproduced_pvalue":
            reproduced_p,

        "p_abs_diff":
            abs(
                reproduced_p
                -
                saved_p
            ),

        "p_match":
            p_match,

        "saved_n_ROI":
            saved_n_roi,

        "reproduced_n_ROI":
            result[
                "n_ROI"
            ],

        "n_ROI_match":
            n_roi_match,

        "matrix_rank":
            result[
                "matrix_rank"
            ],

        "n_columns":
            result[
                "n_columns"
            ],

        "full_rank":
            (
                result[
                    "matrix_rank"
                ]
                ==
                result[
                    "n_columns"
                ]
            )
    })


refit_spec = pd.DataFrame(
    refit_rows
)


print(
    "\n" + "=" * 84
)

print(
    "C. SAVED-BOUND MODEL-SPECIFICATION REPRODUCTION"
)

print(
    "=" * 84
)


display(
    refit_spec
)


spec_p_pass = bool(
    refit_spec[
        "p_match"
    ].all()
)


spec_n_pass = bool(
    refit_spec[
        "n_ROI_match"
    ].all()
)


spec_rank_pass = bool(
    refit_spec[
        "full_rank"
    ].all()
)


print(
    "\nP values matched:",
    f"{int(refit_spec['p_match'].sum())}"
    "/"
    f"{len(refit_spec)}"
)


print(
    "ROI counts matched:",
    f"{int(refit_spec['n_ROI_match'].sum())}"
    "/"
    f"{len(refit_spec)}"
)


print(
    "Full-rank models:",
    f"{int(refit_spec['full_rank'].sum())}"
    "/"
    f"{len(refit_spec)}"
)


# ============================================================
# 9. Search ALL old notebooks for original trimming code
#
# Jupyter notebooks are JSON files.
# We inspect only code-cell text.
#
# Search terms deliberately include:
#   PC1_trim
#   figure4_final_model_specification
#   quantile
#   MAC_to_FIB
#   0.025 / 0.050
#   PC1_low / PC1_high
# ============================================================

search_terms = [

    "PC1_trim",

    "figure4_final_model_specification",

    "model_specification",

    "PC1_low",

    "PC1_high",

    "quantile",

    "MAC_to_FIB",

    "0.025",

    "0.050"
]


notebook_paths = sorted(
    BASE.rglob(
        "*.ipynb"
    )
)


print(
    "\n" + "=" * 84
)

print(
    "D. SEARCHING OLD NOTEBOOKS FOR ORIGINAL TRIMMING CODE"
)

print(
    "=" * 84
)


print(
    "Notebooks found:",
    len(
        notebook_paths
    )
)


notebook_hits = []


for notebook_path in notebook_paths:

    # Do not search the current reproduction notebook first
    # because it would produce lots of irrelevant matches.
    if (
        "REPRODUCTION"
        in notebook_path.name.upper()
    ):

        continue


    try:

        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:

            notebook_json = json.load(
                f
            )


    except Exception:

        continue


    cells_json = notebook_json.get(
        "cells",
        []
    )


    for cell_number, cell in enumerate(
        cells_json
    ):

        if (
            cell.get(
                "cell_type"
            )
            !=
            "code"
        ):

            continue


        source = cell.get(
            "source",
            []
        )


        if isinstance(
            source,
            list
        ):

            code = "".join(
                source
            )


        else:

            code = str(
                source
            )


        code_lower = (
            code.lower()
        )


        matched_terms = [

            term

            for term
            in search_terms

            if term.lower()
            in code_lower
        ]


        # Require at least TWO matching terms.
        # This removes many irrelevant quantile cells.
        if len(
            matched_terms
        ) >= 2:

            notebook_hits.append({

                "notebook":
                    notebook_path.name,

                "cell_number":
                    cell_number,

                "matched_terms":
                    ", ".join(
                        matched_terms
                    ),

                "code":
                    code
            })


print(
    "\nPotential original-code cells found:",
    len(
        notebook_hits
    )
)


# ============================================================
# 10. Print the strongest candidate cells
# ============================================================

if len(
    notebook_hits
) > 0:

    # Rank candidates by number of search-term matches
    for hit in notebook_hits:

        hit[
            "match_count"
        ] = len(
            hit[
                "matched_terms"
            ].split(
                ", "
            )
        )


    notebook_hits_sorted = sorted(

        notebook_hits,

        key=lambda x:
            x[
                "match_count"
            ],

        reverse=True
    )


    print(
        "\n" + "=" * 84
    )

    print(
        "TOP ORIGINAL-CODE CANDIDATES"
    )

    print(
        "=" * 84
    )


    for i, hit in enumerate(
        notebook_hits_sorted[
            :8
        ]
    ):

        print(
            "\n"
            +
            "-" * 84
        )


        print(
            f"CANDIDATE {i + 1}"
        )


        print(
            "Notebook:",
            hit[
                "notebook"
            ]
        )


        print(
            "Cell:",
            hit[
                "cell_number"
            ]
        )


        print(
            "Matched:",
            hit[
                "matched_terms"
            ]
        )


        print(
            "-" * 84
        )


        # Limit screen output but keep enough code
        print(
            hit[
                "code"
            ][
                :5000
            ]
        )


else:

    print(
        "\nNo strong notebook code candidates found."
    )


# ============================================================
# 11. Save ALL matching code cells to a text file
# ============================================================

code_search_path = (
    REPRO_RESULTS /
    "cell5C_original_trimming_code_search.txt"
)


with open(
    code_search_path,
    "w",
    encoding="utf-8"
) as f:

    for i, hit in enumerate(
        notebook_hits
    ):

        f.write(
            "=" * 100
            +
            "\n"
        )


        f.write(
            f"HIT {i + 1}\n"
        )


        f.write(
            f"Notebook: {hit['notebook']}\n"
        )


        f.write(
            f"Cell: {hit['cell_number']}\n"
        )


        f.write(
            f"Matched: {hit['matched_terms']}\n"
        )


        f.write(
            "=" * 100
            +
            "\n\n"
        )


        f.write(
            hit[
                "code"
            ]
        )


        f.write(
            "\n\n\n"
        )


# ============================================================
# 12. Save reproduction tables
# ============================================================

formal_audit_path = (
    REPRO_RESULTS /
    "cell5C_formal_numeric_audit.csv"
)


spec_refit_path = (
    REPRO_RESULTS /
    "cell5C_saved_bound_specification_refit.csv"
)


formal_compare.to_csv(
    formal_audit_path,
    index=False
)


refit_spec.to_csv(
    spec_refit_path,
    index=False
)


# ============================================================
# 13. Final Cell 5C status
# ============================================================

print(
    "\n" + "=" * 84
)

print(
    "CELL 5C FINAL STATUS"
)

print(
    "=" * 84
)


summary = pd.DataFrame({

    "check": [

        "Four formal spatial P/FDR values",

        "Nine model-specification P values",

        "Nine model-specification ROI counts",

        "Nine model-specification model ranks"
    ],

    "status": [

        (
            "PASS"
            if formal_results_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if spec_p_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if spec_n_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if spec_rank_pass
            else
            "CHECK"
        )
    ]
})


display(
    summary
)


if (
    formal_results_pass
    and
    spec_p_pass
    and
    spec_n_pass
    and
    spec_rank_pass
):

    print(
        "\n✅ FIGURE 4 STATISTICAL MODELS REPRODUCED"
    )


    print(
        "\nThe remaining task is only to recover/document "
        "the original rule that generated the frozen "
        "PC1 trimming bounds."
    )


    print(
        "\nThis does NOT affect the reproduced P values."
    )


else:

    print(
        "\n⚠ At least one statistical item still needs review."
    )


print(
    "\nSaved:"
)


print(
    formal_audit_path
)


print(
    spec_refit_path
)


print(
    code_search_path
)


print(
    "=" * 84
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 6
#
# FINAL MASTER REPRODUCTION AUDIT
#
# Summarizes independent reproduction of:
#
#   Figure 1
#   Figure 2
#   Figure 3
#   Figure 4
#
# No statistical model is fitted in this cell.
# It only audits results already regenerated in Cells 1–5.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_ROOT = (
    BASE /
    "reproduction"
)


REPRO_RESULTS = (
    REPRO_ROOT /
    "results"
)


REPRO_LOGS = (
    REPRO_ROOT /
    "logs"
)


REPRO_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Required reproduction outputs
# ============================================================

FILES = {

    # --------------------------------------------------------
    # Cell 1
    # --------------------------------------------------------

    "cell1":
        REPRO_LOGS /
        "cell1_data_integrity_summary.csv",


    # --------------------------------------------------------
    # Figure 1
    # --------------------------------------------------------

    "figure1":
        REPRO_RESULTS /
        "cell2_figure1_reproduction.csv",


    # --------------------------------------------------------
    # Figure 2
    # --------------------------------------------------------

    "figure2":
        REPRO_RESULTS /
        "cell3_figure2_reproduction_check.csv",

    "figure2_summary":
        REPRO_RESULTS /
        "cell3_figure2_manuscript_summary.csv",


    # --------------------------------------------------------
    # Figure 3
    # --------------------------------------------------------

    "figure3_gene_family":
        REPRO_RESULTS /
        "cell4B_formal_FDR_family_summary.csv",

    "figure3_module":
        REPRO_RESULTS /
        "cell4_module_sensitivity_comparison.csv",

    "figure3_module_scores":
        REPRO_RESULTS /
        "cell4_figure3_reproduction_summary.csv",


    # --------------------------------------------------------
    # Figure 4
    # --------------------------------------------------------

    "figure4_formal":
        REPRO_RESULTS /
        "cell5C_formal_numeric_audit.csv",

    "figure4_spec":
        REPRO_RESULTS /
        "cell5C_saved_bound_specification_refit.csv",

    "figure4_summary":
        REPRO_RESULTS /
        "cell5_figure4_reproduction_summary.csv"
}


# ============================================================
# 3. File-existence audit
# ============================================================

file_rows = []


for label, path in FILES.items():

    file_rows.append({

        "component":
            label,

        "exists":
            path.exists(),

        "path":
            str(
                path
            )
    })


file_audit = pd.DataFrame(
    file_rows
)


print(
    "=" * 88
)

print(
    "CELL 6 — FINAL MASTER REPRODUCTION AUDIT"
)

print(
    "=" * 88
)


print(
    "\nRequired reproduction outputs:"
)


display(
    file_audit
)


missing = (
    file_audit[
        ~file_audit[
            "exists"
        ]
    ]
)


if len(
    missing
) > 0:

    print(
        "\n❌ Missing reproduction outputs:"
    )


    display(
        missing
    )


    raise FileNotFoundError(
        """
One or more reproduction output files are missing.

Do not recreate them manually.
Send the missing-file table to me.
"""
    )


# ============================================================
# 4. FIGURE 1 AUDIT
# ============================================================

fig1 = pd.read_csv(
    FILES[
        "figure1"
    ]
)


fig1_pass = bool(
    (
        fig1[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


# capture important values
def get_metric_value(
    df,
    metric
):

    rows = (
        df[
            df[
                "metric"
            ]
            ==
            metric
        ]
    )


    if len(
        rows
    ) == 0:

        return np.nan


    return (
        rows[
            "reproduced"
        ]
        .iloc[
            0
        ]
    )


fig1_rho = get_metric_value(
    fig1,
    "PC1 vs DPT Spearman rho"
)


fig1_low = get_metric_value(
    fig1,
    "Common PC1 low"
)


fig1_high = get_metric_value(
    fig1,
    "Common PC1 high"
)


# ============================================================
# 5. FIGURE 2 AUDIT
# ============================================================

fig2 = pd.read_csv(
    FILES[
        "figure2"
    ]
)


fig2_summary = pd.read_csv(
    FILES[
        "figure2_summary"
    ]
)


fig2_numeric_pass = bool(
    (
        fig2[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


n_shared_sig = int(
    fig2_summary[
        "shared_significant_FDR005"
    ].sum()
)


n_interaction_sig = int(
    fig2_summary[
        "interaction_significant_FDR005"
    ].sum()
)


interaction_celltypes = (
    fig2_summary.loc[
        fig2_summary[
            "interaction_significant_FDR005"
        ],
        "celltype"
    ]
    .astype(str)
    .tolist()
)


fig2_pattern_pass = (
    n_shared_sig
    ==
    7
    and
    n_interaction_sig
    ==
    1
    and
    interaction_celltypes
    ==
    [
        "PEC"
    ]
)


fig2_pass = (
    fig2_numeric_pass
    and
    fig2_pattern_pass
)


# ============================================================
# 6. FIGURE 3 AUDIT
# ============================================================

fig3_family = pd.read_csv(
    FILES[
        "figure3_gene_family"
    ]
)


fig3_module = pd.read_csv(
    FILES[
        "figure3_module"
    ]
)


fig3_summary = pd.read_csv(
    FILES[
        "figure3_module_scores"
    ]
)


# ------------------------------------------------------------
# Formal gene-testing family
# ------------------------------------------------------------

formal_family_pass = bool(
    (
        fig3_family[
            "formal_test_status"
        ]
        ==
        "PASS"
    ).all()
    and
    (
        fig3_family[
            "significant_count_status"
        ]
        ==
        "PASS"
    ).all()
)


mac_formal_tests = int(
    fig3_family.loc[
        fig3_family[
            "celltype"
        ]
        ==
        "MAC",
        "reproduced_formal_tests"
    ].iloc[
        0
    ]
)


fib_formal_tests = int(
    fig3_family.loc[
        fig3_family[
            "celltype"
        ]
        ==
        "FIB",
        "reproduced_formal_tests"
    ].iloc[
        0
    ]
)


mac_sig_genes = int(
    fig3_family.loc[
        fig3_family[
            "celltype"
        ]
        ==
        "MAC",
        "reproduced_FDR_lt_005"
    ].iloc[
        0
    ]
)


fib_sig_genes = int(
    fig3_family.loc[
        fig3_family[
            "celltype"
        ]
        ==
        "FIB",
        "reproduced_FDR_lt_005"
    ].iloc[
        0
    ]
)


# ------------------------------------------------------------
# 9 selected module sensitivity models
# ------------------------------------------------------------

module_p_pass = bool(
    fig3_module[
        "p_match"
    ].all()
)


n_module_p_match = int(
    fig3_module[
        "p_match"
    ].sum()
)


# ------------------------------------------------------------
# Frozen module sizes / scores / rank
# ------------------------------------------------------------

# Ignore the original Cell-4 FDR-family CHECK rows here,
# because Cell 4B subsequently resolved that bookkeeping
# issue exactly.
fig3_key_checks = (
    fig3_summary[
        fig3_summary[
            "check"
        ].isin(
            [
                "Frozen module sizes",
                "Recalculated module scores",
                "Selected module P values",
                "Selected module model rank"
            ]
        )
    ]
)


module_structure_pass = bool(
    (
        fig3_key_checks[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


fig3_pass = (
    formal_family_pass
    and
    module_p_pass
    and
    module_structure_pass
)


# ============================================================
# 7. FIGURE 4 AUDIT
# ============================================================

fig4_formal = pd.read_csv(
    FILES[
        "figure4_formal"
    ]
)


fig4_spec = pd.read_csv(
    FILES[
        "figure4_spec"
    ]
)


fig4_summary = pd.read_csv(
    FILES[
        "figure4_summary"
    ]
)


# ------------------------------------------------------------
# Four formal P/FDR results
# ------------------------------------------------------------

formal_spatial_pass = bool(
    fig4_formal[
        "p_match"
    ].all()
    and
    fig4_formal[
        "FDR_match"
    ].all()
)


# ------------------------------------------------------------
# 9 saved-bound specification refits
# ------------------------------------------------------------

spec_pass = bool(
    fig4_spec[
        "p_match"
    ].all()
    and
    fig4_spec[
        "n_ROI_match"
    ].all()
    and
    fig4_spec[
        "full_rank"
    ].all()
)


n_spec_pass = int(
    fig4_spec[
        "p_match"
    ].sum()
)


# ------------------------------------------------------------
# Secondary Figure 4 robustness checks
#
# Use the rows that were already PASS in Cell 5:
#   Primary FDR
#   LOO
#   slide sensitivity
#   trajectory geometry
#   k sensitivity
#   k=6 cross-check
#
# Do NOT use the earlier model-spec CHECK row because
# Cell 5C subsequently resolved it.
# ------------------------------------------------------------

fig4_secondary_labels = [

    "Primary MAC–FIB FDR",

    "Leave-one-anti-GBM-out",

    "Slide sensitivity P values",

    "Slide-adjusted trajectory geometry",

    "k sensitivity frozen output",

    "k=6 independent cross-check"
]


fig4_secondary = (
    fig4_summary[
        fig4_summary[
            "check"
        ].isin(
            fig4_secondary_labels
        )
    ]
)


fig4_secondary_pass = bool(
    (
        fig4_secondary[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


fig4_pass = (
    formal_spatial_pass
    and
    spec_pass
    and
    fig4_secondary_pass
)


# ============================================================
# 8. Build MASTER SUMMARY
# ============================================================

master_rows = [

    # --------------------------------------------------------
    # Input integrity
    # --------------------------------------------------------

    {
        "section":
            "Input audit",

        "check":
            "Core data integrity",

        "key_result":
            (
                "3,218,210 × 480 Xenium; "
                "782 × 480 ROI object; "
                "586,628 ROI-associated cells"
            ),

        "status":
            "PASS"
    },


    # --------------------------------------------------------
    # Figure 1
    # --------------------------------------------------------

    {
        "section":
            "Figure 1",

        "check":
            "Study cohort / ROI framework",

        "key_result":
            (
                "63 patients; 782 ROIs"
            ),

        "status":
            (
                "PASS"
                if fig1_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 1",

        "check":
            "Common PC1 support",

        "key_result":
            (
                f"{float(fig1_low):.6f} "
                f"to "
                f"{float(fig1_high):.6f}"
            ),

        "status":
            (
                "PASS"
                if fig1_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 1",

        "check":
            "PC1–DPT agreement",

        "key_result":
            (
                f"Spearman rho = "
                f"{float(fig1_rho):.6f}"
            ),

        "status":
            (
                "PASS"
                if fig1_pass
                else
                "CHECK"
            )
    },


    # --------------------------------------------------------
    # Figure 2
    # --------------------------------------------------------

    {
        "section":
            "Figure 2",

        "check":
            "Composition statistics",

        "key_result":
            (
                "7/8 shared PC1 FDR<0.05; "
                "1/8 Disease×PC1 FDR<0.05"
            ),

        "status":
            (
                "PASS"
                if fig2_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 2",

        "check":
            "Composition interaction pattern",

        "key_result":
            (
                "PEC only"
            ),

        "status":
            (
                "PASS"
                if fig2_pattern_pass
                else
                "CHECK"
            )
    },


    # --------------------------------------------------------
    # Figure 3
    # --------------------------------------------------------

    {
        "section":
            "Figure 3",

        "check":
            "Formal gene testing family",

        "key_result":
            (
                f"{mac_formal_tests} MAC + "
                f"{fib_formal_tests} FIB = "
                f"{mac_formal_tests + fib_formal_tests}"
            ),

        "status":
            (
                "PASS"
                if formal_family_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 3",

        "check":
            "Global FDR-significant genes",

        "key_result":
            (
                f"MAC {mac_sig_genes}; "
                f"FIB {fib_sig_genes}"
            ),

        "status":
            (
                "PASS"
                if formal_family_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 3",

        "check":
            "Frozen molecular modules",

        "key_result":
            (
                "module sizes and scores regenerated"
            ),

        "status":
            (
                "PASS"
                if module_structure_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 3",

        "check":
            "Slide-sensitivity models",

        "key_result":
            (
                f"{n_module_p_match}/9 "
                "interaction P values matched"
            ),

        "status":
            (
                "PASS"
                if module_p_pass
                else
                "CHECK"
            )
    },


    # --------------------------------------------------------
    # Figure 4
    # --------------------------------------------------------

    {
        "section":
            "Figure 4",

        "check":
            "Formal spatial relationships",

        "key_result":
            (
                "4/4 P values and FDR values matched"
            ),

        "status":
            (
                "PASS"
                if formal_spatial_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 4",

        "check":
            "MAC–FIB model specifications",

        "key_result":
            (
                f"{n_spec_pass}/9 models matched; "
                "ROI counts and ranks matched"
            ),

        "status":
            (
                "PASS"
                if spec_pass
                else
                "CHECK"
            )
    },


    {
        "section":
            "Figure 4",

        "check":
            "Spatial robustness",

        "key_result":
            (
                "LOO, slide adjustment, "
                "curve geometry, k sensitivity"
            ),

        "status":
            (
                "PASS"
                if fig4_secondary_pass
                else
                "CHECK"
            )
    }
]


master = pd.DataFrame(
    master_rows
)


# ============================================================
# 9. Add provenance status
#
# Important:
#
# The original code that GENERATED the three frozen
# PC1 trimming bounds was not found in the searched
# data directory.
#
# However:
# using those frozen prespecified bounds,
# all 9 statistical models reproduce exactly.
#
# This is therefore tracked separately from statistical
# reproducibility.
# ============================================================

provenance = pd.DataFrame({

    "item": [

        "Figure 4 PC1 trimming-bound generation rule",

        "Figure 3 pathway-library online state"
    ],

    "status": [

        "DOCUMENTATION PENDING",

        "FROZEN OUTPUT"
    ],

    "note": [

        (
            "Original bound-generation code was not "
            "located under the current data-directory "
            "search scope. All 9 frozen-bound statistical "
            "models reproduce exactly."
        ),

        (
            "Pathway enrichment is intentionally preserved "
            "as frozen output because external GO/Reactome/"
            "Enrichr libraries can change over time."
        )
    ]
})


# ============================================================
# 10. Print master table
# ============================================================

print(
    "\n" + "=" * 88
)

print(
    "MASTER REPRODUCTION CHECK"
)

print(
    "=" * 88
)


display(
    master
)


print(
    "\n" + "=" * 88
)

print(
    "PROVENANCE / FROZEN-OUTPUT NOTES"
)

print(
    "=" * 88
)


display(
    provenance
)


# ============================================================
# 11. Figure-level status
# ============================================================

figure_status = pd.DataFrame({

    "component": [

        "Input integrity",

        "Figure 1",

        "Figure 2",

        "Figure 3",

        "Figure 4"
    ],

    "status": [

        "PASS",

        (
            "PASS"
            if fig1_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if fig2_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if fig3_pass
            else
            "CHECK"
        ),

        (
            "PASS"
            if fig4_pass
            else
            "CHECK"
        )
    ]
})


print(
    "\n" + "=" * 88
)

print(
    "FIGURE-LEVEL REPRODUCTION STATUS"
)

print(
    "=" * 88
)


display(
    figure_status
)


# ============================================================
# 12. Save final reproduction files
# ============================================================

master_path = (
    REPRO_RESULTS /
    "reproduction_master_check.csv"
)


figure_status_path = (
    REPRO_RESULTS /
    "reproduction_figure_status.csv"
)


provenance_path = (
    REPRO_RESULTS /
    "reproduction_provenance_notes.csv"
)


master.to_csv(
    master_path,
    index=False
)


figure_status.to_csv(
    figure_status_path,
    index=False
)


provenance.to_csv(
    provenance_path,
    index=False
)


# ============================================================
# 13. Final verdict
# ============================================================

all_figures_pass = bool(
    (
        figure_status[
            "status"
        ]
        ==
        "PASS"
    ).all()
)


print(
    "\n" + "=" * 88
)


if all_figures_pass:

    print(
        "✅✅✅ REPRODUCTION SUCCESSFUL ✅✅✅"
    )


    print(
        "\nAll core manuscript statistical results "
        "for Figures 1–4 were independently reproduced."
    )


    print(
        "\nFigure 1: PASS"
    )


    print(
        "Figure 2: PASS"
    )


    print(
        "Figure 3: PASS"
    )


    print(
        "Figure 4: PASS"
    )


    print(
        "\nOne documentation item remains:"
    )


    print(
        "the original code that generated the frozen "
        "Figure 4 PC1 trimming bounds should be recovered "
        "if available."
    )


    print(
        "\nThis does not affect the reproduced "
        "statistical results."
    )


else:

    print(
        "⚠ FINAL REPRODUCTION HAS CHECK ITEMS"
    )


    print(
        "\nDo not finalize the manuscript yet."
    )


    print(
        "Inspect the figure-level table above."
    )


print(
    "\nSaved:"
)


print(
    master_path
)


print(
    figure_status_path
)


print(
    provenance_path
)


print(
    "=" * 88
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 7
#
# SMALL-CLUSTER ROBUSTNESS
# Restricted wild cluster bootstrap
#
# Primary endpoints:
#   1. MAC M2
#   2. FIB M1
#   3. FIB M3
#   4. MAC–FIB spatial neighborhood
#
# Goal:
# Assess Disease × PC1 interaction inference when the
# anti-GBM group contains only 5 patient clusters.
#
# IMPORTANT:
# - Patient remains the cluster unit
# - Original patient-balanced weights are retained
# - Bootstrap is performed under the restricted null model
# - No gene/module selection is repeated
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

import patsy
import statsmodels.api as sm

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


REPRO_RESULTS = (
    BASE /
    "reproduction" /
    "results"
)


REPRO_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


MODULE_SCORE_PATH = (
    BASE /
    "figure5_frozen_module_scores.csv"
)


SPATIAL_PATH = (
    BASE /
    "figure4_driver_neighbor_data_smoothed.csv"
)


for p in [
    MODULE_SCORE_PATH,
    SPATIAL_PATH
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required file:\n{p}"
        )


# ============================================================
# 2. Bootstrap settings
# ============================================================

SEED = 20260908

B_INITIAL = 1999

B_FINAL = 9999


# If initial bootstrap P lies in this zone,
# automatically increase to B_FINAL.
REFINE_LOW = 0.03

REFINE_HIGH = 0.08


rng_master = np.random.default_rng(
    SEED
)


print(
    "=" * 88
)

print(
    "CELL 7 — SMALL-CLUSTER WILD BOOTSTRAP"
)

print(
    "=" * 88
)


print(
    "\nInitial bootstrap replications:",
    B_INITIAL
)


print(
    "Adaptive final replications:",
    B_FINAL
)


print(
    "Random seed:",
    SEED
)


# ============================================================
# 3. Expected primary asymptotic P values
#
# Used ONLY as a sanity check.
# They are NOT used in bootstrap calculations.
# ============================================================

EXPECTED_PRIMARY_P = {

    "MAC M2":
        7.462782e-05,

    "FIB M1":
        1.267494e-06,

    "FIB M3":
        3.961392e-08,

    "MAC-FIB":
        1.794172e-02
}


# ============================================================
# 4. Load module scores
# ============================================================

module_scores = pd.read_csv(
    MODULE_SCORE_PATH
)


print(
    "\nModule-score table:",
    module_scores.shape
)


print(
    "Module-score columns:"
)


print(
    module_scores.columns.tolist()
)


required_module_cols = [
    "module_label",
    "module_score",
    "Disease",
    "Patient",
    "PC1"
]


missing_module_cols = [
    c
    for c in required_module_cols
    if c not in module_scores.columns
]


if len(
    missing_module_cols
) > 0:

    raise ValueError(
        "Module score table missing columns:\n"
        +
        "\n".join(
            missing_module_cols
        )
    )


# ============================================================
# 5. Load spatial source table
# ============================================================

spatial = pd.read_csv(
    SPATIAL_PATH,
    index_col=0
)


spatial.index = (
    spatial.index.astype(str)
)


required_spatial_cols = [
    "MAC_to_FIB",
    "Disease",
    "Patient",
    "PC1"
]


missing_spatial_cols = [
    c
    for c in required_spatial_cols
    if c not in spatial.columns
]


if len(
    missing_spatial_cols
) > 0:

    raise ValueError(
        "Spatial table missing columns:\n"
        +
        "\n".join(
            missing_spatial_cols
        )
    )


print(
    "\nSpatial source table:",
    spatial.shape
)


# ============================================================
# 6. Webb six-point multipliers
#
# Mean = 0
# Variance = 1
#
# Particularly useful as a small-cluster wild-bootstrap
# multiplier distribution.
# ============================================================

WEBB_VALUES = np.array([

    -np.sqrt(1.5),

    -1.0,

    -np.sqrt(0.5),

    np.sqrt(0.5),

    1.0,

    np.sqrt(1.5)

])


# ============================================================
# 7. Prepare one endpoint
# ============================================================

def prepare_endpoint(
    data,
    outcome
):

    d = (
        data[
            [
                outcome,
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .dropna()
        .copy()
    )


    # --------------------------------------------------------
    # LN vs anti-GBM only
    # --------------------------------------------------------

    d = (
        d[
            d[
                "Disease"
            ].astype(str)
            .isin(
                [
                    "SLE",
                    "GBM"
                ]
            )
        ]
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(

        d[
            "Disease"
        ].astype(str),

        categories=[
            "SLE",
            "GBM"
        ]
    )


    d[
        "Patient"
    ] = (
        d[
            "Patient"
        ].astype(str)
    )


    d[
        "PC1"
    ] = pd.to_numeric(
        d[
            "PC1"
        ],
        errors="raise"
    )


    d[
        outcome
    ] = pd.to_numeric(
        d[
            outcome
        ],
        errors="raise"
    )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    n_roi_per_patient = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0
        /
        n_roi_per_patient
    )


    return d


# ============================================================
# 8. Design matrices
#
# FULL:
# y ~ spline(PC1) * Disease
#
# RESTRICTED NULL:
# y ~ spline(PC1) + Disease
#
# Wild bootstrap residuals are generated under the
# restricted null hypothesis of no Disease × PC1 interaction.
# ============================================================

def make_design(
    d
):

    full_formula = (
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    )


    restricted_formula = (
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "+ C(Disease)"
    )


    X_full_df = patsy.dmatrix(
        full_formula,
        d,
        return_type="dataframe"
    )


    X_restricted_df = patsy.dmatrix(
        restricted_formula,
        d,
        return_type="dataframe"
    )


    full_columns = (
        X_full_df.columns.tolist()
    )


    interaction_indices = [

        i

        for i, name
        in enumerate(
            full_columns
        )

        if (
            ":"
            in name
            and
            "C(Disease)"
            in name
            and
            "bs(PC1"
            in name
        )
    ]


    if len(
        interaction_indices
    ) == 0:

        raise ValueError(
            "Could not identify Disease × PC1 terms."
        )


    R = np.zeros(
        (
            len(
                interaction_indices
            ),
            len(
                full_columns
            )
        )
    )


    for row_i, col_i in enumerate(
        interaction_indices
    ):

        R[
            row_i,
            col_i
        ] = 1.0


    return (
        np.asarray(
            X_full_df,
            dtype=float
        ),

        np.asarray(
            X_restricted_df,
            dtype=float
        ),

        full_columns,

        interaction_indices,

        R
    )


# ============================================================
# 9. Fit observed full + restricted models
# ============================================================

def observed_model(
    d,
    outcome
):

    (
        X_full,
        X_restricted,
        full_columns,
        interaction_indices,
        R
    ) = make_design(
        d
    )


    y = (
        d[
            outcome
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = (
        d[
            "patient_weight"
        ]
        .to_numpy(
            dtype=float
        )
    )


    groups = (
        d[
            "Patient"
        ]
        .astype(str)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Full model with clustered covariance
    # --------------------------------------------------------

    full_fit = sm.WLS(
        y,
        X_full,
        weights=weights
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                groups
        }
    )


    observed_wald = (
        full_fit.wald_test(
            R,
            scalar=True
        )
    )


    observed_stat = float(
        observed_wald.statistic
    )


    observed_p = float(
        observed_wald.pvalue
    )


    # --------------------------------------------------------
    # Restricted model for null-residual bootstrap
    # --------------------------------------------------------

    restricted_fit = sm.WLS(
        y,
        X_restricted,
        weights=weights
    ).fit()


    yhat_null = np.asarray(
        restricted_fit.fittedvalues,
        dtype=float
    )


    residual_null = np.asarray(
        restricted_fit.resid,
        dtype=float
    )


    return {

        "X_full":
            X_full,

        "R":
            R,

        "y":
            y,

        "weights":
            weights,

        "groups":
            groups,

        "yhat_null":
            yhat_null,

        "residual_null":
            residual_null,

        "observed_stat":
            observed_stat,

        "observed_p":
            observed_p,

        "full_columns":
            full_columns,

        "interaction_indices":
            interaction_indices
    }


# ============================================================
# 10. One wild-cluster bootstrap run
# ============================================================

def wild_cluster_bootstrap(
    d,
    outcome,
    B,
    seed
):

    model = observed_model(
        d,
        outcome
    )


    X_full = (
        model[
            "X_full"
        ]
    )


    R = (
        model[
            "R"
        ]
    )


    weights = (
        model[
            "weights"
        ]
    )


    groups = (
        model[
            "groups"
        ]
    )


    yhat_null = (
        model[
            "yhat_null"
        ]
    )


    residual_null = (
        model[
            "residual_null"
        ]
    )


    observed_stat = (
        model[
            "observed_stat"
        ]
    )


    # --------------------------------------------------------
    # Unique patient clusters
    # --------------------------------------------------------

    unique_groups = np.array(
        sorted(
            pd.unique(
                groups
            )
        )
    )


    group_to_index = {

        g:
            i

        for i, g
        in enumerate(
            unique_groups
        )
    }


    row_group_index = np.array([

        group_to_index[
            g
        ]

        for g in groups
    ])


    rng = np.random.default_rng(
        seed
    )


    bootstrap_stats = []


    invalid = 0


    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------

    for b in range(
        B
    ):

        cluster_multiplier = (
            rng.choice(
                WEBB_VALUES,
                size=len(
                    unique_groups
                ),
                replace=True
            )
        )


        multiplier_rows = (
            cluster_multiplier[
                row_group_index
            ]
        )


        y_star = (
            yhat_null
            +
            residual_null
            *
            multiplier_rows
        )


        try:

            fit_star = sm.WLS(
                y_star,
                X_full,
                weights=weights
            ).fit(
                cov_type="cluster",
                cov_kwds={
                    "groups":
                        groups
                }
            )


            stat_star = float(
                fit_star.wald_test(
                    R,
                    scalar=True
                ).statistic
            )


            if np.isfinite(
                stat_star
            ):

                bootstrap_stats.append(
                    stat_star
                )


            else:

                invalid += 1


        except Exception:

            invalid += 1


        # ----------------------------------------------------
        # Progress every ~10%
        # ----------------------------------------------------

        progress_step = max(
            1,
            B // 10
        )


        if (
            (
                b + 1
            )
            %
            progress_step
            ==
            0
        ):

            print(
                f"      {b + 1:,} / {B:,}"
            )


    bootstrap_stats = np.asarray(
        bootstrap_stats,
        dtype=float
    )


    valid_B = len(
        bootstrap_stats
    )


    if valid_B < (
        B * 0.95
    ):

        raise RuntimeError(
            f"""
Too many invalid bootstrap fits.

Requested: {B}
Valid: {valid_B}
Invalid: {invalid}
"""
        )


    exceed = int(
        np.sum(
            bootstrap_stats
            >=
            observed_stat
        )
    )


    # +1 correction
    bootstrap_p = (
        exceed
        +
        1
    ) / (
        valid_B
        +
        1
    )


    # --------------------------------------------------------
    # Monte-Carlo uncertainty of bootstrap P
    # Wilson CI for exceedance proportion
    # --------------------------------------------------------

    ci_low, ci_high = (
        proportion_confint(
            count=exceed,
            nobs=valid_B,
            alpha=0.05,
            method="wilson"
        )
    )


    return {

        "observed_p":
            model[
                "observed_p"
            ],

        "observed_stat":
            observed_stat,

        "bootstrap_p":
            bootstrap_p,

        "bootstrap_ci_low":
            float(
                ci_low
            ),

        "bootstrap_ci_high":
            float(
                ci_high
            ),

        "B_requested":
            B,

        "B_valid":
            valid_B,

        "invalid_bootstraps":
            invalid,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_LN_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                ==
                "SLE",
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                ==
                "GBM",
                "Patient"
            ].nunique()
    }


# ============================================================
# 11. Endpoint datasets
# ============================================================

endpoints = {}


for module_label in [

    "MAC M2",

    "FIB M1",

    "FIB M3"

]:

    d_module = (
        module_scores[
            module_scores[
                "module_label"
            ]
            .astype(str)
            ==
            module_label
        ]
        .copy()
    )


    d_module = prepare_endpoint(
        d_module,
        "module_score"
    )


    endpoints[
        module_label
    ] = {

        "data":
            d_module,

        "outcome":
            "module_score"
    }


# ------------------------------------------------------------
# Spatial MAC–FIB
# ------------------------------------------------------------

d_spatial = (
    spatial[
        spatial[
            "Disease"
        ]
        .astype(str)
        .isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ]
    .copy()
)


d_spatial = prepare_endpoint(
    d_spatial,
    "MAC_to_FIB"
)


endpoints[
    "MAC-FIB"
] = {

    "data":
        d_spatial,

    "outcome":
        "MAC_to_FIB"
}


# ============================================================
# 12. Endpoint audit BEFORE bootstrap
# ============================================================

audit_rows = []


for label, obj in endpoints.items():

    d = (
        obj[
            "data"
        ]
    )


    audit_rows.append({

        "endpoint":
            label,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_LN_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                ==
                "SLE",
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                ==
                "GBM",
                "Patient"
            ].nunique()
    })


endpoint_audit = pd.DataFrame(
    audit_rows
)


print(
    "\n" + "=" * 88
)

print(
    "ENDPOINT CLUSTER AUDIT"
)

print(
    "=" * 88
)


display(
    endpoint_audit
)


# ============================================================
# 13. First check ordinary asymptotic P values
#
# This ensures the dataset/model specification matches the
# already reproduced primary analysis BEFORE bootstrap.
# ============================================================

sanity_rows = []


for label, obj in endpoints.items():

    d = (
        obj[
            "data"
        ]
    )


    outcome = (
        obj[
            "outcome"
        ]
    )


    obs = observed_model(
        d,
        outcome
    )


    reproduced_p = float(
        obs[
            "observed_p"
        ]
    )


    expected_p = float(
        EXPECTED_PRIMARY_P[
            label
        ]
    )


    if (
        reproduced_p > 0
        and
        expected_p > 0
    ):

        log_diff = abs(
            np.log10(
                reproduced_p
            )
            -
            np.log10(
                expected_p
            )
        )


    else:

        log_diff = np.inf


    p_match = (
        log_diff
        <
        1e-4
        or
        abs(
            reproduced_p
            -
            expected_p
        )
        <
        1e-8
    )


    sanity_rows.append({

        "endpoint":
            label,

        "expected_primary_P":
            expected_p,

        "reproduced_primary_P":
            reproduced_p,

        "log10_difference":
            log_diff,

        "status":
            (
                "PASS"
                if p_match
                else
                "CHECK"
            )
    })


sanity = pd.DataFrame(
    sanity_rows
)


print(
    "\n" + "=" * 88
)

print(
    "PRIMARY MODEL SANITY CHECK"
)

print(
    "=" * 88
)


display(
    sanity
)


if not (
    sanity[
        "status"
    ]
    ==
    "PASS"
).all():

    raise RuntimeError(
        """
Primary asymptotic P values did not reproduce.

STOP here and send me the sanity-check table.
Do not run the bootstrap.
"""
    )


print(
    "\n✅ All four original primary models reproduced."
)


# ============================================================
# 14. Wild-cluster bootstrap
# ============================================================

start_time = time.time()


bootstrap_rows = []


for endpoint_i, (
    label,
    obj
) in enumerate(
    endpoints.items()
):

    print(
        "\n" + "=" * 88
    )


    print(
        f"BOOTSTRAP: {label}"
    )


    print(
        "=" * 88
    )


    d = (
        obj[
            "data"
        ]
    )


    outcome = (
        obj[
            "outcome"
        ]
    )


    endpoint_seed = (
        SEED
        +
        endpoint_i
        *
        10000
    )


    # --------------------------------------------------------
    # Initial run
    # --------------------------------------------------------

    print(
        f"\nInitial Webb wild cluster bootstrap: "
        f"B={B_INITIAL:,}"
    )


    initial = wild_cluster_bootstrap(
        d=d,
        outcome=outcome,
        B=B_INITIAL,
        seed=endpoint_seed
    )


    final_result = initial


    # --------------------------------------------------------
    # Adaptive refinement if close to 0.05
    # --------------------------------------------------------

    if (
        initial[
            "bootstrap_p"
        ]
        >=
        REFINE_LOW
        and
        initial[
            "bootstrap_p"
        ]
        <=
        REFINE_HIGH
    ):

        print(
            "\nBootstrap P is near the 0.05 region."
        )


        print(
            f"Refining with B={B_FINAL:,}..."
        )


        final_result = (
            wild_cluster_bootstrap(
                d=d,
                outcome=outcome,
                B=B_FINAL,
                seed=endpoint_seed + 500000
            )
        )


    bootstrap_rows.append({

        "endpoint":
            label,

        "n_ROI":
            final_result[
                "n_ROI"
            ],

        "n_patients":
            final_result[
                "n_patients"
            ],

        "n_LN_patients":
            final_result[
                "n_LN_patients"
            ],

        "n_GBM_patients":
            final_result[
                "n_GBM_patients"
            ],

        "asymptotic_clustered_P":
            final_result[
                "observed_p"
            ],

        "wild_cluster_bootstrap_P":
            final_result[
                "bootstrap_p"
            ],

        "bootstrap_MC_CI_low":
            final_result[
                "bootstrap_ci_low"
            ],

        "bootstrap_MC_CI_high":
            final_result[
                "bootstrap_ci_high"
            ],

        "B_valid":
            final_result[
                "B_valid"
            ],

        "invalid_bootstraps":
            final_result[
                "invalid_bootstraps"
            ]
    })


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ============================================================
# 15. BH correction across the four prespecified
#     small-cluster sensitivity endpoints
#
# This is deliberately conservative and supplementary.
# ============================================================

bootstrap_results[
    "wild_bootstrap_FDR_across_4"
] = multipletests(
    bootstrap_results[
        "wild_cluster_bootstrap_P"
    ],
    method="fdr_bh"
)[1]


bootstrap_results[
    "wild_bootstrap_nominal_P_lt_005"
] = (
    bootstrap_results[
        "wild_cluster_bootstrap_P"
    ]
    <
    0.05
)


bootstrap_results[
    "wild_bootstrap_FDR_lt_005"
] = (
    bootstrap_results[
        "wild_bootstrap_FDR_across_4"
    ]
    <
    0.05
)


# ============================================================
# 16. Display
# ============================================================

print(
    "\n" + "=" * 88
)

print(
    "SMALL-CLUSTER ROBUSTNESS RESULTS"
)

print(
    "=" * 88
)


display(
    bootstrap_results
)


# ============================================================
# 17. Interpretation summary
# ============================================================

interpretation_rows = []


for row in bootstrap_results.itertuples():

    p_boot = float(
        row.wild_cluster_bootstrap_P
    )


    fdr_boot = float(
        row.wild_bootstrap_FDR_across_4
    )


    if fdr_boot < 0.05:

        interpretation = (
            "Robust after wild-cluster bootstrap "
            "and BH correction"
        )

        status = "STRONG"

    elif p_boot < 0.05:

        interpretation = (
            "Nominally robust under wild-cluster bootstrap; "
            "not FDR<0.05 across four sensitivity endpoints"
        )

        status = "SUPPORTED"

    else:

        interpretation = (
            "Not nominally significant under "
            "wild-cluster bootstrap"
        )

        status = "ATTENUATED"


    interpretation_rows.append({

        "endpoint":
            row.endpoint,

        "asymptotic_P":
            row.asymptotic_clustered_P,

        "wild_bootstrap_P":
            p_boot,

        "wild_bootstrap_FDR":
            fdr_boot,

        "interpretation":
            interpretation,

        "status":
            status
    })


interpretation_df = pd.DataFrame(
    interpretation_rows
)


print(
    "\n" + "=" * 88
)

print(
    "REVIEWER-ORIENTED INTERPRETATION"
)

print(
    "=" * 88
)


display(
    interpretation_df
)


# ============================================================
# 18. Save
# ============================================================

result_path = (
    REPRO_RESULTS /
    "cell7_small_cluster_wild_bootstrap.csv"
)


interpretation_path = (
    REPRO_RESULTS /
    "cell7_small_cluster_interpretation.csv"
)


audit_path = (
    REPRO_RESULTS /
    "cell7_small_cluster_endpoint_audit.csv"
)


bootstrap_results.to_csv(
    result_path,
    index=False
)


interpretation_df.to_csv(
    interpretation_path,
    index=False
)


endpoint_audit.to_csv(
    audit_path,
    index=False
)


# ============================================================
# 19. Runtime
# ============================================================

runtime_minutes = (
    time.time()
    -
    start_time
) / 60


print(
    "\nRuntime:"
)


print(
    f"{runtime_minutes:.2f} minutes"
)


# ============================================================
# 20. Final message
# ============================================================

n_nominal = int(
    bootstrap_results[
        "wild_bootstrap_nominal_P_lt_005"
    ].sum()
)


n_fdr = int(
    bootstrap_results[
        "wild_bootstrap_FDR_lt_005"
    ].sum()
)


print(
    "\n" + "=" * 88
)

print(
    "CELL 7 COMPLETE"
)

print(
    "=" * 88
)


print(
    f"\nWild-cluster nominal P < 0.05: "
    f"{n_nominal}/4"
)


print(
    f"Wild-cluster FDR < 0.05 across 4: "
    f"{n_fdr}/4"
)


print(
    "\nDo NOT interpret a non-significant bootstrap "
    "result as a failed reproduction."
)


print(
    "This is a deliberately stricter small-cluster "
    "sensitivity analysis."
)


print(
    "\nSaved:"
)


print(
    result_path
)


print(
    interpretation_path
)


print(
    audit_path
)


print(
    "=" * 88
)

In [ ]:
# ============================================================
# REPRODUCTION — CELL 8
#
# FIGURE 2 COMPOSITIONAL SENSITIVITY
#
# Goal:
# Test whether the main Figure 2 conclusion is robust when
# cell composition is analysed on a compositional scale.
#
# Strategy:
#
#   Exact cell counts from roi782_cell_metadata.pkl
#
#       ↓
#
#   8 focal populations + "Other"
#
#       ↓
#
#   add 0.5-cell pseudocount
#
#       ↓
#
#   centered log-ratio (CLR)
#
#       ↓
#
#   same patient-balanced clustered spline models
#
# This is a sensitivity analysis.
# It does NOT replace the original raw-fraction analysis.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. Paths
# ============================================================

BASE = Path(
    str(PROJECT_DIR)
)


ROI_PATH = (
    BASE /
    "roi_782_PC1_primary.h5ad"
)


CELL_META_PATH = (
    BASE /
    "roi782_cell_metadata.pkl"
)


PRIMARY_SUMMARY_PATH = (
    BASE /
    "reproduction" /
    "results" /
    "cell3_figure2_manuscript_summary.csv"
)


OUT_DIR = (
    BASE /
    "reproduction" /
    "results"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for p in [
    ROI_PATH,
    CELL_META_PATH,
    PRIMARY_SUMMARY_PATH
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Missing required file:\n{p}"
        )


print(
    "=" * 88
)

print(
    "CELL 8 — FIGURE 2 COMPOSITIONAL SENSITIVITY"
)

print(
    "=" * 88
)


# ============================================================
# 2. Load source objects
# ============================================================

roi_ad = ad.read_h5ad(
    ROI_PATH
)


roi_meta = (
    roi_ad.obs.copy()
)


roi_meta.index = (
    roi_meta.index.astype(str)
)


cells = pd.read_pickle(
    CELL_META_PATH
)


cells[
    "roi_id"
] = (
    cells[
        "roi_id"
    ].astype(str)
)


cells[
    "celltype_l1"
] = (
    cells[
        "celltype_l1"
    ].astype(str)
)


print(
    "\nROI object:",
    roi_ad.shape
)


print(
    "Cell metadata:",
    cells.shape
)


# ============================================================
# 3. Required metadata check
# ============================================================

required_roi_cols = [
    "Disease",
    "Patient_Sample_ID",
    "PC1_crescent"
]


required_cell_cols = [
    "roi_id",
    "celltype_l1"
]


missing_roi = [
    x
    for x in required_roi_cols
    if x not in roi_meta.columns
]


missing_cells = [
    x
    for x in required_cell_cols
    if x not in cells.columns
]


if len(
    missing_roi
) > 0:

    raise ValueError(
        "ROI metadata missing:\n"
        +
        "\n".join(
            missing_roi
        )
    )


if len(
    missing_cells
) > 0:

    raise ValueError(
        "Cell metadata missing:\n"
        +
        "\n".join(
            missing_cells
        )
    )


# ============================================================
# 4. Reconstruct three-disease common PC1 support
#
# Figure 2:
#   ANCA
#   SLE / LN
#   GBM / anti-GBM
# ============================================================

analysis_diseases = [
    "ANCA",
    "SLE",
    "GBM"
]


meta3 = (
    roi_meta[
        roi_meta[
            "Disease"
        ].isin(
            analysis_diseases
        )
    ]
    .copy()
)


ranges = (
    meta3
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        [
            "min",
            "max"
        ]
    )
)


common_low = float(
    ranges[
        "min"
    ].max()
)


common_high = float(
    ranges[
        "max"
    ].min()
)


analysis_meta = (
    meta3[
        meta3[
            "PC1_crescent"
        ].between(
            common_low,
            common_high
        )
    ]
    .copy()
)


analysis_meta.index = (
    analysis_meta.index.astype(str)
)


print(
    "\nCommon PC1 support:"
)


print(
    f"{common_low:.9f} to {common_high:.9f}"
)


print(
    "\nAnalysis ROI counts:"
)


display(
    analysis_meta[
        "Disease"
    ]
    .value_counts()
    .rename(
        "n_ROI"
    )
    .to_frame()
)


print(
    "\nAnalysis patient counts:"
)


display(
    analysis_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient_Sample_ID"
    ]
    .nunique()
    .rename(
        "n_patients"
    )
    .to_frame()
)


# ============================================================
# 5. Resolve the same 8 focal populations
# ============================================================

available_labels = set(
    cells[
        "celltype_l1"
    ]
    .dropna()
    .astype(str)
    .unique()
)


alias_map = {

    "MAC": [
        "MAC",
        "Macrophage"
    ],

    "Mono": [
        "Mono",
        "MONO",
        "Monocyte"
    ],

    "B": [
        "B",
        "B cell",
        "Bcell"
    ],

    "T": [
        "T",
        "T cell",
        "Tcell"
    ],

    "FIB": [
        "FIB",
        "Fibroblast"
    ],

    "EC": [
        "EC",
        "Endothelial"
    ],

    "PEC": [
        "PEC"
    ],

    "POD": [
        "POD",
        "Podocyte"
    ]
}


celltype_order = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]


resolved = {}


for desired in celltype_order:

    candidates = [
        x
        for x in alias_map[
            desired
        ]
        if x in available_labels
    ]


    if len(
        candidates
    ) == 0:

        print(
            "\nAvailable labels:"
        )


        print(
            sorted(
                available_labels
            )
        )


        raise ValueError(
            f"Cannot resolve {desired}"
        )


    resolved[
        desired
    ] = candidates[
        0
    ]


print(
    "\nResolved cell labels:"
)


display(
    pd.DataFrame({

        "analysis_label":
            list(
                resolved.keys()
            ),

        "stored_label":
            list(
                resolved.values()
            )
    })
)


# ============================================================
# 6. Restrict cell metadata to analytical ROIs
# ============================================================

analysis_roi_ids = set(
    analysis_meta.index
)


cells_use = (
    cells[
        cells[
            "roi_id"
        ].isin(
            analysis_roi_ids
        )
    ]
    .copy()
)


print(
    "\nCells in analytical ROIs:",
    len(
        cells_use
    )
)


# ============================================================
# 7. Exact total-cell counts per ROI
# ============================================================

total_counts = (
    cells_use
    .groupby(
        "roi_id",
        observed=True
    )
    .size()
)


count_table = pd.DataFrame(
    index=analysis_meta.index
)


for celltype in celltype_order:

    stored_label = (
        resolved[
            celltype
        ]
    )


    counts = (
        cells_use[
            cells_use[
                "celltype_l1"
            ]
            ==
            stored_label
        ]
        .groupby(
            "roi_id",
            observed=True
        )
        .size()
    )


    count_table[
        celltype
    ] = (
        count_table.index
        .to_series()
        .map(
            counts
        )
        .fillna(
            0
        )
        .astype(int)
        .to_numpy()
    )


count_table[
    "Total"
] = (
    count_table.index
    .to_series()
    .map(
        total_counts
    )
    .fillna(
        0
    )
    .astype(int)
    .to_numpy()
)


# ============================================================
# 8. "Other" as ninth composition component
#
# Ensures the CLR is applied to a complete partition of
# all annotated cells in an ROI.
# ============================================================

focal_sum = (
    count_table[
        celltype_order
    ]
    .sum(
        axis=1
    )
)


count_table[
    "Other"
] = (
    count_table[
        "Total"
    ]
    -
    focal_sum
)


if (
    count_table[
        "Other"
    ]
    <
    0
).any():

    raise ValueError(
        "Negative Other count detected."
    )


components = (
    celltype_order
    +
    [
        "Other"
    ]
)


print(
    "\nComposition-count audit:"
)


display(
    count_table[
        components
        +
        [
            "Total"
        ]
    ]
    .describe()
)


# ============================================================
# 9. Check that the 9 components exactly sum to total cells
# ============================================================

reconstructed_total = (
    count_table[
        components
    ]
    .sum(
        axis=1
    )
)


max_count_mismatch = int(
    (
        reconstructed_total
        -
        count_table[
            "Total"
        ]
    )
    .abs()
    .max()
)


print(
    "\nMaximum component-count mismatch:",
    max_count_mismatch
)


if max_count_mismatch != 0:

    raise ValueError(
        "Composition components do not sum to total cells."
    )


# ============================================================
# 10. CLR transformation
#
# Pseudocount:
#   +0.5 cells to each of 9 components
#
# For each ROI:
#
# CLR_j =
#   log(count_j + 0.5)
#   -
#   mean_j(log(count_j + 0.5))
# ============================================================

PSEUDOCOUNT = 0.5


adjusted_counts = (
    count_table[
        components
    ]
    .astype(float)
    +
    PSEUDOCOUNT
)


log_counts = np.log(
    adjusted_counts
)


row_log_mean = (
    log_counts.mean(
        axis=1
    )
)


clr = (
    log_counts
    .sub(
        row_log_mean,
        axis=0
    )
)


# numerical identity:
# CLR components should sum ~0 within each ROI
clr_sum_abs_max = float(
    clr.sum(
        axis=1
    )
    .abs()
    .max()
)


print(
    "\nMaximum absolute CLR row sum:",
    clr_sum_abs_max
)


# ============================================================
# 11. Assemble model table
# ============================================================

model_df = (
    clr[
        celltype_order
    ]
    .copy()
)


model_df[
    "Disease"
] = (
    analysis_meta[
        "Disease"
    ]
    .astype(str)
)


model_df[
    "Patient"
] = (
    analysis_meta[
        "Patient_Sample_ID"
    ]
    .astype(str)
)


model_df[
    "PC1"
] = pd.to_numeric(
    analysis_meta[
        "PC1_crescent"
    ],
    errors="raise"
)


model_df[
    "Total"
] = (
    count_table[
        "Total"
    ]
)


# ============================================================
# 12. Patient-balanced clustered spline model
# ============================================================

def fit_clr_models(
    data,
    outcome
):

    d = (
        data[
            [
                outcome,
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .dropna()
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(

        d[
            "Disease"
        ].astype(str),

        categories=[
            "ANCA",
            "SLE",
            "GBM"
        ]
    )


    # --------------------------------------------------------
    # Patient-balanced weights
    # --------------------------------------------------------

    patient_n = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0
        /
        patient_n
    )


    # --------------------------------------------------------
    # Shared progression model
    # --------------------------------------------------------

    shared_formula = (
        f"{outcome} ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "+ C(Disease)"
    )


    shared_fit = smf.wls(
        shared_formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    shared_terms = [

        term

        for term
        in shared_fit.params.index

        if (
            "bs(PC1"
            in term
            and
            ":"
            not in term
        )
    ]


    R_shared = np.zeros(
        (
            len(
                shared_terms
            ),
            len(
                shared_fit.params
            )
        )
    )


    for i, term in enumerate(
        shared_terms
    ):

        R_shared[
            i,
            shared_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    shared_p = float(
        shared_fit.wald_test(
            R_shared,
            scalar=True
        ).pvalue
    )


    # --------------------------------------------------------
    # Disease × progression model
    # --------------------------------------------------------

    interaction_formula = (
        f"{outcome} ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    )


    interaction_fit = smf.wls(
        interaction_formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    interaction_terms = [

        term

        for term
        in interaction_fit.params.index

        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    R_interaction = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                interaction_fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R_interaction[
            i,
            interaction_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    interaction_p = float(
        interaction_fit.wald_test(
            R_interaction,
            scalar=True
        ).pvalue
    )


    shared_rank = int(
        np.linalg.matrix_rank(
            shared_fit.model.exog
        )
    )


    shared_cols = int(
        shared_fit.model.exog.shape[
            1
        ]
    )


    interaction_rank = int(
        np.linalg.matrix_rank(
            interaction_fit.model.exog
        )
    )


    interaction_cols = int(
        interaction_fit.model.exog.shape[
            1
        ]
    )


    return {

        "shared_p":
            shared_p,

        "interaction_p":
            interaction_p,

        "shared_rank":
            shared_rank,

        "shared_columns":
            shared_cols,

        "interaction_rank":
            interaction_rank,

        "interaction_columns":
            interaction_cols,

        "n_ROI":
            len(
                d
            ),

        "n_patients":
            d[
                "Patient"
            ].nunique()
    }


# ============================================================
# 13. Fit all 8 focal populations
# ============================================================

rows = []


print(
    "\n" + "=" * 88
)

print(
    "REFITTING CLR COMPOSITION MODELS"
)

print(
    "=" * 88
)


for celltype in celltype_order:

    print(
        "Fitting",
        celltype,
        "..."
    )


    result = fit_clr_models(
        model_df,
        celltype
    )


    rows.append({

        "celltype":
            celltype,

        "CLR_shared_PC1_pvalue":
            result[
                "shared_p"
            ],

        "CLR_interaction_pvalue":
            result[
                "interaction_p"
            ],

        "shared_rank":
            result[
                "shared_rank"
            ],

        "shared_columns":
            result[
                "shared_columns"
            ],

        "interaction_rank":
            result[
                "interaction_rank"
            ],

        "interaction_columns":
            result[
                "interaction_columns"
            ],

        "n_ROI":
            result[
                "n_ROI"
            ],

        "n_patients":
            result[
                "n_patients"
            ]
    })


clr_results = pd.DataFrame(
    rows
)


# ============================================================
# 14. BH-FDR across 8 cell populations
# ============================================================

clr_results[
    "CLR_shared_PC1_FDR"
] = multipletests(
    clr_results[
        "CLR_shared_PC1_pvalue"
    ],
    method="fdr_bh"
)[1]


clr_results[
    "CLR_interaction_FDR"
] = multipletests(
    clr_results[
        "CLR_interaction_pvalue"
    ],
    method="fdr_bh"
)[1]


clr_results[
    "CLR_shared_sig"
] = (
    clr_results[
        "CLR_shared_PC1_FDR"
    ]
    <
    0.05
)


clr_results[
    "CLR_interaction_sig"
] = (
    clr_results[
        "CLR_interaction_FDR"
    ]
    <
    0.05
)


print(
    "\n" + "=" * 88
)

print(
    "CLR COMPOSITION RESULTS"
)

print(
    "=" * 88
)


display(
    clr_results
)


# ============================================================
# 15. Load primary Figure 2 classification
# ============================================================

primary = pd.read_csv(
    PRIMARY_SUMMARY_PATH
)


primary_small = (
    primary[
        [
            "celltype",
            "shared_PC1_FDR",
            "Disease_x_PC1_FDR",
            "shared_significant_FDR005",
            "interaction_significant_FDR005"
        ]
    ]
    .copy()
)


comparison = (
    primary_small
    .merge(
        clr_results[
            [
                "celltype",
                "CLR_shared_PC1_FDR",
                "CLR_interaction_FDR",
                "CLR_shared_sig",
                "CLR_interaction_sig"
            ]
        ],
        on="celltype",
        how="inner"
    )
)


comparison[
    "shared_significance_concordant"
] = (
    comparison[
        "shared_significant_FDR005"
    ]
    ==
    comparison[
        "CLR_shared_sig"
    ]
)


comparison[
    "interaction_significance_concordant"
] = (
    comparison[
        "interaction_significant_FDR005"
    ]
    ==
    comparison[
        "CLR_interaction_sig"
    ]
)


print(
    "\n" + "=" * 88
)

print(
    "PRIMARY FRACTION vs CLR COMPOSITIONAL SENSITIVITY"
)

print(
    "=" * 88
)


display(
    comparison
)


# ============================================================
# 16. Compact interpretation
# ============================================================

primary_shared_n = int(
    primary_small[
        "shared_significant_FDR005"
    ].sum()
)


clr_shared_n = int(
    clr_results[
        "CLR_shared_sig"
    ].sum()
)


primary_interaction_n = int(
    primary_small[
        "interaction_significant_FDR005"
    ].sum()
)


clr_interaction_n = int(
    clr_results[
        "CLR_interaction_sig"
    ].sum()
)


primary_interaction_types = (
    primary_small.loc[
        primary_small[
            "interaction_significant_FDR005"
        ],
        "celltype"
    ]
    .tolist()
)


clr_interaction_types = (
    clr_results.loc[
        clr_results[
            "CLR_interaction_sig"
        ],
        "celltype"
    ]
    .tolist()
)


shared_exact_concordance = int(
    comparison[
        "shared_significance_concordant"
    ].sum()
)


interaction_exact_concordance = int(
    comparison[
        "interaction_significance_concordant"
    ].sum()
)


summary = pd.DataFrame({

    "metric": [

        "Primary shared-PC1 significant populations",

        "CLR shared-PC1 significant populations",

        "Primary interaction significant populations",

        "CLR interaction significant populations",

        "Shared-effect significance concordance",

        "Interaction significance concordance",

        "Primary significant interactions",

        "CLR significant interactions"
    ],

    "result": [

        f"{primary_shared_n}/8",

        f"{clr_shared_n}/8",

        f"{primary_interaction_n}/8",

        f"{clr_interaction_n}/8",

        f"{shared_exact_concordance}/8",

        f"{interaction_exact_concordance}/8",

        ", ".join(
            primary_interaction_types
        )
        if len(
            primary_interaction_types
        ) > 0
        else "None",

        ", ".join(
            clr_interaction_types
        )
        if len(
            clr_interaction_types
        ) > 0
        else "None"
    ]
})


print(
    "\n" + "=" * 88
)

print(
    "COMPOSITIONAL SENSITIVITY SUMMARY"
)

print(
    "=" * 88
)


display(
    summary
)


# ============================================================
# 17. Descriptive conclusion
#
# Do NOT force this analysis to reproduce every individual P.
# The reviewer-level question is whether the broad biological
# structure persists after an appropriate compositional
# transformation.
# ============================================================

if (
    clr_shared_n >= 6
    and
    clr_interaction_n <= 2
):

    conclusion = (
        "BROADLY CONCORDANT"
    )


else:

    conclusion = (
        "MATERIAL DIFFERENCE — REVIEW REQUIRED"
    )


print(
    "\nOverall compositional sensitivity:"
)


print(
    conclusion
)


# ============================================================
# 18. Save
# ============================================================

count_path = (
    OUT_DIR /
    "cell8_CLR_component_counts.csv"
)


clr_path = (
    OUT_DIR /
    "cell8_CLR_transformed_composition.csv"
)


result_path = (
    OUT_DIR /
    "cell8_CLR_composition_results.csv"
)


comparison_path = (
    OUT_DIR /
    "cell8_primary_vs_CLR_comparison.csv"
)


summary_path = (
    OUT_DIR /
    "cell8_CLR_compositional_sensitivity_summary.csv"
)


count_table.to_csv(
    count_path
)


clr.to_csv(
    clr_path
)


clr_results.to_csv(
    result_path,
    index=False
)


comparison.to_csv(
    comparison_path,
    index=False
)


summary.to_csv(
    summary_path,
    index=False
)


print(
    "\nSaved:"
)


for p in [
    count_path,
    clr_path,
    result_path,
    comparison_path,
    summary_path
]:

    print(
        p
    )


print(
    "=" * 88
)